# Book Sales Demand Forecasting

Weekly unit sales for two long-selling titles (Title A, Title B) from 2012, final 32 weeks held out. Compares SARIMA (auto_arima), XGBoost, an LSTM tuned with KerasTuner, sequential and parallel SARIMA-LSTM hybrids, and monthly XGBoost/SARIMA.

*Origin: University of Cambridge Data Science Career Accelerator coursework (2024-25). Data source anonymised and not included. Cell outputs cleared to protect the source data; results quoted in the README come from the original run.*

## **Section 1 - Part 1: Import packages and data**

**Criteria**

    1.	Load the appropriate libraries and models.
    2.	Import both data files.
          Each file has four tabs, so ensure that you read the data from all four tabs.

Checking environment to install compatible libraries.

In [ ]:
import os
import subprocess
import sys
import warnings
warnings.filterwarnings("ignore")

# Check Colab environment
def check_colab_environment():
    try:
        import google.colab
        print("Running in Google Colab")
    except ImportError:
        print("Warning: Not running in Google Colab. Some features may not work.")
    print(f"Python version: {sys.version}")
    if sys.version_info < (3, 9) or sys.version_info >= (3, 12):
        print("Warning: Python version is not 3.9–3.11. Compatibility issues may arise.")

check_colab_environment()

In [ ]:
# Install required packages for all sections (Criteria 1: Load appropriate libraries)
!pip install --upgrade --force-reinstall \
    numpy==1.26.4 \
    pandas==2.1.3 \
    matplotlib==3.8.4 \
    seaborn==0.13.2 \
    statsmodels==0.14.1 \
    pmdarima==2.0.3 \
    scikit-learn==1.3.2 \
    tensorflow==2.15.0 \
    openpyxl==3.1.2 \
    xlrd==2.0.1 \
    sktime==0.25.0

### **Restart**

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.tsa.api as tsa
import pmdarima as pm
import tensorflow as tf
import sktime
from datetime import datetime, timedelta
from matplotlib.patches import Rectangle
import matplotlib.animation as animation
from matplotlib import animation
from matplotlib import rc
import statsmodels.graphics.api as smgraphics # gives access to all plotting functions in statsmodels.
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller as adf
from statsmodels.tsa.seasonal import STL
from scipy.stats import boxcox
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from pmdarima.arima import auto_arima
import warnings
# Suppress all warnings.
warnings.filterwarnings("ignore")
from statsmodels.tsa.stattools import acf
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
from sktime.forecasting.compose import (TransformedTargetForecaster, make_reduction)
from sktime.forecasting.model_selection import (ExpandingWindowSplitter, ForecastingGridSearchCV)
from sktime.performance_metrics.forecasting import MeanAbsolutePercentageError
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error
plt.rcParams["figure.figsize"] = (10, 5)

print(" All libraries imported successfully!")

In [ ]:
from sktime.forecasting.trend import PolynomialTrendForecaster
from sktime.transformations.series.detrend import Deseasonalizer, Detrender

In [ ]:
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe

In [ ]:
from keras.models import Sequential

In [ ]:
from keras.layers import Dense, LSTM, Dropout

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
!pip install keras-tuner --upgrade

In [ ]:
from kerastuner.tuners import RandomSearch
from kerastuner.engine.hyperparameters import HyperParameters

In [ ]:
pass  # Colab Drive import removed - place data files in ./data

The ensure_test_size function (below) generates a specified number of future periods with consistent frequency and format. It uses zeros as placeholders for unknown values. The function checks input data and ensures consistency.

In [ ]:
import pandas as pd
import numpy as np

def ensure_test_size(data, test_size, freq='W-SUN', date_column='Date', value_column='Volume'):
    """
    Ensure the test set has the specified size, padding with zeros if needed.

    Parameters:
    - data: DataFrame or Series with time series data
    - test_size: Desired number of periods (weeks or months)
    - freq: Frequency ('W-SUN' for weekly, 'M' for monthly)
    - date_column: Name of the date column
    - value_column: Name of the value column (e.g., 'Volume')

    Returns:
    - Series with exactly test_size rows and a DatetimeIndex
    """
    print(f"=== ensure_test_size called with test_size={test_size}, freq={freq}, date_column={date_column}, value_column={value_column} ===")
    print(f"Input data type: {type(data)}, shape: {data.shape}")
    print(f"Input data index type: {type(data.index)}")

    # Convert to dataframe if series
    if isinstance(data, pd.Series):
        print("Converting Series to DataFrame...")
        data = data.to_frame(name=value_column)
        data[date_column] = data.index
        data.reset_index(drop=True, inplace=True)

    print(f"Data head after conversion:\n{data.head()}")

    # Ensure date_column exists
    if date_column not in data.columns:
        raise ValueError(f"Date column '{date_column}' not found in data. Available columns: {data.columns}")

    # Ensure date_column is datetime
    print(f"Converting '{date_column}' to datetime...")
    data[date_column] = pd.to_datetime(data[date_column], errors='coerce')
    if data[date_column].isna().all():
        raise ValueError(f"All dates in '{date_column}' are invalid (NaT). Check input data.")

    # Get last valid date
    last_date = data[date_column].max()
    if pd.isna(last_date):
        raise ValueError(f"Last date in '{date_column}' is NaT. Check input data.")

    print(f"Last date in data: {last_date}")

    # Create test set dates (start one week after last_date)
    print("Generating test dates...")
    test_dates = pd.date_range(start=last_date + pd.Timedelta(days=7),
                               periods=test_size, freq=freq, name=date_column)

    # Debug: Print the first and last test dates
    print(f"Generated test dates: {test_dates[0]} to {test_dates[-1]}")
    print(f"Generated test dates length: {len(test_dates)}")

    # Create new test data with zeros
    print("Creating test data with zeros...")
    test_data = pd.DataFrame({
        date_column: test_dates,
        value_column: 0,  # Future data, set to 0
        'ISBN': data['ISBN'].iloc[0] if 'ISBN' in data.columns else None,
        'Title': data['Title'].iloc[0] if 'Title' in data.columns else None
    })

    # Convert to series with DatetimeIndex
    print("Converting test data to Series...")
    result = test_data.set_index(date_column)[value_column]

    # Validate the result
    if len(result) != test_size:
        raise ValueError(f"Test set length ({len(result)}) does not match test_size ({test_size})")
    if result.index.min() != last_date + pd.Timedelta(days=7):
        raise ValueError(f"Test set start date ({result.index.min()}) does not match expected ({last_date + pd.Timedelta(days=7)})")
    if result.index.max() != test_dates[-1]:
        raise ValueError(f"Test set end date ({result.index.max()}) does not match expected ({test_dates[-1]})")

    print(f"Output test set shape: {result.shape}, index type: {type(result.index)}")
    print(f"Test set date range: {result.index.min()} to {result.index.max()}")
    print("=== ensure_test_size completed ===")

    return result

In [ ]:
import inspect
print(inspect.getsource(ensure_test_size))

## **Section 1 - Part 2: Import the data set with the provided URL**

In [ ]:
# This code downloads excel files from my cloud storage, loads data from specified
# sheets (e.g., "F Adult Fiction"), and merges it into a cleaned dataframe
# The download_excel function saves files locally, normalize_sheet_name
# standardises sheet names, and load_data processes sheets, maps columns
# (e.g., Sales to Volume), adds missing columns, and ensures consistency

# URLs for excel files (already downloaded into my personal cloud storage)
urls = [
    'data/sales_file1.xlsx',
    'data/sales_file2.xlsx'
]

# Import required libraries
import requests
import pandas as pd
import re

# Download multiple excel files - Disclosure: parts of this code were generated with AI's help
def download_excel(urls, local_paths=['temp_excel_file1.xlsx', 'temp_excel_file2.xlsx']):
    """Download multiple Excel files from URLs to local paths."""
    local_files = []
    for url, local_path in zip(urls, local_paths):
        try:
            response = requests.get(url)
            response.raise_for_status()
            with open(local_path, 'wb') as f:
                f.write(response.content)
            local_files.append(local_path)
        except Exception as e:
            raise RuntimeError(f"Error downloading Excel file from {url}: {e}")
    return local_files

# Normalise sheet names to handle variations (e.g., "F - Adult Fiction" vs "F Adult Fiction")
def normalize_sheet_name(name):
    """Normalize sheet names by removing hyphens and extra spaces."""
    return re.sub(r'\s*-\s*', ' ', name.strip())

# Load data from multiple local files
def load_data(file_paths, tabs=['F Adult Fiction', 'S Adult Non-Fiction Specialist', 'T Adult Non-Fiction Trade', "Y Children's, YA & Educational"]):
    """Load data from all tabs of multiple local Excel files, clean it, and raise errors if issues occur."""
    dfs = []
    # Relaxed critical columns to allow partial matches
    expected_columns = {'ISBN', 'Title'}  # Minimal requirements
    optional_columns = {'Date', 'Volume'}  # Optional, will be filled if missing
    tab_mapping = {normalize_sheet_name(tab): tab for tab in tabs}

    for file_path in file_paths: # Disclosure: parts of the code below were created with AI's help.
        try:
            # Get actual sheet names
            xl = pd.ExcelFile(file_path, engine='openpyxl')
            available_sheets = xl.sheet_names
            normalized_available_sheets = [normalize_sheet_name(sheet) for sheet in available_sheets]
            print(f"Sheets in {file_path}: {available_sheets}")

            for tab in tabs:
                normalized_tab = normalize_sheet_name(tab)
                if normalized_tab not in normalized_available_sheets:
                    print(f"Warning: Sheet '{tab}' not found in {file_path}")
                    continue
                actual_sheet = available_sheets[normalized_available_sheets.index(normalized_tab)]
                try:
                    df = pd.read_excel(file_path, sheet_name=actual_sheet, engine='openpyxl')
                    print(f"Columns in {actual_sheet} ({file_path}): {list(df.columns)}")

                    # Map similar column names
                    if 'Interval' in df.columns and 'Date' not in df.columns:
                        df['Date'] = pd.to_datetime(df['Interval'], errors='coerce')
                    if 'Publication Date' in df.columns and 'Date' not in df.columns:
                        df['Date'] = pd.to_datetime(df['Publication Date'], errors='coerce')
                    if 'Value' in df.columns and 'Volume' not in df.columns:
                        df['Volume'] = df['Value']
                    if 'Sales' in df.columns and 'Volume' not in df.columns:
                        df['Volume'] = df['Sales']

                    # Check for critical columns
                    missing_critical = expected_columns - set(df.columns)
                    if missing_critical:
                        print(f"Warning: Sheet '{actual_sheet}' in {file_path} lacks critical columns {missing_critical}. Skipping.")
                        continue

                    # Add missing optional columns
                    for col in optional_columns:
                        if col not in df.columns:
                            df[col] = 0 if col == 'Volume' else pd.NaT
                            print(f"Added missing column '{col}' in {actual_sheet} ({file_path})")

                    df = df.dropna(how='all')  # Drop fully empty rows
                    df = df.fillna({'Volume': 0, 'Date': pd.NaT})  # Fill NaNs for optional columns
                    if 'Date' in df.columns:
                        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

                    dfs.append(df)
                except Exception as e:
                    print(f"Warning: Error loading sheet '{actual_sheet}' from {file_path}: {e}")
                    continue
        except Exception as e:
            print(f"Warning: Error accessing file {file_path}: {e}")
            continue

    if not dfs:
        raise ValueError("No valid data loaded from provided files")

    # Check column consistency
    first_columns = set(dfs[0].columns)
    valid_dfs = [dfs[0]]
    for i, df in enumerate(dfs[1:], 1):
        if set(df.columns) != first_columns:
            print(f"Warning: Sheet {i+1} has different columns ({df.columns}) than first sheet ({first_columns}). Skipping.")
            continue
        valid_dfs.append(df)

    try:
        return pd.concat(valid_dfs, ignore_index=True)
    except Exception as e:
        raise RuntimeError(f"Error concatenating dataframes: {e}")

# Download and load data
local_files = download_excel(urls)
tabs = ['F Adult Fiction', 'S Adult Non-Fiction Specialist', 'T Adult Non-Fiction Trade', "Y Children's, YA & Educational"]
data = load_data(local_files, tabs=tabs)

# Inspect the loaded data
print("Loaded data shape:", data.shape)
print("Columns:", data.columns)

In [ ]:
# Check the data
print("First few rows of the data set:")
print(data.head())

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
# Check for Duplicated data
data.duplicated().sum()

In [ ]:
# Mount cloud storage
pass  # drive.mount(...) removed for local execution

# Save the concatenated data set as a CSV file to cloud storage
file_path = './concatenated_data.csv'
data.to_csv(file_path, index=False)
print(f"Data set saved to {file_path}")

# Verify the saved file
try:
    saved_data = pd.read_csv(file_path)
    print(f"Verification: File loaded successfully with {len(saved_data)} rows and {len(saved_data.columns)} columns.")
except Exception as e:
    raise RuntimeError(f"Error verifying saved file: {e}")

## **Section 2 - Initial Data Investigation**

In [ ]:
## **Section 2 - Initial Data Investigation**

# Criteria (3–9) of total 37 criteria points

import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
pass  # Colab Drive import removed - place data files in ./data

# **Initial Data exploration**
# Check the combined data set
print("First few rows of the data set:")
print(data.head())

# Check data set shape
print("\nData set shape:", data.shape)

In [ ]:
# Drop 'Date' column - No good
# data = data.drop(columns=['Date'])

In [ ]:
# Check data set info
print("\nData set info:")
print(data.info())

In [ ]:
# Check for missing values
print("\nMissing values in each column:")
print(data.isna().sum())

Finding missing values in 'Author', 'ASP', and 'RRP' columns

In [ ]:
# Check for non-ASCII characters in string columns
print("\nColumns with non-ASCII characters:")
print(data.apply(lambda x: x.astype(str).str.contains(r'[^\x00-\x7F]').any()))

Found only 'Title' and 'Author' contain non-ASCII characters, which is expected.

In [ ]:
# Function to count non-ASCII characters
def get_non_ascii_chars(text):
    if isinstance(text, str):
        return [char for char in text if ord(char) > 127]
    return []

In [ ]:
def count_non_ascii_occurrences(df):
    non_ascii_counter = Counter()
    for column in df.select_dtypes(include='object').columns:
        for value in df[column]:
            non_ascii_counter.update(get_non_ascii_chars(value))
    non_ascii_counts = sorted(non_ascii_counter.items(), key=lambda x: x[1], reverse=True)
    if non_ascii_counts:
        print("Non-ASCII character occurrences:")
        total_occurrences = 0
        for char, count in non_ascii_counts:
            print(f"Character '{char}' (Unicode U+{ord(char):04X}): {count} times")
            total_occurrences += count
        print(f"\nTotal non-ASCII character occurrences: {total_occurrences}")
    else:
        print("No non-ASCII characters found.")
    return non_ascii_counts

In [ ]:
# Apply the function to the data set
non_ascii_counts = count_non_ascii_occurrences(data)

In [ ]:
# Clean non-ASCII characters from 'Title'
data['Title'] = data['Title'].apply(lambda x: ''.join(char for char in str(x) if ord(char) <= 127))

In [ ]:
# **Clean 'Volume', 'Value', and 'ASP' for consistency**
# Handle negative values in 'Volume' by setting them to NaN
data.loc[data['Volume'] < 0, 'Volume'] = None

In [ ]:
# Verify no negative 'Volume' values remain
assert len(data[data['Volume'] < 0]) == 0, "Negative 'Volume' values remain after cleaning."

In [ ]:
# If 'Volume' is 0 and 'Value' is not 0, set 'Value' to NaN
data.loc[(data['Volume'] == 0) & (data['Value'] != 0), 'Value'] = None

In [ ]:
# If 'Volume' is NaN, set 'Value' to NaN
data.loc[data['Volume'].isna(), 'Value'] = None

In [ ]:
# If 'Volume' is 0, set 'Value' to 0 (ensures consistency)
data.loc[data['Volume'] == 0, 'Value'] = 0

In [ ]:
# Handle negative 'Value' by setting to NaN
data.loc[data['Value'] < 0, 'Value'] = None

In [ ]:
# Verify no negative 'Volume' or 'Value' remain
assert len(data[(data['Volume'] < 0) | (data['Value'] < 0)]) == 0, "Negative 'Volume' or 'Value' values remain."

In [ ]:
# Handle negative 'ASP' by setting to NaN
data.loc[data['ASP'] < 0, 'ASP'] = None

In [ ]:
# Calculate 'ASP' safely, avoiding division by zero
data['ASP'] = data.apply(lambda x: round(x['Value'] / x['Volume'], 2) if x['Volume'] != 0 else 0, axis=1)
data['ASP'] = data['ASP'].fillna(0)

In [ ]:
# Verify no inconsistencies (e.g., 'Value' != 0 when 'Volume' == 0)
assert len(data[(data['Volume'] == 0) & (data['Value'] != 0)]) == 0, "Inconsistencies found: 'Value' != 0 when 'Volume' == 0."

In [ ]:
print("\nAfter cleaning, 'Volume', 'Value', and 'ASP' are mathematically consistent.")

In [ ]:
# **Criteria 4: Convert the ISBNs to a String Value**
# Combine conversion and cleaning into one step
data['ISBN'] = data['ISBN'].astype(str).str.replace(r'[^0-9]', '', regex=True)

# Filter data set for specific ISBNs to verify (for "Title A" and "Title B")
filtered_data_2_books = data[data['ISBN'].isin(['ISBN_OTHER', 'ISBN_TITLE_B', 'ISBN_TITLE_A'])]
print("\nFiltered dataset for 'Title A' and 'Title B':")
print(filtered_data_2_books)

In [ ]:
# **Criteria 5: Convert Interval to Datetime Object and Set as Index**
# Ensure 'Interval' is a string and validate format (YYYYWW)
data['Interval'] = data['Interval'].astype(str)
invalid_intervals = data[~data['Interval'].str.match(r'^\d{6}$', na=False)]
if not invalid_intervals.empty:
    print(f"Warning: {len(invalid_intervals)} invalid 'Interval' values found. Setting to NaT.")
    data.loc[~data['Interval'].str.match(r'^\d{6}$', na=False), 'Interval'] = None

In [ ]:
# Convert 'Interval' (YYYYWW) to datetime using ISO week numbers

# Converts a year-week column (Interval, formatted as YYYYWW)
# into datetime values representing the Monday of each ISO week (00–53, where weeks start on Monday).
# ensures the input is a string, uses pd.to_datetime with the format %Y%W-%w, and handles errors by setting invalid dates to NaT

try:
    data['YearWeek'] = data['Interval'].astype(str)
    data['Interval'] = pd.to_datetime(data['YearWeek'] + '-1', format='%Y%W-%w')  # Monday of each week
    print("Successfully converted 'Interval' to datetime.")
except Exception as e:
    print(f"Error converting 'Interval': {e}")
    data['Interval'] = pd.NaT

In [ ]:
# Drop rows with invalid dates
data = data.dropna(subset=['Interval'])
print(f"Rows after dropping invalid dates: {len(data)}")

In [ ]:
# Set 'Interval' as index
data.set_index('Interval', inplace=True)

In [ ]:
# Display updated data set info
print("\nUpdated dataset info after setting index:")
print(data.info())
print("\nFirst 20 rows of the dataset:")
print(data.head(20))

In [ ]:
# **Criteria 3: Resample the Data and Fill Missing Values with 0**
# Resample to weekly frequency, ensuring all weeks are represented with 0 sales for missing weeks
weekly_data = (
    data.groupby(['ISBN', 'Title'])
        .resample('W')
        .agg({'Volume': 'sum', 'Value': 'sum', 'Author': 'first', 'ASP': 'first'})
        .fillna({'Volume': 0, 'Value': 0, 'ASP': 0})
        .reset_index()
)

# Validate resampled data
print("\nWeekly Data Sample:")
print(weekly_data.head())
print(f"Unique ISBNs: {weekly_data['ISBN'].nunique()}")
print(f"Date Range: {weekly_data['Interval'].min()} to {weekly_data['Interval'].max()}")

# Check for missing weeks per ISBN
for isbn in weekly_data['ISBN'].unique():
    isbn_data = weekly_data[weekly_data['ISBN'] == isbn]
    expected_weeks = pd.date_range(start=isbn_data['Interval'].min(), end=isbn_data['Interval'].max(), freq='W')
    if len(isbn_data) != len(expected_weeks):
        print(f"Warning: ISBN {isbn} has missing weeks after resampling.")

The code above resamples the book sales data set to a weekly frequency, ensuring all weeks are represented and missing sales are zero-filled. It aggregates Volume, Value, Author and ASP and includes validation to confirm data integrity.

In [ ]:
# **Criteria 6: Filter ISBNs with Sales Beyond 2024-07-01**
filtered_data = weekly_data[weekly_data['Interval'] > '2024-07-01']
if filtered_data.empty:
    print("No sales data beyond 2024-07-01.")
else:
    isbn_title_map = filtered_data[['ISBN', 'Title']].drop_duplicates()
    print(f"ISBNs with sales beyond 2024-07-01 ({len(isbn_title_map)}):")
    for _, row in isbn_title_map.iterrows():
        print(f"ISBN: {row['ISBN']}, Title: {row['Title']}")

61 titles, including "Title A" and "Title B".

In [ ]:
# **Criteria 7: Plot the Data of All Filtered ISBNs**

# This section creats charts showing sales trends for the chosen books
# using data after 1 January 2024. It focuses on the books' ISBNs,
# smooths sales data with a 2-week average, and displays the results in a grid
# of plots with a reference line at 1 July 2024.

# Extending date range to last 6 months for better trends
filtered_data = weekly_data[weekly_data['Interval'] > '2024-01-01']
if filtered_data['Interval'].min() <= pd.to_datetime('2024-07-01'):
    print("Note: Including data from 2024-01-01 for better trend visualisation.")

if not filtered_data.empty:
    # Get unique ISBNs
    isbns = filtered_data['ISBN'].unique()
    print(f"Total ISBNs to plot: {len(isbns)}")

    # Prioritise focus books and select a subset for clarity
    focus_isbns = ['ISBN_TITLE_A', 'ISBN_TITLE_B', 'ISBN_OTHER']  # Title A, and Title B
    additional_isbns = ['ISBN_OTHER', 'ISBN_OTHER', 'ISBN_OTHER']  # three further titles for comparison
    subset_isbns = focus_isbns + additional_isbns

    # Filter to subset for plotting
    isbns_to_plot = [isbn for isbn in isbns if isbn in subset_isbns]
    print(f"ISBNs selected for plotting: {isbns_to_plot}")

    # Set up subplot grid
    n_isbns = len(isbns_to_plot)
    ncols = 2
    nrows = (n_isbns // ncols) + (1 if n_isbns % ncols != 0 else 0)
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows), sharex=True)
    axes = axes.flatten()

    # Plot each ISBN
    for i, isbn in enumerate(isbns_to_plot):
        isbn_data = filtered_data[filtered_data['ISBN'] == isbn].sort_values('Interval')
        print(f"ISBN {isbn}: {isbn_data['Interval'].min()} to {isbn_data['Interval'].max()}")

        # Use a 2-week rolling average due to shorter date range
        isbn_data['Volume_Smoothed'] = isbn_data['Volume'].rolling(window=2, min_periods=1).mean()

        axes[i].plot(isbn_data['Interval'], isbn_data['Volume_Smoothed'], label=f'ISBN: {isbn}', color='blue')
        axes[i].set_title(f'ISBN: {isbn}')
        axes[i].set_xlabel('Date')
        axes[i].set_ylabel('Volume (Smoothed)')
        axes[i].legend()
        axes[i].grid(True, linestyle='--', alpha=0.7)
        axes[i].axvline(pd.to_datetime('2024-07-01'), color='red', linestyle='--', label='2024-07-01', alpha=0.5)
        axes[i].legend()

    # Hide empty subplots
    for i in range(len(isbns_to_plot), len(axes)):
        axes[i].set_visible(False)

    plt.tight_layout()
    plt.suptitle('Recent Sales Patterns for Selected ISBNs (Smoothed)', y=1.02)
    plt.subplots_adjust(top=0.95)

    output_path = './recent_sales_patterns_subset.png'
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Plot saved to {output_path}")
    plt.show()
else:
    print("No sales data to plot beyond 2024-01-01.")

In [ ]:
# **Criteria 8: Investigate Sales Patterns and Compare Periods (2000–2012 vs. 2012–2024)**

# Here the code compares book sales patterns in weekly_data between 2000–2012 (Period 1)
# and 2012–2024 (Period 2). It verifies the date range, splits the data at 1 January 2012
# (upper bound 1 July 2024 for Period 2), and calculates summary statistics for total
# sales volumes per ISBN in each period. Grouping by ISBN, summing
# volumes, and using .describe() to show sales distributions.

# Verify actual date range
print(f"\nData Range: {weekly_data['Interval'].min()} to {weekly_data['Interval'].max()}")

# Define periods
period_1 = weekly_data[(weekly_data['Interval'] >= '2000-01-01') & (weekly_data['Interval'] < '2012-01-01')]
period_2 = weekly_data[(weekly_data['Interval'] >= '2012-01-01') & (weekly_data['Interval'] <= '2024-07-01')]

# Summary statistics
print("Period 1 (2000-2012) Stats:")
print(period_1.groupby('ISBN')['Volume'].sum().describe())
print("\nPeriod 2 (2012-2024) Stats:")
print(period_2.groupby('ISBN')['Volume'].sum().describe())

In [ ]:
# Plot top 5 ISBNs for each period
top_isbns_p1 = period_1.groupby('ISBN')['Volume'].sum().nlargest(5).index
top_isbns_p2 = period_2.groupby('ISBN')['Volume'].sum().nlargest(5).index

plt.figure(figsize=(12, 8))
focus_books = ['ISBN_TITLE_A', 'ISBN_TITLE_B']  # Title A, Title B

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Initialise plot with larger figure size
plt.figure(figsize=(14, 8), facecolor='white')

# Plot Period 1 (2000-2012)
for isbn in top_isbns_p1:
    isbn_data = period_1[period_1['ISBN'] == isbn].sort_values('Interval')
    if isbn_data.empty:
        print(f"Warning: No data found for ISBN {isbn} in period_1")
        continue
    print(f"Plotting ISBN {isbn} with {len(isbn_data)} data points")  # Debug output
    isbn_data['Volume_Smoothed'] = isbn_data['Volume'].rolling(window=4, min_periods=1).mean() + 1e-6  # Small constant for log scale
    if isbn == 'ISBN_TITLE_A':  # Title A
        linestyle, color, label = '-', 'green', 'Title A (2000-2012)'
        linewidth = 2.5  # Thicker line
        alpha = 1.0  # Full opacity
    elif isbn == 'ISBN_TITLE_B':  # Title B
        linestyle, color, label = '-', 'red', 'Title B (2000-2012)'
        linewidth = 2.5
        alpha = 1.0
    else:
        linestyle, color, label = '--', 'blue', f"ISBN {isbn} (2000-2012)"
        linewidth = 1.0
        alpha = 0.5  # Lower opacity for non-focus books
    plt.plot(isbn_data['Interval'], isbn_data['Volume_Smoothed'], label=label, linestyle=linestyle,
             color=color, linewidth=linewidth, alpha=alpha)

# Add period divider
plt.axvline(pd.to_datetime('2012-01-01'), color='black', linestyle='--', label='Period Divide (2012-01-01)',
            alpha=0.7, linewidth=1.5)

# Annotations for major peaks
# Title A
if 'ISBN_TITLE_A' in period_1['ISBN'].values:
    isbn_data_title_a = period_1[period_1['ISBN'] == 'ISBN_TITLE_A'].sort_values('Interval')
    isbn_data_title_a['Volume_Smoothed'] = isbn_data_title_a['Volume'].rolling(window=4, min_periods=1).mean() + 1e-6
    if not isbn_data_title_a.empty:
        peak_date_title_a = isbn_data_title_a['Interval'][isbn_data_title_a['Volume_Smoothed'].idxmax()]
        peak_value_title_a = isbn_data_title_a['Volume_Smoothed'].max()
        plt.annotate(f'Title_A Peak (~{int(peak_value_title_a):,d} units)',
                     xy=(peak_date_title_a, peak_value_title_a),
                     xytext=(peak_date_title_a + pd.Timedelta(days=180), peak_value_title_a * 0.8),  # Dynamic offset
                     arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5),
                     fontsize=12, color='black', bbox=dict(facecolor='white', alpha=0.8, edgecolor='black', boxstyle='round,pad=0.3'))

# Title B
if 'ISBN_TITLE_B' in period_1['ISBN'].values:
    isbn_data_title_b = period_1[period_1['ISBN'] == 'ISBN_TITLE_B'].sort_values('Interval')
    isbn_data_title_b['Volume_Smoothed'] = isbn_data_title_b['Volume'].rolling(window=4, min_periods=1).mean() + 1e-6
    if not isbn_data_title_b.empty:
        peak_date_title_b = isbn_data_title_b['Interval'][isbn_data_title_b['Volume_Smoothed'].idxmax()]
        peak_value_title_b = isbn_data_title_b['Volume_Smoothed'].max()
        plt.annotate(f'Title B Peak (~{int(peak_value_title_b):,d} units)',
                     xy=(peak_date_title_b, peak_value_title_b),
                     xytext=(peak_date_title_b + pd.Timedelta(days=180), peak_value_title_b * 1.2),  # Dynamic offset
                     arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5),
                     fontsize=12, color='black', bbox=dict(facecolor='white', alpha=0.8, edgecolor='black', boxstyle='round,pad=0.3'))

# Apply logarithmic scale
plt.yscale('log')
plt.ylim(bottom=1)
plt.yticks([1, 10, 100, 1000, 10000, 100000], ['1', '10', '100', '1K', '10K', '100K'], fontsize=10)

# Customise plot
plt.grid(True, linestyle='-', alpha=0.7, which='both', color='gray')  # Bolder grid
plt.title('Sales Comparison: Title A & Title B (2000-2012, Smoothed, Log Scale)',
          fontsize=14, pad=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Volume (Log Scale)', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10, frameon=True, edgecolor='black')
plt.tight_layout()

# Save plot
output_path = './sales_comparison_title_a_title_b_enhanced.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Plot saved to {output_path}")
plt.show()

In [ ]:
# Initialise plot
plt.figure(figsize=(14, 8), facecolor='white')  #

# Plot Period 2 (2012-2024)
for isbn in top_isbns_p2:
    isbn_data = period_2[period_2['ISBN'] == isbn].sort_values('Interval')
    if isbn_data.empty:
        print(f"Warning: No data found for ISBN {isbn} in period_2")
        continue
    print(f"Plotting ISBN {isbn} with {len(isbn_data)} data points")  # Debug output
    isbn_data['Volume_Smoothed'] = isbn_data['Volume'].rolling(window=4, min_periods=1).mean() + 1e-6  # Small constant for log scale
    if isbn == 'ISBN_TITLE_A':  # Title A
        linestyle, color, label = '-', 'green', 'Title A (2012-2024)'
        linewidth = 2.5  # Thicker line
        alpha = 1.0  # Full opacity
    elif isbn == 'ISBN_TITLE_B':  # Title B
        linestyle, color, label = '-', 'red', 'Title B (2012-2024)'
        linewidth = 2.5
        alpha = 1.0
    else:
        linestyle, color, label = '--', 'blue', f"ISBN {isbn} (2012-2024)"
        linewidth = 1.0
        alpha = 0.5  # Lower opacity for non-focus books
    plt.plot(isbn_data['Interval'], isbn_data['Volume_Smoothed'], label=label, linestyle=linestyle,
             color=color, linewidth=linewidth, alpha=alpha)

# Add period divider
plt.axvline(pd.to_datetime('2012-01-01'), color='black', linestyle='--', label='Period Divide (2012-01-01)',
            alpha=0.7, linewidth=1.5)

# Annotations
# Title A
if 'ISBN_TITLE_A' in period_2['ISBN'].values:
    isbn_data_title_a = period_2[period_2['ISBN'] == 'ISBN_TITLE_A'].sort_values('Interval')
    isbn_data_title_a['Volume_Smoothed'] = isbn_data_title_a['Volume'].rolling(window=4, min_periods=1).mean() + 1e-6
    if not isbn_data_title_a.empty:
        peak_date_title_a = isbn_data_title_a['Interval'][isbn_data_title_a['Volume_Smoothed'].idxmax()]
        peak_value_title_a = isbn_data_title_a['Volume_Smoothed'].max()
        plt.annotate(f'Title_A Peak (~{int(peak_value_title_a):,d} units)',
                     xy=(peak_date_title_a, peak_value_title_a),
                     xytext=(peak_date_title_a + pd.Timedelta(days=180), peak_value_title_a * 0.8),  # Dynamic offset
                     arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5),
                     fontsize=12, color='black',
                     bbox=dict(facecolor='white', alpha=0.8, edgecolor='black', boxstyle='round,pad=0.3'))

# Title B
if 'ISBN_TITLE_B' in period_2['ISBN'].values:
    isbn_data_title_b = period_2[period_2['ISBN'] == 'ISBN_TITLE_B'].sort_values('Interval')
    isbn_data_title_b['Volume_Smoothed'] = isbn_data_title_b['Volume'].rolling(window=4, min_periods=1).mean() + 1e-6
    if not isbn_data_title_b.empty:
        peak_date_title_b = isbn_data_title_b['Interval'][isbn_data_title_b['Volume_Smoothed'].idxmax()]
        peak_value_title_b = isbn_data_title_b['Volume_Smoothed'].max()
        plt.annotate(f'Title B Peak (~{int(peak_value_title_b):,d} units)',
                     xy=(peak_date_title_b, peak_value_title_b),
                     xytext=(peak_date_title_b + pd.Timedelta(days=180), peak_value_title_b * 1.2),  # Dynamic offset
                     arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5),
                     fontsize=12, color='black',
                     bbox=dict(facecolor='white', alpha=0.8, edgecolor='black', boxstyle='round,pad=0.3'))

# Apply logarithmic scale
plt.yscale('log')
plt.ylim(bottom=1)
plt.yticks([1, 10, 100, 1000, 10000, 100000], ['1', '10', '100', '1K', '10K', '100K'], fontsize=10)

# Customise plot
plt.grid(True, linestyle='-', alpha=0.7, which='both', color='gray')  # Bolder grid
plt.title('Sales Comparison: Title A & Title B (2012-2024, Smoothed, Log Scale)\n'
          '(Note: No Digital Sales Tracked in this Plot)', fontsize=14, pad=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Volume (Log Scale)', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10, frameon=True, edgecolor='black')
plt.tight_layout()

# Save plot
output_path = './sales_comparison_title_a_title_b_p2_enhanced.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Plot saved to {output_path}")
plt.show()

In [ ]:
# **Criteria 9: Select Two Books for Further Analysis Post-2012-01-01**
two_books_analysis = weekly_data[weekly_data['Title'].str.lower().isin(['very hungry title_b, the', 'title_a, the'])]
two_books_analysis = two_books_analysis[two_books_analysis['Interval'] > '2012-01-01']

# Validate filtered data
print("\nTwo Books Analysis Data:")
print(two_books_analysis.groupby('Title')['Interval'].agg(['min', 'max', 'count']))

In [ ]:
# Plot smoothed sales
plt.figure(figsize=(12, 8))
for title in two_books_analysis['Title'].unique():
    book_data = two_books_analysis[two_books_analysis['Title'] == title].sort_values('Interval')
    book_data['Volume_Smoothed'] = book_data['Volume'].rolling(window=4, min_periods=1).mean()
    plt.plot(book_data['Interval'], book_data['Volume_Smoothed'], label=title)
plt.title('Weekly Sales for Title A and Title B (>2012-01-01, Smoothed)')
plt.xlabel('Date')
plt.ylabel('Volume')
plt.legend()
plt.show()

In [ ]:
# Save filtered data to cloud storage (avoid redundant mount)
file_path = './two_books_analysis.csv'
two_books_analysis.to_csv(file_path, index=False)
print(f"Two books dataset saved to {file_path}")

## **Section 3 - Classical techniques**

In [ ]:
## **Section 3 - Classical Techniques**

### Criteria 10–17

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
import os
import logging
warnings.filterwarnings('ignore')

# Set up logging for debugging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
# Mount cloud storage
pass  # Colab Drive import removed - place data files in ./data
try:
    pd.read_csv('./two_books_analysis.csv', nrows=1)
except:
    pass  # drive.mount(...) removed for local execution

# Load and preprocess data
data_path = './two_books_analysis.csv'
try:
    data = pd.read_csv(data_path)
    data['Interval'] = pd.to_datetime(data['Interval'])
    data.set_index('Interval', inplace=True)
except FileNotFoundError:
    logger.error("Data file not found.")
    raise
except Exception as e:
    logger.error(f"Error loading data: {e}")
    raise

In [ ]:
# Extract series for both books
title_a_data = data[data['Title'].str.lower() == 'title_a, the']['Volume']
title_b_data = data[data['Title'].str.lower() == 'very hungry title_b, the']['Volume']

# Validate data
if title_a_data.empty or title_b_data.empty:
    logger.error("Data for one or both books is missing.")
    raise ValueError("Missing data for Title A or Title B.")
if title_a_data.isna().any() or title_b_data.isna().any():
    logger.warning("Missing values detected. Interpolating...")
    title_a_data = title_a_data.interpolate()
    title_b_data = title_b_data.interpolate()

In [ ]:
title_a_data.head()

In [ ]:
title_b_data.head()

In [ ]:
title_b_data.tail()

In [ ]:
# Preprocess Title_B data to handle sparsity
title_b_data = title_b_data.replace(1.0, np.nan).interpolate() # with this approach title_b_data has its sparse 1.0 values
# replaced with interpolated values, creating a smoother and more continuous data set

# Smooth data to reduce volatility
title_b_data = title_b_data.rolling(window=3, min_periods=1).mean().fillna(title_b_data)
# The title_b_data is now smoothed, with short-term fluctuations reduced, making it easier to identify longer-term trends

# Define forecast horizon
forecast_horizon = 32  # As specified in Criteria 13, 16

# Ensure data is sorted and has DatetimeIndex
title_a_data = title_a_data.sort_index()
title_b_data = title_b_data.sort_index()
if not isinstance(title_a_data.index, pd.DatetimeIndex) or not isinstance(title_b_data.index, pd.DatetimeIndex):
    raise ValueError("Data must have a DatetimeIndex")

In [ ]:
title_b_data.tail()

In [ ]:
title_b_data.head()

In [ ]:
# Define training and test sets (final 32 weeks for testing)
title_a_train = title_a_data.iloc[:-forecast_horizon]
title_a_test = title_a_data.iloc[-forecast_horizon:]
title_b_train = title_b_data.iloc[:-forecast_horizon]
title_b_test = title_b_data.iloc[-forecast_horizon:]

In [ ]:
# Verify data lengths and ranges of the training and testing sets for Title A and Title B
logger.info("=== Train-Test Split ===")
logger.info(f"Title_A Train Length: {len(title_a_train)}, Range: {title_a_train.index.min()} to {title_a_train.index.max()}")
logger.info(f"Title_A Test Length: {len(title_a_test)}, Range: {title_a_test.index.min()} to {title_a_test.index.max()}")
logger.info(f"Title_B Train Length: {len(title_b_train)}, Range: {title_b_train.index.min()} to {title_b_train.index.max()}")
logger.info(f"Title_B Test Length: {len(title_b_test)}, Range: {title_b_test.index.min()} to {title_b_test.index.max()}")

In [ ]:
print(f"Title_A Train Length: {len(title_a_train)}, Range: {title_a_train.index.min()} to {title_a_train.index.max()}")

In [ ]:
print(f"Title_A Test Length: {len(title_a_test)}, Range: {title_a_test.index.min()} to {title_a_test.index.max()}")

In [ ]:
print(f"Title_B Train Length: {len(title_b_train)}, Range: {title_b_train.index.min()} to {title_b_train.index.max()}")

In [ ]:
print(f"Title_B Test Length: {len(title_b_test)}, Range: {title_b_test.index.min()} to {title_b_test.index.max()}")

In [ ]:
# Criteria 10: Decomposition
# This function decomposes a time series into its trend, seasonal, and residual
# components using the seasonal_decompose function from the statsmodels library
def perform_decomposition(series, title, period=52):
    logger.info(f"\n=== {title} Decomposition ===") # The log output will include a clear section header

    try:
        # Check for zero or negative values
        if (series <= 0).any(): # If zero or negative values are present, the
        # function defaults to an additive model, which models the series as = trend + seasonal + residual
            model_choice = 'additive'
            decomposition = seasonal_decompose(series, model='additive', period=period, extrapolate_trend='freq')
        else:
            additive = seasonal_decompose(series, model='additive', period=period, extrapolate_trend='freq')
            adjusted_series = series + 1
            multiplicative = seasonal_decompose(adjusted_series, model='multiplicative', period=period, extrapolate_trend='freq')
            model_choice = 'multiplicative' if multiplicative.resid.std() < additive.resid.std() else 'additive'
            decomposition = multiplicative if model_choice == 'multiplicative' else additive
        # For positive series, the function chooses the decomposition model
        # (additive or multiplicative) that best fits the data, based on the residual standard deviation

        # Plot components
        fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
        axes[0].plot(series, label='Original')
        axes[0].set_title(f'{title} Decomposition ({model_choice.capitalize()})')
        axes[1].plot(decomposition.trend, label='Trend', color='orange')
        axes[2].plot(decomposition.seasonal, label='Seasonal', color='green')
        axes[3].plot(decomposition.resid, label='Residual', color='red')
        for ax in axes:
            ax.legend()
        plt.tight_layout()
        plt.savefig(f'./{title.replace(" ", "_")}_decomposition.png', dpi=300)
        plt.show()

        # Component characteristics
        # Computes and prints key characteristics of the decomposition components to interpret the results
        trend_slope = (decomposition.trend[-1] - decomposition.trend[0]) / len(decomposition.trend) * 52
        seasonal_amplitude = decomposition.seasonal.max() - decomposition.seasonal.min()
        resid_std = decomposition.resid.std()
        print(f"Component Characteristics for {title}:")
        print(f"- Model: {model_choice.capitalize()}")
        print(f"- Trend Slope: {trend_slope:.2f} units/year (indicates {'increasing' if trend_slope > 0 else 'decreasing or stable'} sales trend)")
        print(f"- Seasonal Amplitude: {seasonal_amplitude:.2f} units (larger values indicate stronger seasonality)")
        print(f"- Residual Std: {resid_std:.2f} (lower values suggest better decomposition fit)")
        if model_choice == 'additive':
            print("- Additive model chosen due to consistent seasonal fluctuations.")
        else:
            print("- Multiplicative model chosen as seasonality scales with the trend.")
        return model_choice, decomposition
    except Exception as e:
        logger.error(f"Error in decomposition for {title}: {e}")
        return 'error', None

In [ ]:
# Criteria 11: ACF and PACF
def plot_acf_pacf(series, title, lags=52):
    logger.info(f"\n=== {title} ACF/PACF Analysis ===")
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    plot_acf(series, ax=ax1, lags=lags, title=f'{title} ACF')
    plot_pacf(series, ax=ax2, lags=lags, title=f'{title} PACF')
    plt.tight_layout()
    plt.savefig(f'./{title.replace(" ", "_")}_acf_pacf.png', dpi=300)
    plt.show()

    # Identify significant lags
    acf_conf_int = 1.96 / np.sqrt(len(series))
    acf_values = acf(series, nlags=lags)
    pacf_values = pacf(series, nlags=lags)
    significant_acf_lags = [i for i, val in enumerate(acf_values[1:], start=1) if abs(val) > acf_conf_int]
    significant_pacf_lags = [i for i, val in enumerate(pacf_values[1:], start=1) if abs(val) > acf_conf_int]
    print(f"ACF/PACF Analysis for {title}:")
    print(f"- Significant ACF Lags: {significant_acf_lags} (suggest MA terms or seasonality at lag 52)")
    print(f"- Significant PACF Lags: {significant_pacf_lags} (suggest AR terms)")
    print("- Lag 52 prominence indicates yearly seasonality, guiding seasonal ARIMA parameters.")

In [ ]:
# Criteria 12: Stationarity Check
#  Checks if a time series is stationary using ADF and KPSS tests, applies
# differencing (up to order 2) if needed, and determines the differencing order (d)

def check_stationarity(series, title):
    logger.info(f"\n=== {title} Stationarity Check ===")
    result = adfuller(series, autolag='AIC')
    kpss_result = kpss(series, regression='c')
    print(f"ADF Statistic: {result[0]:.4f}, p-value: {result[1]:.4f}")
    print(f"KPSS Statistic: {kpss_result[0]:.4f}, p-value: {kpss_result[1]:.4f}")

    d = 0
    if result[1] >= 0.05 or kpss_result[1] <= 0.05:
        diff_series = series.diff().dropna()
        diff_result = adfuller(diff_series, autolag='AIC')
        print(f"After First Differencing - ADF Statistic: {diff_result[0]:.4f}, p-value: {diff_result[1]:.4f}")
        d = 1 if diff_result[1] < 0.05 else 2
        if d == 2:
            diff_series_2 = diff_series.diff().dropna()
            diff_result_2 = adfuller(diff_series_2, autolag='AIC')
            print(f"After Second Differencing - ADF Statistic: {diff_result_2[0]:.4f}, p-value: {diff_result_2[1]:.4f}")

    print(f"Stationarity Result for {title}: Differencing order = {d}")
    print(f"- {'Stationary' if d == 0 else 'Non-stationary, required differencing'}. {'Stable mean and variance' if d == 0 else 'Trend or variance issues resolved by differencing'}.")
    return series if d == 0 else diff_series, d

In [ ]:
# Criteria 13–14: Auto ARIMA and Best Model
# Fits a seasonal ARIMA model to the time series using auto_arima, automatically
# selecting the best parameters based on AIC, with a fallback to a simple ARIMA(1,0,0) if needed

def run_auto_arima(series, title, d_value, forecast_horizon=32):
    logger.info(f"\n=== {title} Auto ARIMA ===")
    try:
        model = auto_arima(
            series,
            seasonal=True,
            m=52,  # Yearly seasonality
            d=d_value,
            start_p=0, max_p=2,
            start_q=0, max_q=2,
            start_P=0, max_P=1,
            start_Q=0, max_Q=1,
            stepwise=True,
            max_order=10,
            maxiter=50,
            suppress_warnings=True,
            trace=True,
            error_action='ignore',
            random_state=42
        )
        print(f"Best Model for {title}:")
        print(model.summary())
        print(f"- Order: {model.order}, Seasonal Order: {model.seasonal_order}, AIC: {model.aic():.2f}")
        print(f"- Model captures {'yearly seasonality' if model.seasonal_order != (0,0,0,52) else 'trend and short-term dynamics'}, with low AIC indicating good fit.")
        return model
    except Exception as e:
        logger.error(f"Auto ARIMA failed for {title}: {e}. Falling back to ARIMA(1,0,0)...")
        model = auto_arima(series, seasonal=False, d=d_value, start_p=1, max_p=1, start_q=0, max_q=0, stepwise=False)
        print(f"Fallback Model for {title}: {model.order}")
        return model

In [ ]:
# Criteria 15: Residual Analysis
#  Analyses the residuals of the fitted time series model to evaluate its performance
def analyse_residuals(model, series, title):
    logger.info(f"\n=== {title} Residual Analysis ===")
    residuals = model.resid()
    plt.figure(figsize=(12, 6))
    plt.subplot(211)
    plt.plot(residuals, label='Residuals')
    plt.axhline(0, color='red', linestyle='--')
    plt.title(f'{title} Residuals')
    plt.legend()
    plt.subplot(212)
    plot_acf(residuals, lags=52, ax=plt.gca())
    plt.title(f'ACF of {title} Residuals')
    plt.tight_layout()
    plt.savefig(f'./{title.replace(" ", "_")}_residuals.png', dpi=300)
    plt.show()

    acf_conf_int = 1.96 / np.sqrt(len(residuals))
    acf_values = acf(residuals, nlags=52)
    significant_lags = [i for i, val in enumerate(acf_values[1:], start=1) if abs(val) > acf_conf_int]
    print(f"Residual Analysis for {title}:")
    print(f"- Mean: {residuals.mean():.4f} (close to 0 indicates good fit)")
    print(f"- Std: {residuals.std():.4f} (lower values suggest better model)")
    print(f"- Significant ACF Lags: {significant_lags} ({'no autocorrelation' if not significant_lags else 'potential unmodeled patterns'})")

In [ ]:
# Criteria 16–17: Forecast and Plot
# Generates a 32-week forecast using the fitted model, evaluates its accuracy
# against test data, plots the results, and returns the forecast and related data

# Disclosure: The code below was created with AI's assistance

def forecast_and_plot(model, train, test, title, forecast_horizon=32):
    logger.info(f"\n=== {title} Forecasting ===")
    try:
        # Generate forecast and confidence intervals
        forecast, conf_int = model.predict(n_periods=forecast_horizon, return_conf_int=True)
        forecast_index = test.index
        if len(forecast) != len(forecast_index):
            logger.warning(f"Length mismatch: forecast ({len(forecast)}) vs index ({len(forecast_index)}). Truncating...")
            forecast_index = forecast_index[:len(forecast)]

        forecast_series = pd.Series(forecast, index=forecast_index, dtype=float)
        conf_int_df = pd.DataFrame(conf_int, index=forecast_index, columns=['lower', 'upper'], dtype=float)

        # Handle NaNs
        if forecast_series.isna().any():
            logger.warning("NaNs in forecast. Filling with last valid value...")
            forecast_series = forecast_series.fillna(train[-10:].mean())
        if conf_int_df.isna().any().any():
            logger.warning("NaNs in confidence intervals. Filling with forecast ± std...")
            conf_int_df = conf_int_df.fillna({
                'lower': forecast_series - train.std(),
                'upper': forecast_series + train.std()
            })

        # Ensure non-negative forecasts
        forecast_series = forecast_series.clip(lower=0)
        conf_int_df['lower'] = conf_int_df['lower'].clip(lower=0)
        conf_int_df['upper'] = conf_int_df['upper'].clip(lower=0)

        # Compute metrics
        mae = mean_absolute_error(test, forecast_series)
        rmse = np.sqrt(mean_squared_error(test, forecast_series))

        # Plotting
        plt.figure(figsize=(12, 6))
        plt.plot(train.index, train, label='Training Data', color='blue')
        plt.plot(test.index, test, label='Actual Test Data', color='green')
        plt.plot(forecast_series.index, forecast_series, label='Forecast', color='red')
        plt.fill_between(forecast_index, conf_int_df['lower'], conf_int_df['upper'],
                         color='pink', alpha=0.3, label='95% Confidence Interval')
        plt.legend()
        plt.title(f'{title} - 32-Week Forecast')
        plt.xlabel('Date')
        plt.ylabel('Sales Volume')
        plt.grid(True)
        plt.savefig(f'./{title.replace(" ", "_")}_forecast.png', dpi=300)
        plt.show()

        # Forecast evaluation
        print(f"Forecast Evaluation for {title}:")
        print(f"- MAE: {mae:.2f} (average prediction error)")
        print(f"- RMSE: {rmse:.2f} (penalizes larger errors)")
        print(f"- {'Reasonable accuracy' if mae < train.std() else 'High errors, possibly due to data volatility or spikes'}.")

        # Create forecast DataFrame
        forecast_df = pd.DataFrame({
            'Date': forecast_index,
            'Forecast': forecast_series,
            'Lower_CI': conf_int_df['lower'],
            'Upper_CI': conf_int_df['upper'],
            'Actual': test
        })
        return forecast_series, conf_int_df, forecast_df
    except Exception as e:
        logger.error(f"Error in forecast_and_plot for {title}: {e}")
        return None, None, None

# # Initialise results dictionary
# results = {
#     'Book': [],
#     'Decomposition_Model': [],
#     'Differencing_Order': [],
#     'ARIMA_Order': [],
#     'Seasonal_Order': [],
#     'AIC': [],
#     'Residual_Mean': [],
#     'Residual_Std': [],
#     'MAE': [],
#     'RMSE': []
# }

In [ ]:
# To avoid the error
from tabulate import tabulate

# Define a function to calculate MAPE
def calculate_mape(actual, forecast):
    # Avoid division by zero by filtering out actual values <= 0
    valid_mask = actual > 0
    actual = actual[valid_mask]
    forecast = forecast[valid_mask]
    if len(actual) == 0:
        logger.warning("MAPE undefined: No valid actual values (all zeros or negative).")
        return np.nan  # Return NaN if no valid data
    absolute_percentage_error = np.abs((actual - forecast) / actual) * 100
    mape = np.mean(absolute_percentage_error)
    logger.info(f"MAPE calculated: {mape:.2f}%")
    return mape

# Initialise results dictionary with MAPE
results = {
    'Book': [],
    'Decomposition_Model': [],
    'Differencing_Order': [],
    'ARIMA_Order': [],
    'Seasonal_Order': [],
    'AIC': [],
    'Residual_Mean': [],
    'Residual_Std': [],
    'classical_MAE': [],
    'classical_MAPE': []  # Added MAPE to results
}

# Dictionary to store forecast DataFrames for both books
forecast_dfs = {}

# Run analysis for both books
for series, test, title in [(title_a_train, title_a_test, "Title A"),
                           (title_b_train, title_b_test, "Title B")]:
    # Criteria 10: Decomposition
    decomp_model, decomp = perform_decomposition(series, title)

    # Criteria 11: ACF/PACF
    plot_acf_pacf(series, title)

    # Criteria 12: Stationarity
    _, d = check_stationarity(series, title)

    # Criteria 13–14: Auto ARIMA
    model = run_auto_arima(series, title, d)

    if model:
        # Criteria 15: Residuals
        analyse_residuals(model, series, title)

        # Criteria 16–17: Forecast and Plot
        forecast_series, conf_int_df, forecast_df = forecast_and_plot(model, series, test, title, forecast_horizon)

        # Store results
        if forecast_series is not None:
            # Calculate MAPE
            mape = calculate_mape(test, forecast_series)

            # Append metrics to results
            results['Book'].append(title)
            results['Decomposition_Model'].append(decomp_model)
            results['Differencing_Order'].append(d)
            results['ARIMA_Order'].append(model.order)
            results['Seasonal_Order'].append(model.seasonal_order)
            results['AIC'].append(model.aic())
            results['Residual_Mean'].append(model.resid().mean())
            results['Residual_Std'].append(model.resid().std())
            results['classical_MAE'].append(mean_absolute_error(test, forecast_series))
            results['classical_MAPE'].append(mape)  # Store MAPE

            # Store forecast DataFrame with renamed columns
            forecast_df['Book'] = title  # Add Book column
            forecast_df = forecast_df.rename(columns={
                'Forecast': 'classical_Forecast',
                'Lower_CI': 'classical_Lower_CI',
                'Upper_CI': 'classical_Upper_CI'
            })[['Date', 'Book', 'classical_Forecast', 'classical_Lower_CI', 'classical_Upper_CI', 'Actual']]
            forecast_dfs[title] = forecast_df

# Combine forecast DataFrames for both books
forecasts_df = pd.concat(list(forecast_dfs.values()), ignore_index=True)

# Save results to classical_techniques_results.xlsx
try:
    # Create Metrics DataFrame
    results_df = pd.DataFrame(results)

    # Print the results table (using tabulate, now that it's imported)
    table = tabulate(results_df, headers='keys', tablefmt='psql', showindex=False)
    print("\nCompiled Results Table:")
    print(table)

    # Save to excel with two tabs: Metrics and Forecasts
    output_path = './classical_techniques_results.xlsx'
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        results_df.to_excel(writer, sheet_name='Metrics', index=False)
        forecasts_df.to_excel(writer, sheet_name='Forecasts', index=False)
    logger.info(f"Successfully saved results to {output_path}")

except Exception as e:
    logger.error(f"Error saving results: {e}")

## **Section 4 - Machine learning and deep learning techniques**

In [ ]:
## **Section 4 - Machine learning and deep learning techniques**

# Install required packages
!pip install xgboost keras-tuner tabulate openpyxl -q

# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from tabulate import tabulate
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from keras_tuner.tuners import RandomSearch
pass  # Colab Drive import removed - place data files in ./data
import warnings
import xgboost as xgb
warnings.filterwarnings('ignore')

In [ ]:
# Mount cloud storage and load data
try:
    pd.read_csv('./two_books_analysis.csv', nrows=1)
except:
    pass  # drive.mount(...) removed for local execution

data_path = './two_books_analysis.csv'
try:
    data = pd.read_csv(data_path)
    data['Interval'] = pd.to_datetime(data['Interval'])
    data.set_index('Interval', inplace=True)
except FileNotFoundError:
    print("Error: Data file not found.")
    raise
except Exception as e:
    print(f"Error loading data: {e}")
    raise

In [ ]:
# Split data by book (case-insensitive)
title_a_data = data[data['Title'].str.lower() == 'title_a, the']['Volume']
title_b_data = data[data['Title'].str.lower() == 'very hungry title_b, the']['Volume']

# Validate data
if title_a_data.empty or title_b_data.empty:
    raise ValueError("Missing data for one or both books.")

# Print data set size and date range
print("Title_A data size:", len(title_a_data))
print("Title_A date range:", title_a_data.index.min(), "to", title_a_data.index.max())
print("Title_B data size:", len(title_b_data))
print("Title_B date range:", title_b_data.index.min(), "to", title_b_data.index.max())

In [ ]:
# Checks if title_b_data is daily by time differences. If daily, aggregates
# it and title_a_data to weekly sums; otherwise, assumes weekly and proceeds.
# Ensures weekly frequency
time_diffs = title_b_data.index[1:] - title_b_data.index[:-1]
if len(time_diffs) > 0 and time_diffs[0] == pd.Timedelta(days=1):
    print("Warning: Data appears to be daily. Aggregating to weekly.")
    title_b_data = title_b_data.resample('W').sum()
    title_a_data = title_a_data.resample('W').sum()
else:
    print("Data is weekly, proceeding without aggregation.")

In [ ]:
# Impute zeros and handle missing values
# Cleans the data sets by replacing zeros with NaN, interpolating missing values, and ensuring no NaN values remain at the edges of the series
title_a_data = title_a_data.replace(0, np.nan).interpolate(method='linear').fillna(method='bfill').fillna(method='ffill')
title_b_data = title_b_data.replace(0, np.nan).interpolate(method='linear').fillna(method='bfill').fillna(method='ffill')
# Both title_a_data and title_b_data have zeros replaced with interpolated
# values, and any remaining NaN values at the edges are filled using backward and forward filling

# These steps ensure no NaN values remain, particularly at the start or end of
# the series where interpolation cannot estimate values (since there’s no data
# before the first point or after the last)

# Validate no NaN or infinite values
if title_a_data.isna().any() or np.isinf(title_a_data).any():
    raise ValueError("Title_A data contains NaN or infinite values after preprocessing.")
if title_b_data.isna().any() or np.isinf(title_b_data).any():
    raise ValueError("Title_B data contains NaN or infinite values after preprocessing.")

In [ ]:
# Define forecast horizon and split data
forecast_horizon = 32 #  future periods to forecast
min_train_size = 50 # minimum number of periods required for the training data set
cutoff_date = pd.to_datetime('2012-01-01') # the date to split the data into training and testing sets

In [ ]:
# Attempt cutoff-based split

# Splits the title_a_data time series into training and test sets using the cutoff date
title_a_train = title_a_data[title_a_data.index < cutoff_date]
title_a_test = title_a_data[title_a_data.index >= cutoff_date][:forecast_horizon] #  include observations from 1 January 2012 onwards
# limits the test set to the first forecast_horizon periods (32 weeks)

# Splits the title_b_data time series into training and test sets using the cutoff date and forecast horizon
title_b_train = title_b_data[title_b_data.index < cutoff_date]
title_b_test = title_b_data[title_b_data.index >= cutoff_date][:forecast_horizon]

In [ ]:
# Fallback to relative split if training data is insufficient
# Checks if the training sets from the previous cutoff-based split meet the
# minimum size requirement, and if not, falls back to a relative split based on the forecast horizon

if len(title_a_train) < min_train_size or len(title_b_train) < min_train_size:
    print(f"Warning: Insufficient training data with 2012-01-01 cutoff. Title_A: {len(title_a_train)}, Title_B: {len(title_b_train)}. Using relative split.")
    title_a_train = title_a_data.iloc[:-forecast_horizon] # takes all data except the last 32 weks
    title_a_test = title_a_data.iloc[-forecast_horizon:]
    title_b_train = title_b_data.iloc[:-forecast_horizon]
    title_b_test = title_b_data.iloc[-forecast_horizon:] # contains the last 32 weeks for testing

# Validates that the test sets match the expected size (forecast_horizon)
if len(title_a_test) != forecast_horizon or len(title_b_test) != forecast_horizon:
    raise ValueError(f"Test data size mismatch. Expected {forecast_horizon} weeks, got {len(title_a_test)} for Title_A, {len(title_b_test)} for Title_B.")

# Validate training data size - Validates that the training sets meet the minimum size requirement after the fallback split
if len(title_a_train) < min_train_size or len(title_b_train) < min_train_size: # ensures the training sets are large enough for robust modelling
    raise ValueError(f"Insufficient training data after fallback. Title_A: {len(title_a_train)}, Title_B: {len(title_b_train)}. Minimum required: {min_train_size}")

# Print data statistics
print("Title_A train size:", len(title_a_train), "min:", title_a_train.min(), "max:", title_a_train.max())
print("Title_B train size:", len(title_b_train), "min:", title_b_train.min(), "max:", title_b_train.max())
print("Title_A test size:", len(title_a_test))
print("Title_B test size:", len(title_b_test))
print("Test data date range:", title_a_test.index.min(), "to", title_a_test.index.max())

In [ ]:
# **Criteria 18: Prepare Data for Machine Learning Models**
# This section of the code prepares the data for XGBoost and LSTM models
# For XGBoost, it creates lags, rolling averages, and week/month.
# For LSTM, it generates sequences of scaled sales and temporal features.
# It ensures data quality with checks for missing or invalid values

# Disclosure: the code in this section was created with AI's assistance
# XGBoost Feature Preparation
class FeaturePreparer: # defines a class to prepare features for XGBoost by specifying lags and rolling window sizes
    def __init__(self, lags=[1, 2, 3], windows=[4, 8]):
        self.lags = lags
        self.windows = windows
        self.feature_names = ['week_of_year', 'month'] + \
                            [f'lag_{lag}' for lag in lags] + \
                            [f'roll_mean_{window}' for window in windows]
        # Initialises a FeaturePreparer object to generate features for time series data

    def prepare_features(self, data, is_train=True): # Generates features for XGBoost from the input data and returns feature matrix X, target y, and the index
        if isinstance(data, pd.Series):
            data = data.to_frame(name='sales')
        elif isinstance(data, pd.DataFrame):
            data = data.copy()
            if 'sales' not in data.columns:
                data['sales'] = data.iloc[:, 0]

        data['week_of_year'] = data.index.isocalendar().week # extracts the week of the year (1 to 53) from the DatetimeIndex
        data['month'] = data.index.month # extracts the month (1 to 12)

        for lag in self.lags:
            data[f'lag_{lag}'] = data['sales'].shift(lag) # Creates lagged sales features (e.g., lag_1 is sales from 1 week ago)

        for window in self.windows:
            data[f'roll_mean_{window}'] = data['sales'].shift(1).rolling(window=window).mean()

        if is_train:
            data = data.dropna() # For training data, drops rows with NaN values
        else:
            data = data.fillna(data.mean()) # For test data, fills NaN values with the mean of each column

        X = data.drop(columns=['sales'], errors='ignore') # Creates the feature matrix X by dropping the sales column
        # Ensure all expected features are present
        for feat in self.feature_names:
            if feat not in X.columns:
                X[feat] = X.mean(axis=1)
        X = X[self.feature_names]  # Reorder columns
        y = data['sales'] if is_train else None
        return X, y, data.index
        # Generates a feature matrix with 7 columns (week_of_year, month, lag_1, lag_2, lag_3, roll_mean_4, roll_mean_8)
        # and a target variable for training

def prepare_xgboost_features(series, lags=[1, 2, 3], windows=[4, 8]):
    preparer = FeaturePreparer(lags=lags, windows=windows)
    X, y, index = preparer.prepare_features(series, is_train=True)
    if X.empty or y.empty:
        raise ValueError(f"Empty feature or target data after preparation. Data size: {len(series)}, lags: {lags}, windows: {windows}")
    print(f"XGBoost features prepared: X shape={X.shape}, y shape={y.shape}")
    return X, y, preparer

# LSTM Data Preparation
def prepare_lstm_sequences(series, lookback=10, forecast_horizon=32):
    """Create scaled sequences with additional features for LSTM input."""
    if series.isna().any() or np.isinf(series).any():
        raise ValueError("Input series contains NaN or infinite values.")

    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(series.values.reshape(-1, 1))
    dates = series.index
    week_of_year = np.array([d.isocalendar().week for d in dates]).reshape(-1, 1)
    month = np.array([d.month for d in dates]).reshape(-1, 1)
    scaler_features = MinMaxScaler()
    scaled_features = scaler_features.fit_transform(np.hstack([week_of_year, month]))

    X, y = [], []
    for i in range(len(scaled_data) - lookback - forecast_horizon + 1):
        X.append(np.hstack([scaled_data[i:i + lookback], scaled_features[i:i + lookback]]))
        y.append(scaled_data[i + lookback])  # Single-step prediction
    X = np.array(X)
    y = np.array(y).reshape(-1, 1)

    if len(X) == 0:
        raise ValueError(f"No sequences generated. Data length: {len(series)}, lookback: {lookback}, forecast_horizon: {forecast_horizon}")

    if np.any(np.isnan(X)) or np.any(np.isinf(X)) or np.any(np.isnan(y)) or np.any(np.isinf(y)):
        raise ValueError("Generated sequences contain NaN or infinite values.")

    return X, y, scaler, scaler_features

# Prepare data for both models
title_a_xgb_X, title_a_xgb_y, title_a_preparer = prepare_xgboost_features(title_a_train)
title_b_xgb_X, title_b_xgb_y, title_b_preparer = prepare_xgboost_features(title_b_train)
title_a_lstm_X, title_a_lstm_y, title_a_scaler, title_a_feature_scaler = prepare_lstm_sequences(title_a_train, lookback=10, forecast_horizon=32)
title_b_lstm_X, title_b_lstm_y, title_b_scaler, title_b_feature_scaler = prepare_lstm_sequences(title_b_train, lookback=10, forecast_horizon=32)

# Validate LSTM input data
print("Title_A LSTM X shape:", title_a_lstm_X.shape, "min:", title_a_lstm_X.min(), "max:", title_a_lstm_X.max())
print("Title_A LSTM y shape:", title_a_lstm_y.shape, "min:", title_a_lstm_y.min(), "max:", title_a_lstm_y.max())
print("Title_B LSTM X shape:", title_b_lstm_X.shape, "min:", title_b_lstm_X.min(), "max:", title_b_lstm_X.max())
print("Title_B LSTM y shape:", title_b_lstm_y.shape, "min:", title_b_lstm_y.min(), "max:", title_b_lstm_y.max())

In [ ]:
# **Criteria 19–22: XGBoost Pipeline, Cross-Validation, Parameter Tuning, and Best Model**
# This code trains XGBoost models to forecast 32 weeks for Title A
# and Title B. It tunes hyperparameters, tests feature windows, and
# uses time series cross-validation to select the best model.
# It then generates forecasts with 95% confidence intervals, evaluates accuracy with
# MAE, MAPE, and RMSE, and plots results, saving them to my cloud storage.

# Disclosure: parts of this section were created with AI's assistance

# Trains XGBoost model with hyperparameter tuning and feature window selection using time series cross-validation
def train_xgboost(X, y, lags=[1, 2, 3]):
    param_grid = { # defines the search space for XGBoost hyperparameters
        'n_estimators': [50, 100], # Number of trees in the model
        'max_depth': [3, 5], # Maximum depth of each tree
        'learning_rate': [0.01, 0.1] # Step size for learning
    }
     # Define different window sizes for creating features (e.g., rolling averages over 4 and 8 weeks)
    window_options = [[4, 8], [4, 12], [8, 12]]
    # Set up time-series cross-validation: splits data into 5 folds, each test set has 8 time points
    tscv = TimeSeriesSplit(n_splits=5, test_size=8)
    # Initialise variables to store the best model, its error score, and its feature preparer
    best_model, best_score, best_preparer = None, float('inf'), None
    # Loop through each window option to test feature creation
    for window in window_options:
        # Create a FeaturePreparer to generate features using specified lags (past time steps) and windows
        preparer = FeaturePreparer(lags=lags, windows=window)
        try:
            # Generate features (X_new) and target (y_new) for training
            X_new, y_new, _ = preparer.prepare_features(X, is_train=True)
            # Initialise XGBoost model for regression with squared error loss, fixed random seed, and GPU acceleration
            model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, tree_method='gpu_hist')
            # Perform grid search to test all hyperparameter combinations using time-series cross-validation
            grid_search = GridSearchCV(model, param_grid, cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
            # Fit the model to the data
            grid_search.fit(X_new, y_new)
            # Check if this model’s error (negative MAE) is better than the current best
            if -grid_search.best_score_ < best_score:
                best_score = -grid_search.best_score_ # Update best error score
                best_model = grid_search.best_estimator_ # Save the best model
                best_preparer = preparer # Save the feature preparer
                print(f"Updated best model with windows={window}, score={-grid_search.best_score_}")
        except Exception as e:
            # If an error occurs (e.g., invalid features), print it and skip to the next window
            print(f"Error with windows={window}: {e}")
            continue
    # If no valid model was found (e.g., all attempts failed), use a default model
    if best_model is None:
        print("Warning: No valid XGBoost model found. Using default model with windows=[4, 8].")
        # Create a default feature preparer with standard window sizes
        default_preparer = FeaturePreparer(lags=lags, windows=[4, 8])
        # Generate default features and target
        X_default, y_default, _ = default_preparer.prepare_features(X, is_train=True)
        # Train a default XGBoost model with standard settings
        default_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, tree_method='gpu_hist')
        default_model.fit(X_default, y_default)
        best_model = default_model
        best_preparer = default_preparer
    # Print the best hyperparameters (or 'default' if no grid search succeeded) and the best window
    print(f"XGBoost best params: {grid_search.best_params_ if 'grid_search' in locals() else 'default'}, best window: {best_preparer.windows}")
    # Return the best model and its feature preparer for use in predictions
    return best_model, best_preparer
    # Summary: Finds the best XGBoost model by testing hyperparameter combinations and feature windows,
    # using time-series cross-validation to ensure robust performance.

# Generates 32-week forecasts using the trained XGBoost model, estimates 95%
# confidence intervals, evaluates accuracy, and plots result
def forecast_xgboost_with_ci(model, preparer, train_data, test_data, forecast_horizon=32, n_samples=100, title=""):
    # Check if test data length matches the forecast horizon (32 weeks)
    if len(test_data) != forecast_horizon:
        raise ValueError(f"Test data length ({len(test_data)}) does not match forecast_horizon ({forecast_horizon}).")

    # Convert training and test data to DataFrames with 'sales' column
    train_df = train_data.to_frame(name='sales')
    test_df = test_data.to_frame(name='sales')
    current_data = train_df.copy() # Copy training data to update with predictions

    predictions = [] # Store predicted sales for each week
    all_samples = [] # Store multiple prediction samples for confidence intervals

    # Loop through each week in the 32-week forecast horizon
    for step in range(forecast_horizon):
        try:
            # Generate features for the current step using the feature preparer
            X_pred, _, _ = preparer.prepare_features(current_data, is_train=False)
            if X_pred.empty:
                raise ValueError(f"Empty features at step {step}.")

            # Use the last row of features for prediction
            X_pred = X_pred.iloc[-1:]
            sample_preds = [] # Store predictions from multiple samples
            # Generate multiple predictions (n_samples=100) for uncertainty estimation
            for _ in range(n_samples):
                pred = model.predict(X_pred)
                sample_preds.append(pred[0])
            y_pred = np.mean(sample_preds) # Average the samples for the final prediction
            predictions.append(y_pred) # Save the prediction
            all_samples.append(sample_preds) # Save all samples for confidence intervals

            # Add the predicted sales to the data set for the next step
            next_date = test_data.index[step]
            new_row = pd.DataFrame({
                'sales': [y_pred],
                'week_of_year': [next_date.isocalendar().week], # Add week of year
                'month': [next_date.month] # Add month
            }, index=[next_date])
            current_data = pd.concat([current_data, new_row])

            # Update lag features (past sales) and rolling mean features
            for lag in preparer.lags:
                current_data[f'lag_{lag}'] = current_data['sales'].shift(lag)
            for window in preparer.windows:
                current_data[f'roll_mean_{window}'] = current_data['sales'].shift(1).rolling(window=window).mean()
            # Fill any missing values with the mean of the column
            current_data = current_data.fillna(current_data.mean())
        except Exception as e:
            # If an error occurs, log it and append NaN values
            print(f"Error at forecasting step {step}: {e}")
            predictions.append(np.nan)
            all_samples.append([np.nan] * n_samples)
            continue

    # Ensure predictions match the forecast horizon
    if len(predictions) != forecast_horizon:
        raise ValueError(f"Prediction length ({len(predictions)}) does not match forecast_horizon ({forecast_horizon}).")

    # Convert predictions and samples to arrays
    y_pred = np.array(predictions)
    all_samples = np.array(all_samples)

    # Calculate 95% confidence intervals (2.5th and 97.5th percentiles of samples)
    lower_ci = np.nanpercentile(all_samples, 2.5, axis=1)
    upper_ci = np.nanpercentile(all_samples, 97.5, axis=1)

    # Calculate error metrics by comparing predictions to actual test data
    actual = test_data.values[:forecast_horizon]
    mae = mean_absolute_error(actual, y_pred, multioutput='raw_values').mean() # Mean Absolute Error
    mape = np.mean(np.abs((actual - y_pred) / (actual + 1e-10))) * 100 # Mean Absolute Percentage Error
    rmse = np.sqrt(mean_squared_error(actual, y_pred)) # Root Mean Squared Error

    # Plot the results
    plt.figure(figsize=(12, 6))
    plt.plot(train_data.index[-100:], train_data[-100:], label='Training Data')
    plt.plot(test_data.index, actual, label='Actual Test Data', color='green')
    plt.plot(test_data.index, y_pred, label='XGBoost Forecast', color='blue')
    plt.fill_between(test_data.index, lower_ci, upper_ci, color='blue', alpha=0.2, label='95% Confidence Interval')
    plt.title(f'{title} XGBoost Forecast with CI ({forecast_horizon} Weeks)')
    plt.xlabel('Date')
    plt.ylabel('Sales Volume')
    plt.legend()
    # Save the plot to a file
    plt.savefig(f'./{title.replace(" ", "_")}_xgboost_forecast.png')
    plt.show()

    # Print error metrics and confidence interval coverage
    print(f"\nxgb_MAE for {title}: {mae:.2f}")
    print(f"xgb_MAPE for {title}: {mape:.2f}%")
    print(f"xgb_RMSE for {title}: {rmse:.2f}")
    coverage = np.mean((actual >= lower_ci) & (actual <= upper_ci)) * 100
    print(f"CI Coverage for {title}: {coverage:.2f}%")
    # Return predictions, error metrics, and confidence intervals
    return y_pred, mae, mape, rmse, lower_ci, upper_ci

# Train XGBoost models
# Executes the XGBoost pipeline for both books, handling errors gracefully
try:
    # Train and forecast for "Title A"
    print("Starting XGBoost pipeline for Title A")
    title_a_xgb_model, title_a_preparer = train_xgboost(title_a_xgb_X, title_a_xgb_y)
    title_a_xgb_pred, title_a_xgb_mae, title_a_xgb_mape, title_a_xgb_rmse, title_a_xgb_lower, title_a_xgb_upper = forecast_xgboost_with_ci(
        title_a_xgb_model, title_a_preparer, title_a_train, title_a_test, forecast_horizon=32, title="Title A"
    )
    # Train and forecast for "Title B"
    print("Starting XGBoost pipeline for Title B")
    title_b_xgb_model, title_b_preparer = train_xgboost(title_b_xgb_X, title_b_xgb_y)
    title_b_xgb_pred, title_b_xgb_mae, title_b_xgb_mape, title_b_xgb_rmse, title_b_xgb_lower, title_b_xgb_upper = forecast_xgboost_with_ci(
        title_b_xgb_model, title_b_preparer, title_b_train, title_b_test, forecast_horizon=32, title="Title B"
    )
except Exception as e:
    #  If an error occurs, log it and return empty/NaN results
    print(f"Error in XGBoost pipeline: {e}")
    title_a_xgb_pred = title_a_xgb_lower = title_a_xgb_upper = np.array([])
    title_b_xgb_pred = title_b_xgb_lower = title_b_xgb_upper = np.array([])
    title_a_xgb_mae = title_b_xgb_mae = title_a_xgb_mape = title_b_xgb_mape = title_a_xgb_rmse = title_b_xgb_rmse = np.nan

In [ ]:
# **Criteria 26–30: LSTM Model, Hyperparameter Tuning, Forecasting, Plotting, and Metrics**
# Here, the code trains LSTM models to forecast 32 weeks of book sales for Title A
# and Title B. It tunes hyperparameters, generates forecasts with 95%
# confidence intervals, and compares with XGBoost in plots.
# Results (MAE, MAPE, RMSE) and forecasts are compiled into tables and exported
# to excel/CSV in my cloud storage for future comparison.

# Disclosure: parts of this sections were created with AI's assistance

def build_lstm_model(hp): # defines the LSTM model for hyperparameter tuning using KerasTuner
    """Define LSTM model architecture for hyperparameter tuning."""
    # Initialise sequential model
    model = Sequential()
    # Define the lookback period
    lookback = hp.Int('lookback', 10, 20, step=5) # Number of time steps

    # Add the first LSTM layer
    model.add(LSTM(
        units=hp.Int('units', 32, 128, step=32), # Number of LSTM units in the first layer
        return_sequences=True if hp.Int('layers', 1, 2) > 1 else False, # Returns sequences for multiple layers
        input_shape=(lookback, 3) # Input shape: (lookback timesteps, 3 features)
    ))

    # Add dropout to prevent overfitting
    model.add(Dropout(hp.Float('dropout', 0.3, 0.5, step=0.1)))  # Tests 30%, 40%, or 50% dropout

    # Add additional LSTM layers if specified (1 or 2 total layers)
    for i in range(hp.Int('layers', 1, 2) - 1):
        model.add(LSTM(
            units=hp.Int(f'units_{i}', 32, 128, step=32), # Units for additional layer
            return_sequences=False # No sequences returned for the last LSTM layer
        ))
        model.add(Dropout(hp.Float(f'dropout_{i}', 0.3, 0.5, step=0.1))) # Dropout for additional layer

    # Add a Dense layer to output a single prediction (sales value)
    model.add(Dense(1))

    # Compile the model with Adam optimizer and mean squared error loss
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            hp.Float('learning_rate', 1e-4, 1e-3, sampling='log'), # Tests learning rates between 0.0001 and 0.001
            clipnorm=1.0 # Limits gradient size to prevent unstable training
        ),
        loss='mse' # Mean squared error loss for regression
    )
    return model
    # Returns compiled LSTM model with tunable architecture

In [ ]:
# Tunes LSTM hyperparameters using KerasTuner and trains the best model
def train_lstm(series, book_title):
    """Tune LSTM hyperparameters with KerasTuner and train the best model."""
    # Set minimum number of sequences needed for training
    min_sequences = 20 # Ensures enough data for training

    # Define lookback periods to test
    lookback_range = range(10, 21, 5)

    # Precompute and cache data sequences for each lookback to avoid recomputation
    sequence_cache = {lookback: prepare_lstm_sequences(series, lookback=lookback, forecast_horizon=32) for lookback in lookback_range}

    # Initialise KerasTuner for random hyperparameter search
    tuner = RandomSearch( # Hyperparameter tuning
        build_lstm_model, # Function to build the LSTM model
        objective='val_loss', # Optimise for lowest validation loss
        max_trials=8, # Try up to 8 different hyperparameter combinations
        executions_per_trial=1, # Run each combination once
        directory='lstm_tuner', # Save tuning results here
        project_name=book_title.replace(" ", "_"), # Unique project name for the book
        overwrite=True, # Overwrite previous tuning results
        max_consecutive_failed_trials=5 # Allow up to 5 failed trials before stopping
    )

    # Initialise variables to track the best model and its settings
    best_model, best_lookback, best_val_loss = None, None, float('inf')
    best_scaler, best_feature_scaler = None, None

     # Test each lookback period
    for lookback in lookback_range:
        # Get precomputed sequences (features X, target y, and scalers)
        X_temp, y_temp, scaler, feature_scaler = sequence_cache[lookback]

        # Check if there are enough sequences for training
        if len(X_temp) < min_sequences:
            print(f"Warning: Insufficient sequences ({len(X_temp)}) for lookback {lookback}, skipping")
            continue
        print(f"Tuning with lookback={lookback}, X_shape={X_temp.shape}")
        try:
            # Run hyperparameter tuning with KerasTuner
            tuner.search(
                X_temp, y_temp, # Training data
                epochs=50, # Train for up to 50 epochs
                validation_split=0.2, # Use 20% of data for validation
                callbacks=[
                    # Stop early if validation loss stops improving for 5 epochs
                    tf.keras.callbacks.EarlyStopping(patience=5),
                    # Log if training stops due to NaN loss
                    tf.keras.callbacks.LambdaCallback(on_epoch_end=lambda epoch, logs:
                        print(f"Epoch {epoch+1} stopped due to NaN loss") if np.isnan(logs.get('loss', 0)) else None)
                ],
                verbose=1 # Show progress
            )

            # Get the best trial (lowest validation loss)
            trial = tuner.oracle.get_best_trials(1)[0]
            trial_val_loss = trial.score

            # Update best model if this trial is better
            if trial_val_loss and trial_val_loss < best_val_loss:
                best_val_loss = trial_val_loss
                best_model = tuner.get_best_models(num_models=1)[0] # Get the best model
                best_lookback = lookback
                best_scaler = scaler
                best_feature_scaler = feature_scaler
                print(f"New best model found for {book_title}, lookback={lookback}, val_loss={trial_val_loss}")
        except Exception as e:
            # Log and skip if an error occurs (e.g., invalid model configuration)
            print(f"Error in trial with lookback={lookback}: {e}")
            continue

    # If no valid model was found, use a default LSTM model (fallback)
    if best_model is None:
        print(f"Warning: No valid trials succeeded for {book_title}. Using default LSTM with lookback=10.")
        default_lookback = 10
        # Get or compute sequences for default lookback
        X_default, y_default, default_scaler, default_feature_scaler = sequence_cache.get(default_lookback, prepare_lstm_sequences(series, lookback=default_lookback, forecast_horizon=32))

        # Check if default sequences are sufficient
        if len(X_default) < min_sequences:
            raise ValueError(f"Default lookback {default_lookback} produced insufficient sequences ({len(X_default)}).")

        # Define a simple default LSTM model
        model = Sequential([
            LSTM(64, input_shape=(default_lookback, 3), return_sequences=False), # Single LSTM layer with 64 units
            Dropout(0.3), # 30% dropout to prevent overfitting
            Dense(1) # Output a single prediction
        ])

        # Compile with Adam optimizer and mean squared error loss
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')

        # Train the default model
        model.fit(
            X_default, y_default,
            epochs=50,
            validation_split=0.2,
            callbacks=[tf.keras.callbacks.EarlyStopping(patience=5)],
            verbose=0 # Silent training
        )

        # Set default model and settings as best
        best_model = model
        best_lookback = default_lookback
        best_scaler = default_scaler
        best_feature_scaler = default_feature_scaler

    # Retrain the best model on the full dataset for its lookback
    X_best, y_best, _, _ = sequence_cache[best_lookback]
    best_model.fit(
        X_best, y_best,
        epochs=50,
        validation_split=0.2,
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=5)],
        verbose=0 # Silent training
    )

    # Print the best lookback period
    print(f"Best lookback for {book_title}: {best_lookback}")
    return best_model, best_lookback, best_scaler, best_feature_scaler
    # Produces a tuned LSTM model for forecasting

In [ ]:
#  This function generates multiple predictions for a given input using Monte Carlo dropout to estimate uncertainty in LSTM forecasts
    # Loops 300 times, making predictions with dropout enabled (training=True) to introduce randomness
    # Collects predictions into an array of shape `(300, 1, 1)`

def predict_with_dropout(model, X, n_samples=300):
    """Custom prediction function to enable Monte Carlo dropout."""
    # Disclosure: this was created with AI's assistance

    # Initialise an empty list to store predictions
    sample_preds = []

    # Loop n_samples times (default 300) to generate multiple predictions
    for _ in range(n_samples):
        # Make a prediction with dropout enabled (training=True) for randomness
        pred = model(X, training=True)
        # Convert prediction to numpy array and append to list
        sample_preds.append(pred.numpy())
    # Convert list of predictions to a numpy array of shape (n_samples, 1, 1)
    return np.array(sample_preds)

In [ ]:
# ## This function generates 32-week sales forecasts using an LSTM model, estimates
# 95% confidence intervals with Monte Carlo dropout, evaluates performance, and visualises results

    # Scales sales and temporal features, prepares the initial sequence
    # Iteratively forecasts 32 steps, using Monte Carlo dropout for uncertainty
    # Inverse transforms predictions to original units
    # Computes MAE, MAPE, RMSE, and CI coverage
    # Plots the forecast with confidence intervals
    # Produces detailed LSTM forecasts with uncertainty estimates, enabling comparison with models like SARIMA and XGBoost

def forecast_lstm_with_ci(model, train_data, test_data, scaler, scaler_features, lookback=10, forecast_horizon=32, title=""):
    """Forecast with LSTM, compute MAPE/RMSE, and estimate CIs using Monte Carlo dropout."""
    # Scale training and test sales data using the provided scaler
    train_scaled = scaler.transform(train_data.values.reshape(-1, 1))
    test_scaled = scaler.transform(test_data.values.reshape(-1, 1))

    # Combine train and test dates for feature creation
    dates = pd.concat([pd.Series(index=train_data.index), pd.Series(index=test_data.index)]).index

    # Extract temporal features: week of year and month
    week_of_year = np.array([d.isocalendar().week for d in dates]).reshape(-1, 1)
    month = np.array([d.month for d in dates]).reshape(-1, 1)

    # Scale temporal features using a new MinMaxScaler
    scaler_features = MinMaxScaler()
    scaled_features = scaler_features.fit_transform(np.hstack([week_of_year, month]))

    # Initialise lists to store predictions and confidence intervals
    predictions = []
    lower_cis = []
    upper_cis = []

    # Prepare the initial input sequence: last 'lookback' scaled sales + scaled features
    current_sequence = np.hstack([train_scaled[-lookback:], scaled_features[len(train_scaled)-lookback:len(train_scaled)]])

    # Iteratively forecast for each of the 32 weeks
    for step in range(forecast_horizon):
        # Prepare input for the model (shape: 1, lookback, 3)
        X_test = np.array([current_sequence])

        # Generate 300 predictions using Monte Carlo Dropout for uncertainty
        sample_preds = predict_with_dropout(model, X_test, n_samples=300)

        # Check for invalid predictions (NaN or Inf)
        if np.any(np.isnan(sample_preds)) or np.any(np.isinf(sample_preds)):
            raise ValueError(f"Invalid predictions at step {step} for {title}: NaN or Inf detected.")

        # Calculate mean prediction and 95% confidence intervals
        y_pred = np.mean(sample_preds, axis=0)[0, 0] # Average of 300 predictions
        lower_ci = np.percentile(sample_preds, 2.5, axis=0)[0, 0] # 2.5th percentile
        upper_ci = np.percentile(sample_preds, 97.5, axis=0)[0, 0] # 97.5th percentile

        # Store predictions and CIs
        predictions.append(y_pred)
        lower_cis.append(lower_ci)
        upper_cis.append(upper_ci)

        # Prepare features for the next step (week of year, month)
        next_features = scaled_features[len(train_scaled) + len(predictions) - 1]

        # Update the sequence with the new prediction and features
        current_sequence = np.vstack([current_sequence[1:], np.array([y_pred, next_features[0], next_features[1]])]) # Add new prediction and features

    # Convert predictions and CIs to arrays
    y_pred = np.array(predictions)
    lower_ci = np.array(lower_cis)
    upper_ci = np.array(upper_cis)

    # Inverse transform predictions and CIs to original sales units
    y_pred = scaler.inverse_transform(y_pred.reshape(-1, 1)).flatten()
    lower_ci = scaler.inverse_transform(lower_ci.reshape(-1, 1)).flatten()
    upper_ci = scaler.inverse_transform(upper_ci.reshape(-1, 1)).flatten()

    # Check for invalid values after inverse transform
    if np.any(np.isnan(y_pred)) or np.any(np.isinf(y_pred)):
        raise ValueError(f"Invalid y_pred after inverse transform for {title}: NaN or Inf detected.")
    if np.any(np.isnan(lower_ci)) or np.any(np.isinf(lower_ci)):
        raise ValueError(f"Invalid lower_ci after inverse transform for {title}: NaN or Inf detected.")
    if np.any(np.isnan(upper_ci)) or np.any(np.isinf(upper_ci)):
        raise ValueError(f"Invalid upper_ci after inverse transform for {title}: NaN or Inf detected.")

    # Get actual test data for comparison
    actual = test_data.values[:forecast_horizon]

    # Ensure predictions match actual data length
    if len(y_pred) != len(actual):
        raise ValueError(f"Prediction length ({len(y_pred)}) does not match actual length ({len(actual)}) for {title}.")

    # Calculate error metrics
    mae = mean_absolute_error(actual, y_pred) # Mean Absolute Error
    mape = np.mean(np.abs((actual - y_pred) / (actual + 1e-10))) * 100 # Mean Absolute Percentage Error
    rmse = np.sqrt(mean_squared_error(actual, y_pred)) # Root Mean Squared Error

    # Plot the results
    plt.figure(figsize=(12, 6))
    plt.plot(train_data.index[-100:], train_data[-100:], label='Training Data') # Last 100 weeks of training
    plt.plot(test_data.index, actual, label='Actual Test Data', color='green') # Actual test data
    plt.plot(test_data.index, y_pred, label='LSTM Forecast', color='red') # Predicted values
    plt.fill_between(test_data.index, lower_ci, upper_ci, color='red', alpha=0.2, label='95% Confidence Interval') # CI shading
    plt.title(f'{title} LSTM Forecast with CI ({forecast_horizon} Weeks)')
    plt.xlabel('Date')
    plt.ylabel('Sales Volume')
    plt.legend()
    # Save the plot to cloud storage
    plt.savefig(f'./{title.replace(" ", "_")}_lstm_forecast.png')
    plt.show()

    # Print error metrics and confidence interval coverage
    print(f"\nlstm_MAE for {title}: {mae:.2f}")
    print(f"lstm_MAPE for {title}: {mape:.2f}%")
    print(f"lstm_RMSE for {title}: {rmse:.2f}")
    coverage = np.mean((actual >= lower_ci) & (actual <= upper_ci)) * 100 # % of actuals within CI
    print(f"CI Coverage for {title}: {coverage:.2f}%")
    print(f"LSTM forecast lengths: y_pred={len(y_pred)}, lower_ci={len(lower_ci)}, upper_ci={len(upper_ci)}, actual={len(actual)}")

    # Return predictions, error metrics, and confidence intervals
    return y_pred, mae, mape, rmse, lower_ci, upper_ci

In [ ]:
# LSTM Pipeline Summary
# Trains LSTM models, forecasts 32 weeks, compares with XGBoost, and compiles results

    # Trains and forecasts using LSTM for both books, handling errors
    # Creates combined plots comparing XGBoost and LSTM forecasts
    # Collects metrics (MAE, MAPE, RMSE) and forecasts into dataframes
    # Exports results to excel and CSV in cloud storage, with validation check
    # Provides a comprehensive comparison of XGBoost and LSTM models, with results stored for future analysis

# Train and forecast with LSTM
try:
    # Train and forecast for "Title A" using LSTM
    print("Starting LSTM pipeline for Title A")
    title_a_lstm_model, title_a_lookback, title_a_scaler, title_a_feature_scaler = train_lstm(title_a_train, "Title A")
    title_a_lstm_pred, title_a_lstm_mae, title_a_lstm_mape, title_a_lstm_rmse, title_a_lstm_lower, title_a_lstm_upper = forecast_lstm_with_ci(
        title_a_lstm_model, title_a_train, title_a_test, title_a_scaler, title_a_feature_scaler, lookback=title_a_lookback, forecast_horizon=32, title="Title A"
    )

    # Train and forecast for "Title B" using LSTM
    print("Starting LSTM pipeline for Title B")
    title_b_lstm_model, title_b_lookback, title_b_scaler, title_b_feature_scaler = train_lstm(title_b_train, "Title B")
    title_b_lstm_pred, title_b_lstm_mae, title_b_lstm_mape, title_b_lstm_rmse, title_b_lstm_lower, title_b_lstm_upper = forecast_lstm_with_ci(
        title_b_lstm_model, title_b_train, title_b_test, title_b_scaler, title_b_feature_scaler, lookback=title_b_lookback, forecast_horizon=32, title="Title B"
    )
except Exception as e:
    # If an error occurs, log it and set LSTM results to empty/NaN
    print(f"Error in LSTM pipeline: {e}")
    title_a_lstm_pred = title_a_lstm_lower = title_a_lstm_upper = np.array([])
    title_b_lstm_pred = title_b_lstm_lower = title_b_lstm_upper = np.array([])
    title_a_lstm_mae = title_b_lstm_mae = title_a_lstm_mape = title_b_lstm_mape = title_a_lstm_rmse = title_b_lstm_rmse = np.nan

# Combined plots for comparison
try:
    # Plot for "Title A" if both XGBoost and LSTM predictions are valid
    if len(title_a_xgb_pred) == forecast_horizon and len(title_a_lstm_pred) == forecast_horizon:
        plt.figure(figsize=(12, 6))
        plt.plot(title_a_train.index[-100:], title_a_train[-100:], label='Training Data') # Last 100 weeks of training
        plt.plot(title_a_test.index, title_a_test, label='Actual Test Data', color='green') # Actual test data
        plt.plot(title_a_test.index, title_a_xgb_pred, label='XGBoost Forecast', color='blue') # XGBoost predictions
        plt.plot(title_a_test.index, title_a_lstm_pred, label='LSTM Forecast', color='red') # LSTM predictions
        plt.title('Title A: XGBoost vs. LSTM Forecast (32 Weeks)')
        plt.xlabel('Date')
        plt.ylabel('Sales Volume')
        plt.legend()
        plt.savefig('./The_Title_A_combined_forecast.png')
        plt.show()
    else:
        print("Warning: Skipping Title_A combined plot due to missing predictions.")

    # Plot for "Title B" if both XGBoost and LSTM predictions are valid
    if len(title_b_xgb_pred) == forecast_horizon and len(title_b_lstm_pred) == forecast_horizon:
        plt.figure(figsize=(12, 6))
        plt.plot(title_b_train.index[-100:], title_b_train[-100:], label='Training Data') # Last 100 weeks
        plt.plot(title_b_test.index, title_b_test, label='Actual Test Data', color='green') # Actual test data
        plt.plot(title_b_test.index, title_b_xgb_pred, label='XGBoost Forecast', color='blue') # XGBoost predictions
        plt.plot(title_b_test.index, title_b_lstm_pred, label='LSTM Forecast', color='red') # LSTM predictions
        plt.title('Title B: XGBoost vs. LSTM Forecast (32 Weeks)')
        plt.xlabel('Date')
        plt.ylabel('Sales Volume')
        plt.legend()
        plt.savefig('./The_Very_Hungry_Title_B_combined_forecast.png') # Save plot
        plt.show()
    else:
        print("Warning: Skipping Title_B combined plot due to missing predictions.")
except Exception as e:
    # Log error if plotting fails
    print(f"Error generating combined plots: {e}")

# **Collect Results**
# Initialise dictionary to store performance metrics
results = {
    'Book': [],
    'Model': [],
    'MAE': [],
    'MAPE': [],
    'RMSE': []
}
forecast_dfs = [] # List to store forecast DataFrames

# Add XGBoost results
try:
    # Print lengths of XGBoost arrays for debugging
    print("XGBoost array lengths:", {
        'title_a_xgb_pred': len(title_a_xgb_pred),
        'title_a_xgb_lower': len(title_a_xgb_lower),
        'title_a_xgb_upper': len(title_a_xgb_upper),
        'title_a_test': len(title_a_test),
        'title_b_xgb_pred': len(title_b_xgb_pred),
        'title_b_xgb_lower': len(title_b_xgb_lower),
        'title_b_xgb_upper': len(title_b_xgb_upper),
        'title_b_test': len(title_b_test)
    })

    # Validate XGBoost array lengths
    if not (len(title_a_xgb_pred) == len(title_a_xgb_lower) == len(title_a_xgb_upper) == len(title_a_test) == forecast_horizon):
        raise ValueError("Inconsistent lengths in Title_A XGBoost arrays.")
    if not (len(title_b_xgb_pred) == len(title_b_xgb_lower) == len(title_b_xgb_upper) == len(title_b_test) == forecast_horizon):
        raise ValueError("Inconsistent lengths in Title_B XGBoost arrays.")

    # Add XGBoost metrics to results dictionary
    results['Book'].extend(["Title A", "Title B"])
    results['Model'].extend(["XGBoost", "XGBoost"])
    results['MAE'].extend([title_a_xgb_mae, title_b_xgb_mae])
    results['MAPE'].extend([title_a_xgb_mape, title_b_xgb_mape])
    results['RMSE'].extend([title_a_xgb_rmse, title_b_xgb_rmse])

    # Create DataFrames for XGBoost forecasts
    title_a_xgb_forecast_df = pd.DataFrame({
        'Date': title_a_test.index,
        'Book': "Title A",
        'Model': "XGBoost",
        'Forecast': title_a_xgb_pred,
        'Lower_CI': title_a_xgb_lower,
        'Upper_CI': title_a_xgb_upper,
        'Actual': title_a_test
    })
    title_b_xgb_forecast_df = pd.DataFrame({
        'Date': title_b_test.index,
        'Book': "Title B",
        'Model': "XGBoost",
        'Forecast': title_b_xgb_pred,
        'Lower_CI': title_b_xgb_lower,
        'Upper_CI': title_b_xgb_upper,
        'Actual': title_b_test
    })
except (NameError, ValueError) as e:
    # If XGBoost results are invalid, log warning and create empty DataFrames
    print(f"Warning: XGBoost results invalid or not defined. Skipping XGBoost results. {e}")
    title_a_xgb_forecast_df = pd.DataFrame()
    title_b_xgb_forecast_df = pd.DataFrame()

# Add LSTM results
try:
    # Print lengths of LSTM arrays for debugging
    print("LSTM array lengths:", {
        'title_a_lstm_pred': len(title_a_lstm_pred),
        'title_a_lstm_lower': len(title_a_lstm_lower),
        'title_a_lstm_upper': len(title_a_lstm_upper),
        'title_a_test': len(title_a_test),
        'title_b_lstm_pred': len(title_b_lstm_pred),
        'title_b_lstm_lower': len(title_b_lstm_lower),
        'title_b_lstm_upper': len(title_b_lstm_upper),
        'title_b_test': len(title_b_test)
    })

    # Validate LSTM array lengths
    if not (len(title_a_lstm_pred) == len(title_a_lstm_lower) == len(title_a_lstm_upper) == len(title_a_test) == forecast_horizon):
        raise ValueError("Inconsistent lengths in Title_A LSTM arrays.")
    if not (len(title_b_lstm_pred) == len(title_b_lstm_lower) == len(title_b_lstm_upper) == len(title_b_test) == forecast_horizon):
        raise ValueError("Inconsistent lengths in Title_B LSTM arrays.")

    # Add LSTM metrics to results dictionary
    results['Book'].extend(["Title A", "Title B"])
    results['Model'].extend(["LSTM", "LSTM"])
    results['MAE'].extend([title_a_lstm_mae, title_b_lstm_mae])
    results['MAPE'].extend([title_a_lstm_mape, title_b_lstm_mape])
    results['RMSE'].extend([title_a_lstm_rmse, title_b_lstm_rmse])

    # Create DataFrames for LSTM forecasts
    title_a_lstm_forecast_df = pd.DataFrame({
        'Date': title_a_test.index,
        'Book': "Title A",
        'Model': "LSTM",
        'Forecast': title_a_lstm_pred,
        'Lower_CI': title_a_lstm_lower,
        'Upper_CI': title_a_lstm_upper,
        'Actual': title_a_test
    })
    title_b_lstm_forecast_df = pd.DataFrame({
        'Date': title_b_test.index,
        'Book': "Title B",
        'Model': "LSTM",
        'Forecast': title_b_lstm_pred,
        'Lower_CI': title_b_lstm_lower,
        'Upper_CI': title_b_lstm_upper,
        'Actual': title_b_test
    })

    # Check for NaN values in LSTM DataFrames
    if title_a_lstm_forecast_df.isna().any().any():
        print("Warning: NaN values detected in title_a_lstm_forecast_df")
        print(title_a_lstm_forecast_df.head())
    if title_b_lstm_forecast_df.isna().any().any():
        print("Warning: NaN values detected in title_b_lstm_forecast_df")
        print(title_b_lstm_forecast_df.head())
except (NameError, ValueError) as e:
    # If LSTM results are invalid, log warning and create empty DataFrames
    print(f"Warning: LSTM results invalid or not defined. Skipping LSTM results. {e}")
    title_a_lstm_forecast_df = pd.DataFrame()
    title_b_lstm_forecast_df = pd.DataFrame()

# Combine all forecast DataFrames
forecast_dfs = [
    title_a_xgb_forecast_df,
    title_b_xgb_forecast_df,
    title_a_lstm_forecast_df,
    title_b_lstm_forecast_df
]

# Filter out empty DataFrames
forecast_dfs = [df for df in forecast_dfs if not df.empty]

# Validate results dictionary lengths
lengths = {key: len(value) for key, value in results.items()}
print("Results dictionary lengths:", lengths)
if len(set(lengths.values())) > 1:
    raise ValueError(f"Inconsistent lengths in results dictionary: {lengths}")

# Create results DataFrame from metrics
results_df = pd.DataFrame(results)

# Display results table
print("\n=== Machine Learning and Deep Learning Results Table ===")
print(tabulate(results_df, headers='keys', tablefmt='pretty', showindex=False))

# Combine forecast DataFrames
if forecast_dfs:
    forecasts_combined = pd.concat(forecast_dfs, ignore_index=False)
    forecasts_combined = forecasts_combined.sort_values(['Date', 'Book', 'Model'])
    print("\n=== Forecast Sample Table ===")
    print(tabulate(forecasts_combined.head(10), headers='keys', tablefmt='pretty'))
else:
    print("\nWarning: No forecast DataFrames available.")

# Export to Excel and CSV
output_excel_path = './ml_dl_results.xlsx'
output_csv_path = './ml_dl_results.csv'
try:
    # Export metrics and forecasts to Excel
    with pd.ExcelWriter(output_excel_path, engine='openpyxl') as writer:
        results_df.to_excel(writer, sheet_name='Metrics', index=False)
        if forecast_dfs:
            forecasts_combined.to_excel(writer, sheet_name='Forecasts', index=False)
    print(f"\nResults exported to {output_excel_path}")

    # Export metrics and forecasts to CSV
    results_df.to_csv(output_csv_path, index=False)
    if forecast_dfs:
        forecasts_combined.to_csv(output_csv_path.replace('.csv', '_forecasts.csv'), index=False)
    print(f"Results exported to CSV: {output_csv_path} and {output_csv_path.replace('.csv', '_forecasts.csv')}")
except Exception as e:
    # If Excel export fails, try CSV as fallback
    print(f"Error exporting to Excel/CSV: {e}")
    try:
        results_df.to_csv(output_csv_path, index=False)
        if forecast_dfs:
            forecasts_combined.to_csv(output_csv_path.replace('.csv', '_forecasts.csv'), index=False)
        print(f"Fallback: Results exported to CSV: {output_csv_path} and {output_csv_path.replace('.csv', '_forecasts.csv')}")
    except Exception as e_csv:
        print(f"Error exporting to CSV: {e_csv}")

# Summary: Runs LSTM pipeline, compares with XGBoost, visualizes results, and exports metrics and forecasts.

## **Section 5 - Hybrid model**

In [ ]:
# Install required packages
!pip install --upgrade numpy==1.25.2 pandas==2.0.3 pmdarima==2.0.4 tensorflow==2.15.0 keras-tuner==1.4.6 statsmodels==0.14.1 scikit-learn==1.2.2 scipy==1.14.1 matplotlib==3.8.4 seaborn==0.13.2 tabulate==0.9.0 openpyxl==3.1.5 xlsxwriter==3.1.9 -q

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import keras_tuner as kt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tabulate import tabulate
pass  # Colab Drive import removed - place data files in ./data
import warnings
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
warnings.filterwarnings('ignore')

In [ ]:
# Mount cloud storage
try:
    pd.read_csv('./two_books_analysis.csv', nrows=1)
except:
    pass  # drive.mount(...) removed for local execution

# Load and inspect data
data_path = './two_books_analysis.csv'
try:
    data = pd.read_csv(data_path)
    logger.info(f"Data loaded: shape={data.shape}, columns={data.columns}")
    logger.info(f"Missing values: {data.isna().sum().to_dict()}")
    logger.info(f"Negative or zero volumes: {(data['Volume'] <= 0).sum()}")
    data['Interval'] = pd.to_datetime(data['Interval'])
    data.set_index('Interval', inplace=True)
except FileNotFoundError:
    logger.error("Data file not found.")
    raise
except Exception as e:
    logger.error(f"Error loading data: {e}")
    raise

In [ ]:
# Split data by book
title_a_data = data[data['Title'].str.lower() == 'title_a, the']['Volume']
title_b_data = data[data['Title'].str.lower() == 'very hungry title_b, the']['Volume']

# Validate and log data
if title_a_data.empty or title_b_data.empty:
    logger.error("Missing data for one or both books.")
    raise ValueError("Missing data for one or both books.")
logger.info(f"Title_A Data: len={len(title_a_data)}, zeros={(title_a_data == 0).sum()}, min={title_a_data.min():.2f}, max={title_a_data.max():.2f}")
logger.info(f"Title_B Data: len={len(title_b_data)}, zeros={(title_b_data == 0).sum()}, min={title_b_data.min():.2f}, max={title_b_data.max():.2f}")

In [ ]:
# Simplified scaler
# This code defines PositiveMinMaxScaler, a custom class that wraps MinMaxScaler
# to scale data to [0, 1] and ensure non-negative inverse-transformed values.
# It validates inputs for NaN or infinite values, and logging errors.
# Disclosure: this sections was created with AI's assistance.

# Custom scaler class that wraps MinMaxScaler to ensure non-negative outputs
class PositiveMinMaxScaler:
    def __init__(self):
        # Initialise a standard MinMaxScaler from scikit-learn
        self.scaler = MinMaxScaler()

    def fit_transform(self, data):
        # Convert input data to a numpy array with shape (n_samples, 1)
        data = np.array(data).reshape(-1, 1)

        # Check for NaN or infinite values in the input
        if np.any(np.isnan(data)) or np.any(np.isinf(data)):
            logger.error("Input data contains NaN or Inf")
            raise ValueError("Invalid input data")

        # Fit the scaler to the data and transform it (scales to range [0, 1])
        return self.scaler.fit_transform(data)

    def transform(self, data):
        # Convert input data to a numpy array with shape (n_samples, 1)
        data = np.array(data).reshape(-1, 1)

        # Check for NaN or infinite values in the input
        if np.any(np.isnan(data)) or np.any(np.isinf(data)):
            logger.error("Input data contains NaN or Inf") # Log error
            raise ValueError("Invalid input data") # Raise an exception

        # Transform the data using the fitted scaler
        return self.scaler.transform(data)

    def inverse_transform(self, data):
        # Inverse-transform the scaled data back to original units
        result = self.scaler.inverse_transform(data)

        # Clip the result to ensure no negative values
        return np.clip(result, 0, None)

In [ ]:
# Enhanced preprocess_series function
# Cleans and prepares the time-series by resampling, handling zeros, NaNs, outliers,
# and ensuring a complete date range, with logging and validation

def preprocess_series(series, freq='W'):
    try:
        # Resample the series to the specified frequency (default: weekly) and sum values
        series = series.resample(freq).sum()

        # Log initial series statistics (length, number of zeros, min, max)
        logger.info(f"Raw series: len={len(series)}, zeros={(series == 0).sum()}, min={series.min():.2f}, max={series.max():.2f}")

        # Replace zeros with small positive value to avoid issues in modeling
        series = series.replace(0, 1e-6)

        # Check for NaNs or infinities early
        if series.isna().any() or np.any(np.isinf(series)):
            logger.warning(f"Series contains {series.isna().sum()} NaNs or Inf before preprocessing")
            # Fill NaNs with the series mean (or 1e-6 if mean is invalid)
            series = series.fillna(series.mean() or 1e-6)
            # Clip values to ensure no negatives or infinities
            series = series.clip(lower=1e-6, upper=np.inf)

        # Smooth series
        series = series.rolling(window=3, center=True, min_periods=1).mean()

        # Clip outliers
        series = series.clip(lower=series.quantile(0.01), upper=series.quantile(0.99))

        # Interpolate and fill missing values
        series = series.interpolate(method='time')

        # If interpolation fails, fill remaining NaNs with the series mean
        if series.isna().any():
            logger.warning("Interpolation failed; filling with mean.")
            series = series.fillna(series.mean() or 1e-6)

        # Fill any remaining NaNs using backward and forward filling
        series = series.fillna(method='bfill').fillna(method='ffill')

        # Reindex to ensure complete date range
        expected_dates = pd.date_range(start=series.index.min(), end=series.index.max(), freq=freq)
        series = series.reindex(expected_dates, method='ffill')
        # Final validation
        if series.isna().any() or np.any(np.isinf(series)):
            logger.error(f"Series contains {series.isna().sum()} NaNs or Inf after preprocessing")
            raise ValueError("Invalid series after preprocessing")

        # Perform Augmented Dickey-Fuller (ADF) test to check stationarity
        result = adfuller(series.dropna())
        logger.info(f"ADF test p-value: {result[1]:.4f}")

        # Log final series statistics
        logger.info(f"Preprocessed series: len={len(series)}, min={series.min():.2f}, max={series.max():.2f}")

        # Return the preprocessed series
        return series

    except Exception as e:
        # Log any errors and raise them to halt execution
        logger.error(f"Error preprocessing series: {e}")
        raise

In [ ]:
# Define parameters for time-series forecasting
# Sets key parameters for forecasting, including forecast length, historical input size,
# and minimum training data requirement

forecast_horizon = 32 # Number of weeks to forecast into the future (32 weeks)
lookback = 26 # Number of past weeks used as input for each prediction (26 weeks)
min_train_size = lookback + forecast_horizon + 10  # # Minimum training data needed: 68 weeks

# Explanation of min_train_size:
# lookback (26): Weeks of historical data for model input
# forecast_horizon (32): Weeks to predict
# +10: Extra weeks to ensure enough data for training and validation

In [ ]:
# Ensure test size (adjusted for this function for this section)
# Creates a test data set with future dates for forecasting, ensuring correct size
# and frequency, with placeholder values and metadata, while validating and logging

def ensure_test_size(data, test_size, freq='W', date_column='Interval', value_column='Volume'):
    # Log the test size and frequency being used
    logger.info(f"Ensuring test size: test_size={test_size}, freq={freq}")

    try:
        # Convert Series to DataFrame if necessary, using value_column for values
        if isinstance(data, pd.Series):
            data = data.to_frame(name=value_column)
            data[date_column] = data.index # Use index as date column
            data.reset_index(drop=True, inplace=True)

        # Convert date column to datetime, handling invalid dates
        data[date_column] = pd.to_datetime(data[date_column], errors='coerce')

        # Check if all dates are invalid (NaT)
        if data[date_column].isna().all():
            raise ValueError(f"All dates in '{date_column}' are invalid.")

        # Get the last valid date in the dataset
        last_date = data[date_column].max()
        if pd.isna(last_date):
            raise ValueError("Last date is NaT.")

        # Generate future dates for the test set (starting 1 week after last_date)
        test_dates = pd.date_range(start=last_date + pd.DateOffset(weeks=1), periods=test_size, freq=freq)

        # Create a test DataFrame with dates, placeholder values (0), and metadata
        test_data = pd.DataFrame({
            date_column: test_dates,
            value_column: 0, # Placeholder values for test set
            'ISBN': data['ISBN'].iloc[0] if 'ISBN' in data.columns else None, # Include ISBN if present
            'Title': data['Title'].iloc[0] if 'Title' in data.columns else None # Include Title if present
        })

        # Set the date column as index and extract the value column
        result = test_data.set_index(date_column)[value_column]

        # Validate that the test set has the correct size
        if len(result) != test_size:
            raise ValueError(f"Test set length ({len(result)}) does not match test_size ({test_size})")

        # Log the created test set's date range and size
        logger.info(f"Test set created: {result.index.min()} to {result.index.max()}, size={len(result)}")

        # Return the test set as a Series
        return result
    except Exception as e:
        # Log any errors and raise them to halt execution
        logger.error(f"Error in ensure_test_size: {e}")
        raise

In [ ]:
# Preprocesses sales data for two books and splits each into training
# (all but last 32 weeks) and test sets (32 weeks of future dates).

# Preprocess the sales data for "Title A" using the preprocess_series function
title_a_series = preprocess_series(title_a_data)

# Preprocess the sales data for "Title B"
title_b_series = preprocess_series(title_b_data)

# Create a test set for "Title A" with 32 weeks of future dates
title_a_test = ensure_test_size(title_a_series, test_size=forecast_horizon, freq='W') # Number of weeks to forecast (32) and weekly frequency

# Create the training set for "Title A" by excluding the last 32 weeks
title_a_train = title_a_series.iloc[:-forecast_horizon]

# Create a test set for "Title B" with 32 weeks of future dates
title_b_test = ensure_test_size(title_b_series, test_size=forecast_horizon, freq='W')

# Create the training set for "Title B" by excluding the last 32 weeks
title_b_train = title_b_series.iloc[:-forecast_horizon]

In [ ]:
# Aligns original test data with test set dates, validates training and test data sizes,
# and adjusts forecast horizon if training data is too short.

# Resample original Title_A data to weekly sums, align with test set index, and fill gaps
title_a_test_original = title_a_data.resample('W').sum().reindex(title_a_test.index, method='ffill').fillna(title_a_data.mean() or 1e-6) # Fill remaining NaNs with mean or tiny value

# Resample original Title_B data to weekly sums, align with test set index, and fill gaps
title_b_test_original = title_b_data.resample('W').sum().reindex(title_b_test.index, method='ffill').fillna(title_b_data.mean() or 1e-6)

# Validate data length
# Check if Title_A training data has enough points (min_train_size)
if len(title_a_train) < min_train_size:
    logger.error(f"Title_A training data too short: {len(title_a_train)} points, need {min_train_size}")
    raise ValueError("Title_A training data too short")

# Check if Title_B training data has enough points
if len(title_b_train) < min_train_size:
    logger.warning(f"Title_B training data too short: {len(title_b_train)} points, need {min_train_size}. Trying shorter horizon.")
    forecast_horizon = 16  # Reduce forecast horizon to 16 weeks
    # Recreate Title_A test and train sets with shorter horizon
    title_a_test = ensure_test_size(title_a_series, test_size=forecast_horizon, freq='W')
    title_a_train = title_a_series.iloc[:-forecast_horizon]
    # Recreate Title_B test and train sets with shorter horizon
    title_b_test = ensure_test_size(title_b_series, test_size=forecast_horizon, freq='W')
    title_b_train = title_b_series.iloc[:-forecast_horizon]

# Validate that test sets match the expected forecast horizon
if len(title_a_test) != forecast_horizon or len(title_b_test) != forecast_horizon:
    logger.error(f"Test data size mismatch: Expected {forecast_horizon}, got {len(title_a_test)} for Title_A, {len(title_b_test)} for Title_B")
    raise ValueError("Test data size mismatch")

In [ ]:
# Logs the date ranges and sizes of training and test sets for both books,
# plus a sample of the Title_B training data for inspection

# Log the date range and size of the Title_A training set
logger.info(f"Title_A Train Range: {title_a_train.index.min()} to {title_a_train.index.max()}, size: {len(title_a_train)}")

# Log the date range and size of the Title_A test set
logger.info(f"Title_A Test Range: {title_a_test.index.min()} to {title_a_test.index.max()}, size: {len(title_a_test)}")

# Log the date range and size of the Title_B training set
logger.info(f"Title_B Train Range: {title_b_train.index.min()} to {title_b_train.index.max()}, size: {len(title_b_train)}")

# Log the date range and size of the Title_B test set
logger.info(f"Title_B Test Range: {title_b_test.index.min()} to {title_b_test.index.max()}, size: {len(title_b_test)}")

# Log the first five rows of the Title_B training data as a dictionary
logger.info(f"Title_B Train Sample: {title_b_train.head().to_dict()}")

In [ ]:
# Prepares LSTM input-output sequences by scaling data, adding temporal features,
# and creating sequences with validation and logging

def prepare_lstm_sequences(series, lookback=26, forecast_horizon=32):
    try:
        # Check if the series is long enough for the lookback period
        if len(series) < lookback + 1:
            logger.error(f"Series length ({len(series)}) too short for lookback ({lookback}).")
            raise ValueError("Series length too short")

        # Initialise a scaler to transform data to range [0, 1]
        scaler = PositiveMinMaxScaler()
        # Scale the series values
        scaled_data = scaler.fit_transform(series.values)

        # Get the dates from the series index
        dates = series.index
        # Create normalised features: week of year (1-52, scaled to 0-1) and month (1-12, scaled to 0-1)
        week_of_year = np.array([d.isocalendar().week for d in dates]) / 52
        month = np.array([d.month for d in dates]) / 12

        # Combine scaled sales data with temporal features
        features = np.column_stack([scaled_data, week_of_year, month])

        # Check for NaN or infinite values in features
        if np.any(np.isnan(features)) or np.any(np.isinf(features)):
            logger.error(f"Features contain NaN or Inf: NaNs={np.isnan(features).sum()}, Inf={np.isinf(features).sum()}")
            raise ValueError("Invalid features")

        # Log the shape and a sample of the features for debugging
        logger.info(f"Features shape: {features.shape}, sample: {features[:5]}")

        # Initialise lists for input sequences (X) and outputs (y)
        X, y = [], []

        # Create sequences: each X is 'lookback' time steps, y is the next value
        for i in range(len(series) - lookback):
            X.append(features[i:i + lookback]) # Sequence of lookback time steps
            y.append(scaled_data[i + lookback]) # Next value after the sequence

        # Convert lists to numpy arrays
        X = np.array(X) # Shape: (n_sequences, lookback, n_features)
        y = np.array(y).reshape(-1, 1) # Shape: (n_sequences, 1)

        # Check if any sequences were generated
        if len(X) == 0 or len(y) == 0:
            logger.error(f"No sequences generated for series length {len(series)}, lookback {lookback}")
            raise ValueError("No sequences generated")

        # Log the shapes of the prepared sequences
        logger.info(f"Prepared LSTM sequences: X shape {X.shape}, y shape {y.shape}")

        # Return the sequences, target values, and scaler
        return X, y, scaler

    except Exception as e:
        # Log any errors and raise them to halt execution
        logger.error(f"Error preparing LSTM sequences: {e}")
        raise

In [ ]:
# Fits a SARIMA model, computes residuals and forecasts, with fallback to mean
# forecast if errors occur, including validation and logging

def fit_sarima(series, title, forecast_horizon=32):
    # Log the start of SARIMA fitting for the given title
    logger.info(f"Fitting SARIMA for {title}...")

    try:
        # Check for NaN or infinite values in the series
        if series.isna().any() or np.any(np.isinf(series)):
            logger.error(f"Series contains NaNs or Inf for {title}")
            raise ValueError("Invalid series")

        # Log the first few values of the input series for debugging
        logger.info(f"SARIMA input series head: {series.head().to_dict()}")

        # Initialise a SARIMA model with basic parameters
        model = SARIMAX(
            series,
            order=(0, 0, 0), # Non-seasonal ARIMA parameters (p, d, q)
            seasonal_order=(0, 0, 0, 0), # Seasonal parameters (P, D, Q, s)
            enforce_stationarity=False, # Allow non-stationary models
            enforce_invertibility=False # Allow non-invertible models
        )

        # Fit the model using the Powell optimization method
        fit = model.fit(disp=False, maxiter=1000, method='powell')

        # Compute in-sample predictions for the series' date range
        fitted = fit.predict(start=series.index[0], end=series.index[-1])

        # Validate that the fitted predictions match the series length
        if len(fitted) != len(series):
            logger.error(f"SARIMA fitted length ({len(fitted)}) does not match series length ({len(series)})")
            raise ValueError("SARIMA fitted length mismatch")

        # Check for NaNs in in-sample predictions
        if fitted.isna().any():
            logger.error(f"SARIMA in-sample predictions contain NaNs for {title}")
            raise ValueError("SARIMA in-sample predictions invalid")

        # Calculate residuals (actual - predicted)
        residuals = series - fitted
        residuals = residuals.fillna(0) # Fill any NaNs in residuals with 0

        # Forecast future values for the specified horizon
        test_pred = fit.forecast(steps=forecast_horizon)

        # If forecast contains NaNs, replace with mean forecast
        if test_pred.isna().any():
            logger.warning(f"SARIMA forecast contains {test_pred.isna().sum()} NaNs for {title}. Using mean forecast.")
            test_pred = pd.Series([series.mean()] * forecast_horizon, index=pd.date_range(start=series.index[-1] + pd.DateOffset(weeks=1), periods=forecast_horizon, freq='W')) # Use series mean for all forecast steps

        # Log successful SARIMA fitting
        logger.info(f"SARIMA fitted successfully for {title}")

        # Return the fitted model, residuals, and forecast
        return fit, residuals, test_pred

    except Exception as e:
        # Log SARIMA failure and fall back to mean forecast
        logger.error(f"SARIMA failed for {title}: {e}. Using mean forecast.")
        try:
            # Create a mean forecast for the horizon
            test_pred = pd.Series([series.mean()] * forecast_horizon, index=pd.date_range(start=series.index[-1] + pd.DateOffset(weeks=1), periods=forecast_horizon, freq='W'))

            # Compute residuals as series minus its mean
            residuals = series - series.mean()
            residuals = residuals.fillna(0)

            # Return None for model (failed), residuals, and mean forecast
            return None, residuals, test_pred
        except Exception as e2:
            # Log failure of mean forecast and return None for all outputs
            logger.error(f"Mean forecast failed for {title}: {e2}")
            return None, None, None

In [ ]:
# Build an LSTM model with tunable hyperparameters for time-series forecasting
# Creates a tunable LSTM model with 1-2 layers, variable units, and dropout,
# compiled for time-series forecasting with KerasTuner.

def build_lstm_model(hp):
    # Initialise a sequential model (layers added one by one)
    model = Sequential()
    num_layers = hp.Int('layers', 1, 2)  # Define the number of LSTM layers (1 or 2) using KerasTuner
    # Add the first LSTM layer
    model.add(LSTM(
        units=hp.Int('units_0', min_value=32, max_value=64, step=32), # Test 32 or 64 units
        return_sequences=num_layers > 1, # Return sequences if more layers follow
        input_shape=(lookback, 3) # Input shape: (lookback timesteps, 3 features)
    ))

    # Add dropout to reduce overfitting
    model.add(Dropout(hp.Float('dropout', 0.1, 0.3, step=0.1))) # Test 10%, 20%, or 30% dropout

    # Add additional LSTM layers if num_layers is 2
    for i in range(num_layers - 1):
        model.add(LSTM(
            units=hp.Int(f'units_{i+1}', min_value=32, max_value=64, step=32), # Test 32 or 64 units
            return_sequences=False # Last LSTM layer does not return sequences
        ))
        model.add(Dropout(hp.Float('dropout', 0.1, 0.3, step=0.1))) # Add dropout after additional layer

    # Add a Dense layer to output a single prediction (e.g., next sales value)
    model.add(Dense(1))

    # Compile the model with mean squared error loss and Adam optimizer
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(clipnorm=1.0)) # Minimize mean squared error for regression
    return model # Return the compiled LSTM model

In [ ]:
# Trains an LSTM model with hyperparameter tuning using Hyperband,
# retrains the best model, and handles errors with logging

def train_lstm(X, y, book_title, project_suffix):
    logger.info(f"Training LSTM for {book_title} ({project_suffix})...")
    try: # Initialise a Hyperband tuner for hyperparameter optimization
        tuner = kt.Hyperband(
            build_lstm_model, # Function to build the LSTM model
            objective='val_loss', # Optimise for lowest validation loss
            max_epochs=20, # Maximum number of epochs per trial
            factor=3, # Reduction factor for Hyperband (controls trial pruning)
            directory='lstm_tuner', # Directory to save tuning results
            project_name=f"{book_title.replace(' ', '_')}_{project_suffix}", # Unique project name
            overwrite=True # Overwrite previous tuning results
        )
        # Search for the best hyperparameters using the tuner
        tuner.search(X, y, epochs=20, validation_split=0.2, # Input sequences and target values, Train for up to 20 epochs per trial. Use 20% of data for validation
                     callbacks=[tf.keras.callbacks.EarlyStopping(patience=5)], verbose=1)
        best_model = tuner.get_best_models(num_models=1)[0] # Get the best model from the tuning process
        best_model.fit(X, y, epochs=20, validation_split=0.2,
                       callbacks=[tf.keras.callbacks.EarlyStopping(patience=5)], verbose=0)
        logger.info(f"LSTM trained successfully for {book_title}")
        return best_model
    except Exception as e:
        logger.error(f"Error training LSTM for {book_title}: {e}")
        return None

In [ ]:
# Fallback mean forecast
# Generates a mean forecast as a fallback, computes error metrics (MAE, sMAPE, RMSE),
# and logs/prints results when all models fail.

def forecast_fallback(series, test_series, title, forecast_horizon):
    logger.warning(f"All models failed for {title}. Using mean forecast.")

    # Create a forecast by repeating the training series' mean for the forecast horizon
    mean_pred = pd.Series([series.mean()] * forecast_horizon, index=test_series.index) # Use test series index for dates

    # Select the original test data based on the book title
    test_series_original = title_b_test_original if title == "Title B" else title_a_test_original

    # Calculate error metrics comparing the mean forecast to actual test data
    mae = mean_absolute_error(test_series_original, mean_pred) # Mean Absolute Error
    smape = np.mean(2 * np.abs(test_series_original - mean_pred) / (np.abs(test_series_original) + np.abs(mean_pred) + 1e-10)) * 100 # Symmetric Mean Absolute Percentage Error
    rmse = np.sqrt(mean_squared_error(test_series_original, mean_pred))
    logger.info(f"Mean forecast metrics for {title}: MAE={mae:.2f}, sMAPE={smape:.2f}%, RMSE={rmse:.2f}")

    # Print the error metrics and a comment about using the fallback
    print(f"\nmean_forecast_MAE for {title}: {mae:.2f}")
    print(f"mean_forecast_sMAPE for {title}: {smape:.2f}%")
    print(f"mean_forecast_RMSE for {title}: {rmse:.2f}")
    print(f"Comment: Mean forecast used as final fallback for {title} due to model failures.")

    # Return the forecast (same for prediction, lower CI, upper CI) and metrics
    return mean_pred, mean_pred, mean_pred, mae, smape, rmse

In [ ]:
# Implements a hybrid SARIMA-LSTM model to forecast 32 weeks of book sales.
# SARIMA predicts baseline trends, LSTM models residuals, combining results for
# final forecasts with 95% confidence intervals. Evaluates performance (MAE, sMAPE,
# RMSE, coverage), visualises forecasts, and handles errors with fallbacks.
# Disclosure: this section was created with AI's assistance.

def forecast_sequential_hybrid(sarima_model, lstm_model, residuals, test_series, scaler, lookback=26, forecast_horizon=32, title="", n_samples=100):
    logger.info(f"Starting sequential hybrid forecast for {title}")
    try:
        # Check if residuals are long enough for the lookback period
        if len(residuals) < lookback + 1:
            logger.error(f"Residuals too short for {title}: {len(residuals)} < {lookback + 1}")
            raise ValueError("Residuals too short")

        # Check if either model is invalid, fall back to LSTM-only if so
        if sarima_model is None or lstm_model is None:
            logger.error(f"Invalid SARIMA or LSTM model for {title}. Falling back to LSTM-only.")
            return forecast_lstm_only(lstm_model, residuals, test_series, scaler, lookback, forecast_horizon, title, n_samples)

        # Generate SARIMA forecast for the test period
        sarima_pred = pd.Series(sarima_model.forecast(steps=forecast_horizon), index=test_series.index)

        # If SARIMA predictions contain NaNs, fall back to LSTM-only
        if sarima_pred.isna().any():
            logger.warning(f"SARIMA predictions contain {sarima_pred.isna().sum()} NaNs for {title}. Falling back to LSTM-only.")
            return forecast_lstm_only(lstm_model, residuals, test_series, scaler, lookback, forecast_horizon, title, n_samples)

        # Create temporal features for residuals (week of year and month, normalised)
        week_of_year = np.array([d.isocalendar().week for d in residuals.index]) / 52
        month = np.array([d.month for d in residuals.index]) / 12

         # Clean residuals by filling NaNs with 0
        residuals_clean = residuals.fillna(0)

        # Scale residuals using the provided scaler
        scaled_residuals = scaler.transform(residuals_clean.values)

        # Combine scaled residuals with temporal features
        features = np.column_stack([scaled_residuals, week_of_year, month])

        # Check for NaN or infinite values in features
        if np.any(np.isnan(features)) or np.any(np.isinf(features)):
            logger.error(f"Features contain NaN or Inf for {title}")
            raise ValueError("Invalid features")

        # Initialise lists for predictions and confidence intervals
        predictions = []
        lower_cis = []
        upper_cis = []

        # Start with the last 'lookback' time steps of features
        current_sequence = features[-lookback:]

        # Create temporal features for the test period
        test_week_of_year = np.array([d.isocalendar().week for d in test_series.index]) / 52
        test_month = np.array([d.month for d in test_series.index]) / 12

        # Iteratively forecast residuals for each step in the horizon
        for t in range(forecast_horizon):
            # Prepare input for LSTM (shape: 1, lookback, 3)
            X_test = np.array([current_sequence])

            # Generate multiple predictions for uncertainty estimation
            sample_preds = []
            for _ in range(n_samples):
                # Predict residuals using LSTM
                pred = lstm_model.predict(X_test, verbose=0).flatten()
                # Add small random noise to simulate uncertainty
                pred = pred + np.random.normal(0, 0.01, pred.shape)
                sample_preds.append(pred[0])

            sample_preds = np.array(sample_preds)

            # Check for invalid predictions
            if np.any(np.isnan(sample_preds)) or np.any(np.isinf(sample_preds)):
                logger.error(f"Invalid LSTM predictions at step {t} for {title}")
                raise ValueError(f"Invalid LSTM predictions at step {t}")

            # Calculate mean prediction and 95% confidence intervals
            y_pred = np.mean(sample_preds)
            lower_ci = np.percentile(sample_preds, 2.5)
            upper_ci = np.percentile(sample_preds, 97.5)

            # Store predictions and CIs
            predictions.append(y_pred)
            lower_cis.append(lower_ci)
            upper_cis.append(upper_ci)

            # Update sequence with new prediction and temporal features
            next_features = np.array([y_pred, test_week_of_year[t], test_month[t]])
            current_sequence = np.vstack([current_sequence[1:], next_features])

        # Inverse-transform residual predictions to original scale
        residual_pred = scaler.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()
        residual_lower = scaler.inverse_transform(np.array(lower_cis).reshape(-1, 1)).flatten()
        residual_upper = scaler.inverse_transform(np.array(upper_cis).reshape(-1, 1)).flatten()

        # Combine SARIMA baseline with LSTM residual predictions
        final_pred_transformed = sarima_pred + residual_pred
        final_lower = sarima_pred + residual_lower
        final_upper = sarima_pred + residual_upper

        # Select actual test data based on book title
        test_series_original = title_b_test_original if title == "Title B" else title_a_test_original

        # Convert predictions and CIs to pandas Series
        final_pred = pd.Series(final_pred_transformed, index=test_series.index)
        final_lower = pd.Series(final_lower, index=test_series.index)
        final_upper = pd.Series(final_upper, index=test_series.index)

        # Calculate error metrics
        mae = mean_absolute_error(test_series_original, final_pred)
        smape = np.mean(2 * np.abs(test_series_original - final_pred) / (np.abs(test_series_original) + np.abs(final_pred) + 1e-10)) * 100
        rmse = np.sqrt(mean_squared_error(test_series_original, final_pred))

        # Plot the results
        plt.figure(figsize=(12, 6))
        plt.plot(residuals.index[-100:], residuals[-100:], label='Training Residuals')
        plt.plot(test_series.index, test_series_original, label='Actual Test Data', color='green')
        plt.plot(test_series.index, final_pred, label='Sequential Hybrid Forecast', color='red')
        plt.fill_between(test_series.index, final_lower, final_upper, color='red', alpha=0.2, label='95% Confidence Interval')
        plt.title(f'{title} Sequential Hybrid Forecast ({forecast_horizon} Weeks)')
        plt.xlabel('Date')
        plt.ylabel('Sales Volume')
        plt.legend()
        plt.show()

        # Calculate confidence interval coverage
        coverage = np.mean((test_series_original >= final_lower) & (test_series_original <= final_upper)) * 100

        # Log and print metrics
        logger.info(f"Sequential hybrid metrics for {title}: MAE={mae:.2f}, sMAPE={smape:.2f}%, RMSE={rmse:.2f}, CI Coverage={coverage:.2f}%")
        print(f"\nseq_hybrid_MAE for {title}: {mae:.2f}")
        print(f"seq_hybrid_sMAPE for {title}: {smape:.2f}%")
        print(f"seq_hybrid_RMSE for {title}: {rmse:.2f}")
        print(f"CI Coverage for {title}: {coverage:.2f}%")
        print(f"Comment: The sequential hybrid model integrates SARIMA's baseline trend with LSTM's residual corrections, aiming to capture complex patterns in {title} sales.")
        return final_pred, final_lower, final_upper, mae, smape, rmse # Return predictions, confidence intervals, and metrics
    except Exception as e:
        logger.error(f"Error in sequential hybrid forecast for {title}: {e}") # Log errors and fall back to a mean forecast
        return forecast_fallback(residuals, test_series, title, forecast_horizon)

In [ ]:
# LSTM-only fallback model to forecast 32 weeks of book sales when the hybrid SARIMA-LSTM model fails.
# Uses LSTM to predict sales directly, incorporating scaled week and month features,
# and generates 95% confidence intervals. Evaluates performance (MAE, sMAPE, RMSE, coverage),
# visualizes forecasts, and handles errors with a fallback.
# Disclosure: parts of this code were created with AI's assistance.

def forecast_lstm_only(lstm_model, series, test_series, scaler, lookback=26, forecast_horizon=32, title="", n_samples=100):
    """Forecast sales using an LSTM model as a fallback, with confidence intervals."""
    # Log the start of the LSTM-only forecast
    logger.info(f"Falling back to LSTM-only forecast for {title}")

    try:
        # Check if the LSTM model is valid
        if lstm_model is None:
            logger.error(f"Invalid LSTM model for {title}")
            raise ValueError("Invalid LSTM model")

        # Create normalised temporal features for the training series
        week_of_year = np.array([d.isocalendar().week for d in series.index]) / 52
        month = np.array([d.month for d in series.index]) / 12

        # Clean the series by filling NaNs with 0
        series_clean = series.fillna(0)

        # Scale the series using the provided scaler
        scaled_series = scaler.transform(series_clean.values)

        # Combine scaled series with temporal features
        features = np.column_stack([scaled_series, week_of_year, month])

        # Check for NaN or infinite values in features
        if np.any(np.isnan(features)) or np.any(np.isinf(features)):
            logger.error(f"Features contain NaN or Inf for {title}")
            raise ValueError("Invalid features")

        # Initialise lists for predictions and confidence intervals
        predictions = []
        lower_cis = []
        upper_cis = []

        # Start with the last 'lookback' time steps of features
        current_sequence = features[-lookback:]

        # Create temporal features for the test period
        test_week_of_year = np.array([d.isocalendar().week for d in test_series.index]) / 52
        test_month = np.array([d.month for d in test_series.index]) / 12

        # Iteratively forecast for each step in the horizon
        for t in range(forecast_horizon):
            # Prepare input for LSTM (shape: 1, lookback, 3)
            X_test = np.array([current_sequence])

            # Generate multiple predictions for uncertainty estimation
            sample_preds = []
            for _ in range(n_samples):
                # Predict using LSTM
                pred = lstm_model.predict(X_test, verbose=0).flatten()
                # Add small random noise to simulate uncertainty
                pred = pred + np.random.normal(0, 0.01, pred.shape)
                sample_preds.append(pred[0])

            sample_preds = np.array(sample_preds)

            # Check for invalid predictions
            if np.any(np.isnan(sample_preds)) or np.any(np.isinf(sample_preds)):
                logger.error(f"Invalid LSTM predictions at step {t} for {title}")
                raise ValueError(f"Invalid LSTM predictions at step {t}")

            # Calculate mean prediction and 95% confidence intervals
            y_pred = np.mean(sample_preds)
            lower_ci = np.percentile(sample_preds, 2.5)
            upper_ci = np.percentile(sample_preds, 97.5)

            # Store predictions and CIs
            predictions.append(y_pred)
            lower_cis.append(lower_ci)
            upper_cis.append(upper_ci)

            # Update sequence with new prediction and temporal features
            next_features = np.array([y_pred, test_week_of_year[t], test_month[t]])
            current_sequence = np.vstack([current_sequence[1:], next_features])

        # Inverse-transform predictions and CIs to original scale
        final_pred = scaler.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()
        final_lower = scaler.inverse_transform(np.array(lower_cis).reshape(-1, 1)).flatten()
        final_upper = scaler.inverse_transform(np.array(upper_cis).reshape(-1, 1)).flatten()

        # Convert predictions and CIs to pandas Series
        final_pred = pd.Series(final_pred, index=test_series.index)
        final_lower = pd.Series(final_lower, index=test_series.index)
        final_upper = pd.Series(final_upper, index=test_series.index)

        # Select actual test data based on book title
        test_series_original = title_b_test_original if title == "Title B" else title_a_test_original

        # Calculate error metrics
        mae = mean_absolute_error(test_series_original, final_pred)  # Mean Absolute Error
        smape = np.mean(
            2 * np.abs(test_series_original - final_pred) /
            (np.abs(test_series_original) + np.abs(final_pred) + 1e-10)
        ) * 100  # Symmetric Mean Absolute Percentage Error
        rmse = np.sqrt(mean_squared_error(test_series_original, final_pred))  # Root Mean Squared Error

        # Plot the results
        plt.figure(figsize=(12, 6))
        plt.plot(series.index[-100:], series[-100:], label='Training Data')  # Last 100 training points
        plt.plot(test_series.index, test_series_original, label='Actual Test Data', color='green')  # Actual test data
        plt.plot(test_series.index, final_pred, label='LSTM-Only Forecast', color='red')  # LSTM forecast
        plt.fill_between(test_series.index, final_lower, final_upper, color='red', alpha=0.2, label='95% Confidence Interval')  # CI shading
        plt.title(f'{title} LSTM-Only Forecast ({forecast_horizon} Weeks)')
        plt.xlabel('Date')
        plt.ylabel('Sales Volume')
        plt.legend()
        plt.show()

        # Calculate confidence interval coverage
        coverage = np.mean((test_series_original >= final_lower) & (test_series_original <= final_upper)) * 100

        # Log and print metrics
        logger.info(f"LSTM-only metrics for {title}: MAE={mae:.2f}, sMAPE={smape:.2f}%, RMSE={rmse:.2f}, CI Coverage={coverage:.2f}%")
        print(f"\nlstm_only_MAE for {title}: {mae:.2f}")
        print(f"lstm_only_sMAPE for {title}: {smape:.2f}%")
        print(f"lstm_only_RMSE for {title}: {rmse:.2f}")
        print(f"CI Coverage for {title}: {coverage:.2f}%")
        print(f"Comment: LSTM-only forecast used as fallback due to SARIMA failure for {title}, relying on LSTM's pattern recognition capabilities.")

        # Return predictions, confidence intervals, and metrics
        return final_pred, final_lower, final_upper, mae, smape, rmse

    except Exception as e:
        # Log errors and fall back to a mean forecast
        logger.error(f"Error in LSTM-only forecast for {title}: {e}")
        return forecast_fallback(series, test_series, title, forecast_horizon)

In [ ]:
# Implements a parallel hybrid SARIMA-LSTM model to forecast 32 weeks of book sales.
# Blends SARIMA and LSTM predictions using a weighted average (default: 50/50),
# generates 95% confidence intervals, and evaluates performance (MAE, sMAPE, RMSE, coverage).
# Visualises forecasts and falls back to LSTM-only if errors occur.
# Disclosure: parts of this section were created with AI's assistance.

def forecast_parallel_hybrid(sarima_model, lstm_model, train_series, test_series, scaler, weight=0.5, lookback=26, forecast_horizon=32, title="", n_samples=100, sarima_pred=None):
    """Forecast sales using a parallel hybrid SARIMA-LSTM model with confidence intervals."""
    # Log the start of the parallel hybrid forecast with the weight
    logger.info(f"Starting parallel hybrid forecast for {title} (w={weight})")

    try:
        # Check if models or SARIMA predictions are invalid, fall back to LSTM-only if so
        if sarima_model is None or lstm_model is None or sarima_pred is None:
            logger.error(f"Invalid SARIMA or LSTM model for {title}")
            return forecast_lstm_only(lstm_model, train_series, test_series, scaler, lookback, forecast_horizon, title, n_samples)

        # Check for NaNs in SARIMA predictions, fall back to LSTM-only if found
        if sarima_pred.isna().any():
            logger.warning(f"SARIMA predictions contain {sarima_pred.isna().sum()} NaNs for {title}. Falling back to LSTM-only.")
            return forecast_lstm_only(lstm_model, train_series, test_series, scaler, lookback, forecast_horizon, title, n_samples)

        # Create normalised temporal features for the training series
        week_of_year = np.array([d.isocalendar().week for d in train_series.index]) / 52
        month = np.array([d.month for d in train_series.index]) / 12

        # Scale the training series using the provided scaler
        scaled_series = scaler.transform(train_series.values)

        # Check for NaN or infinite values in scaled series
        if np.any(np.isnan(scaled_series)) or np.any(np.isinf(scaled_series)):
            logger.error(f"Scaled series contains NaN or Inf for {title}")
            raise ValueError("Invalid scaled series")

        # Combine scaled series with temporal features
        features = np.column_stack([scaled_series, week_of_year, month])

        # Check for NaN or infinite values in features
        if np.any(np.isnan(features)) or np.any(np.isinf(features)):
            logger.error(f"Features contain NaN or Inf for {title}")
            raise ValueError("Invalid features")

        # Initialize lists for predictions and confidence intervals
        predictions = []
        lower_cis = []
        upper_cis = []

        # Start with the last 'lookback' time steps of features
        current_sequence = features[-lookback:]

        # Create temporal features for the test period
        test_week_of_year = np.array([d.isocalendar().week for d in test_series.index]) / 52
        test_month = np.array([d.month for d in test_series.index]) / 12

        # Iteratively forecast for each step in the horizon
        for t in range(forecast_horizon):
            # Prepare input for LSTM (shape: 1, lookback, 3)
            X_test = np.array([current_sequence])

            # Generate multiple predictions for uncertainty estimation
            sample_preds = []
            for _ in range(n_samples):
                # Predict using LSTM
                pred = lstm_model.predict(X_test, verbose=0).flatten()
                # Add small random noise and clip to avoid extreme values
                pred = np.clip(pred + np.random.normal(0, 0.01, pred.shape), -1e10, 1e10)
                sample_preds.append(pred[0])

            sample_preds = np.array(sample_preds)

            # Check for invalid predictions
            if np.any(np.isnan(sample_preds)) or np.any(np.isinf(sample_preds)):
                logger.error(f"Invalid LSTM predictions at step {t} for {title}")
                raise ValueError(f"Invalid LSTM predictions at step {t}")

            # Calculate mean prediction and 95% confidence intervals
            y_pred = np.mean(sample_preds)
            lower_ci = np.percentile(sample_preds, 2.5)
            upper_ci = np.percentile(sample_preds, 97.5)

            # Store predictions and CIs
            predictions.append(y_pred)
            lower_cis.append(lower_ci)
            upper_cis.append(upper_ci)

            # Update sequence with new prediction and temporal features
            next_features = np.array([y_pred, test_week_of_year[t], test_month[t]])
            current_sequence = np.vstack([current_sequence[1:], next_features])

        # Inverse-transform LSTM predictions and CIs to original scale
        lstm_pred = scaler.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()
        lstm_lower = scaler.inverse_transform(np.array(lower_cis).reshape(-1, 1)).flatten()
        lstm_upper = scaler.inverse_transform(np.array(upper_cis).reshape(-1, 1)).flatten()

        # Check for NaNs in LSTM predictions
        if np.any(np.isnan(lstm_pred)):
            logger.error(f"LSTM predictions contain NaNs after inverse transform for {title}")
            raise ValueError("LSTM predictions contain NaNs")

        # Blend SARIMA and LSTM predictions using weighted average
        final_pred_transformed = weight * sarima_pred + (1 - weight) * lstm_pred
        final_lower = weight * sarima_pred + (1 - weight) * lstm_lower
        final_upper = weight * sarima_pred + (1 - weight) * lstm_upper

        # Check for NaNs in final predictions
        if np.any(np.isnan(final_pred_transformed)):
            logger.error(f"Final predictions contain NaNs for {title}")
            raise ValueError("Final predictions contain NaNs")

        # Convert predictions and CIs to pandas Series
        final_pred = pd.Series(final_pred_transformed, index=test_series.index)
        final_lower = pd.Series(final_lower, index=test_series.index)
        final_upper = pd.Series(final_upper, index=test_series.index)

        # Select actual test data based on book title
        test_series_original = title_b_test_original if title == "Title B" else title_a_test_original

        # Calculate error metrics
        mae = mean_absolute_error(test_series_original, final_pred)  # Mean Absolute Error
        smape = np.mean(
            2 * np.abs(test_series_original - final_pred) /
            (np.abs(test_series_original) + np.abs(final_pred) + 1e-10)
        ) * 100  # Symmetric Mean Absolute Percentage Error
        rmse = np.sqrt(mean_squared_error(test_series_original, final_pred))  # Root Mean Squared Error

        # Plot the results
        plt.figure(figsize=(12, 6))
        plt.plot(train_series.index[-100:], train_series[-100:], label='Training Data')  # Last 100 training points
        plt.plot(test_series.index, test_series_original, label='Actual Test Data', color='green')  # Actual test data
        plt.plot(test_series.index, final_pred, label=f'Parallel Hybrid Forecast (w={weight})', color='red')  # Hybrid forecast
        plt.fill_between(test_series.index, final_lower, final_upper, color='red', alpha=0.2, label='95% Confidence Interval')  # CI shading
        plt.title(f'{title} Parallel Hybrid Forecast ({forecast_horizon} Weeks)')
        plt.xlabel('Date')
        plt.ylabel('Sales Volume')
        plt.legend()
        plt.show()

        # Calculate confidence interval coverage
        coverage = np.mean((test_series_original >= final_lower) & (test_series_original <= final_upper)) * 100

        # Log and print metrics
        logger.info(f"Parallel hybrid metrics for {title} (w={weight}): MAE={mae:.2f}, sMAPE={smape:.2f}%, RMSE={rmse:.2f}, CI Coverage={coverage:.2f}%")
        print(f"\npar_hybrid_MAE for {title} (w={weight}): {mae:.2f}")
        print(f"par_hybrid_sMAPE for {title} (w={weight}): {smape:.2f}%")
        print(f"par_hybrid_RMSE for {title} (w={weight}): {rmse:.2f}")
        print(f"CI Coverage for {title}: {coverage:.2f}%")
        print(f"Comment: The parallel hybrid model blends SARIMA's stability with LSTM's flexibility, with weight {weight:.1f} balancing contributions for {title}.")

        # Return predictions, confidence intervals, and metrics
        return final_pred, final_lower, final_upper, mae, smape, rmse

    except Exception as e:
        # Log errors and fall back to LSTM-only forecast
        logger.error(f"Error in parallel hybrid forecast for {title}: {e}")
        return forecast_lstm_only(lstm_model, train_series, test_series, scaler, lookback, forecast_horizon, title, n_samples)

# This code blends SARIMA and LSTM predictions in a parallel hybrid model, with uncertainty
# estimates, performance evaluation, visualisation, and LSTM-only fallback.

In [ ]:
# Optimises the weight for blending SARIMA and LSTM predictions in a parallel hybrid model
# to forecast 32 weeks of book sales. Tests weights to minimize MAE, visualizes MAE vs. weight,
# and uses the best weight to generate forecasts with confidence intervals.
# Evaluates performance (MAE, sMAPE, RMSE) and falls back to LSTM-only if errors occur.
# Disclosure: This section was created with AI's assistance.

def optimize_parallel_weights(sarima_model, lstm_model, train_series, test_series, scaler, lookback=26, forecast_horizon=32, title="", n_samples=100, sarima_pred=None):
    """Optimize the blending weight for a parallel SARIMA-LSTM hybrid model."""
    # Log the start of weight optimization
    logger.info(f"Optimizing parallel hybrid weights for {title}")

    try:
        # Check if models or SARIMA predictions are invalid, fall back to LSTM-only if so
        if sarima_model is None or lstm_model is None or sarima_pred is None:
            logger.error(f"Invalid SARIMA or LSTM model for {title}")
            return forecast_lstm_only(lstm_model, train_series, test_series, scaler, lookback, forecast_horizon, title, n_samples) + (0.0,)

        # Check for NaNs in SARIMA predictions, fall back to LSTM-only if found
        if sarima_pred.isna().any():
            logger.warning(f"SARIMA predictions contain {sarima_pred.isna().sum()} NaNs for {title}. Using LSTM-only.")
            return forecast_lstm_only(lstm_model, train_series, test_series, scaler, lookback, forecast_horizon, title, n_samples) + (0.0,)

        # Create normalised temporal features for the training series
        week_of_year = np.array([d.isocalendar().week for d in train_series.index]) / 52
        month = np.array([d.month for d in train_series.index]) / 12

        # Scale the training series
        scaled_series = scaler.transform(train_series.values)

        # Check for NaN or infinite values in scaled series
        if np.any(np.isnan(scaled_series)) or np.any(np.isinf(scaled_series)):
            logger.error(f"Scaled series contains NaN or Inf for {title}")
            raise ValueError("Invalid scaled series")

        # Combine scaled series with temporal features
        features = np.column_stack([scaled_series, week_of_year, month])

        # Check for NaN or infinite values in features
        if np.any(np.isnan(features)) or np.any(np.isinf(features)):
            logger.error(f"Features contain NaN or Inf for {title}")
            raise ValueError("Invalid features")

        # Initialise list for LSTM predictions
        predictions = []

        # Start with the last 'lookback' time steps of features
        current_sequence = features[-lookback:]

        # Create temporal features for the test period
        test_week_of_year = np.array([d.isocalendar().week for d in test_series.index]) / 52
        test_month = np.array([d.month for d in test_series.index]) / 12

        # Generate LSTM predictions for the forecast horizon
        for t in range(forecast_horizon):
            # Prepare input for LSTM (shape: 1, lookback, 3)
            X_test = np.array([current_sequence])

            # Predict using LSTM
            y_pred = lstm_model.predict(X_test, verbose=0).flatten()[0]

            # Check for invalid prediction
            if np.isnan(y_pred) or np.isinf(y_pred):
                logger.error(f"Invalid LSTM prediction at step {t} for {title}")
                raise ValueError("Invalid LSTM prediction")

            # Store prediction
            predictions.append(y_pred)

            # Update sequence with new prediction and temporal features
            next_features = np.array([y_pred, test_week_of_year[t], test_month[t]])
            current_sequence = np.vstack([current_sequence[1:], next_features])

        # Inverse-transform LSTM predictions to original scale
        lstm_pred = scaler.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()

        # Check for NaNs in LSTM predictions
        if np.any(np.isnan(lstm_pred)):
            logger.error(f"LSTM predictions contain NaNs after inverse transform for {title}")
            raise ValueError("LSTM predictions contain NaNs")

        # Select actual test data based on book title
        test_series_original = title_b_test_original if title == "Title B" else title_a_test_original

        # Test a range of weights for blending SARIMA and LSTM
        weights = np.arange(0.1, 1.0, 0.1)
        maes = []
        best_weight, best_mae = None, float('inf')

        # Evaluate MAE for each weight
        for w in weights:
            # Blend SARIMA and LSTM predictions
            final_pred_transformed = w * sarima_pred + (1 - w) * lstm_pred
            final_pred = pd.Series(final_pred_transformed, index=test_series.index)

            # Skip if predictions contain NaNs
            if np.any(np.isnan(final_pred)):
                logger.warning(f"Final predictions contain NaNs for weight {w} in {title}. Skipping.")
                continue

            # Calculate MAE
            mae = mean_absolute_error(test_series_original, final_pred)
            maes.append(mae)

            # Update best weight if MAE is lower
            if mae < best_mae:
                best_mae = mae
                best_weight = w

        # If no valid weight is found, fall back to LSTM-only
        if best_weight is None:
            logger.warning(f"No valid weights found for {title}. Using LSTM-only.")
            best_weight = 0.0
            return forecast_lstm_only(lstm_model, train_series, test_series, scaler, lookback, forecast_horizon, title, n_samples) + (0.0,)

        # Plot MAE vs. SARIMA weight
        plt.figure(figsize=(8, 5))
        plt.plot(weights[:len(maes)], maes, marker='o')
        plt.title(f'{title} MAE vs. SARIMA Weight')
        plt.xlabel('SARIMA Weight (w)')
        plt.ylabel('MAE')
        plt.show()

        # Generate final forecast using the best weight
        final_pred, final_lower, final_upper, mae, smape, rmse = forecast_parallel_hybrid(
            sarima_model, lstm_model, train_series, test_series, scaler, weight=best_weight,
            lookback=lookback, forecast_horizon=forecast_horizon, title=title, n_samples=n_samples, sarima_pred=sarima_pred
        )

        # Log and print results if forecast is successful
        if final_pred is not None:
            logger.info(f"Best weight for {title}: {best_weight:.1f}, MAE={mae:.2f}, sMAPE={smape:.2f}%, RMSE={rmse:.2f}")
            print(f"\nBest Weight for {title}: {best_weight:.1f}")
            print(f"par_hybrid_MAE_best for {title}: {mae:.2f}")
            print(f"par_hybrid_sMAPE_best for {title}: {smape:.2f}%")
            print(f"par_hybrid_RMSE_best for {title}: {rmse:.2f}")
            print(f"Comment: A weight of {best_weight:.1f} indicates {'SARIMA' if best_weight > 0.5 else 'LSTM'} contributes more to the parallel hybrid model for {title}, optimizing predictive accuracy.")

        # Return predictions, confidence intervals, metrics, and best weight
        return final_pred, final_lower, final_upper, mae, smape, rmse, best_weight

    except Exception as e:
        # Log errors and fall back to LSTM-only
        logger.error(f"Error optimizing weights for {title}: {e}")
        return forecast_lstm_only(lstm_model, train_series, test_series, scaler, lookback, forecast_horizon, title, n_samples) + (0.0,)

# Optimises the SARIMA-LSTM blending weight to minimize MAE, generates forecasts
# with the best weight, and falls back to LSTM-only if errors occur.

In [ ]:
# Initialise results storage
results = {'Book': [], 'Model': [], 'MAE': [], 'MAPE': [], 'RMSE': [], 'Weight': []}
forecast_dfs = []

In [ ]:
# Processes hybrid SARIMA-LSTM models for forecasting 32 weeks of book sales for
# Title A and Title B. Applies sequential (LSTM on residuals)
# and parallel (fixed and optimized weights) hybrid models, compiles metrics (MAE, sMAPE, RMSE)
# and forecasts, and exports them to Excel. Compares results with classical and ML/DL models.
# Optimises parallel weights to enhance forecast accuracy.
# Disclosure: the code below was consolidated with AI's assistance.

# Process Title A
logger.info("\n===== Hybrid Models for Title A =====")
# Fit SARIMA model and get residuals and predictions
title_a_sarima_model, title_a_residuals, title_a_sarima_pred = fit_sarima(title_a_train, "Title A", forecast_horizon)

# Sequential Hybrid: LSTM on SARIMA residuals
if title_a_residuals is not None:
    # Prepare sequences from residuals for LSTM training
    X_residuals_title_a, y_residuals_title_a, residuals_scaler_title_a = prepare_lstm_sequences(title_a_residuals, lookback=26, forecast_horizon=forecast_horizon)
    logger.info("\nSequential Hybrid Model for Title A...")
    # Train LSTM model on residuals
    lstm_residuals_model_title_a = train_lstm(X_residuals_title_a, y_residuals_title_a, "Title A", "residuals")

    # Generate sequential hybrid forecast
    seq_pred_title_a, seq_lower_title_a, seq_upper_title_a, seq_mae_title_a, seq_smape_title_a, seq_rmse_title_a = forecast_sequential_hybrid(
        title_a_sarima_model, lstm_residuals_model_title_a, title_a_residuals, title_a_test,
        residuals_scaler_title_a, lookback=26, forecast_horizon=forecast_horizon, title="Title A"
    )

    # Store results if successful
    if seq_mae_title_a is not None:
        results['Book'].append("Title A")
        results['Model'].append("Sequential Hybrid")
        results['MAE'].append(seq_mae_title_a)
        results['MAPE'].append(seq_smape_title_a)
        results['RMSE'].append(seq_rmse_title_a)
        results['Weight'].append(None)
        forecast_dfs.append(pd.DataFrame({
            'Date': title_a_test.index,
            'Book': "Title A",
            'Model': "Sequential Hybrid",
            'Forecast': seq_pred_title_a,
            'Lower_CI': seq_lower_title_a,
            'Upper_CI': seq_upper_title_a,
            'Actual': title_a_test_original
        }))
else:
    # Fallback to mean forecast if residuals are unavailable
    seq_pred_title_a, seq_lower_title_a, seq_upper_title_a, seq_mae_title_a, seq_smape_title_a, seq_rmse_title_a = forecast_fallback(
        title_a_train, title_a_test, "Title A", forecast_horizon
    )
    if seq_mae_title_a is not None:
        results['Book'].append("Title A")
        results['Model'].append("Sequential Hybrid")
        results['MAE'].append(seq_mae_title_a)
        results['MAPE'].append(seq_smape_title_a)
        results['RMSE'].append(seq_rmse_title_a)
        results['Weight'].append(None)
        forecast_dfs.append(pd.DataFrame({
            'Date': title_a_test.index,
            'Book': "Title A",
            'Model': "Sequential Hybrid",
            'Forecast': seq_pred_title_a,
            'Lower_CI': seq_lower_title_a,
            'Upper_CI': seq_upper_title_a,
            'Actual': title_a_test_original
        }))

# Parallel Hybrid (fixed weight = 0.5)
X_series_title_a, y_series_title_a, series_scaler_title_a = prepare_lstm_sequences(title_a_train, lookback=26, forecast_horizon=forecast_horizon)
logger.info("\nParallel Hybrid Model (w=0.5) for Title A...")
# Train LSTM model on raw series
lstm_series_model_title_a = train_lstm(X_series_title_a, y_series_title_a, "Title A", "series")
# Generate parallel hybrid forecast with fixed weight
par_pred_title_a, par_lower_title_a, par_upper_title_a, par_mae_title_a, par_smape_title_a, par_rmse_title_a = forecast_parallel_hybrid(
    title_a_sarima_model, lstm_series_model_title_a, title_a_train, title_a_test,
    series_scaler_title_a, weight=0.5, lookback=26, forecast_horizon=forecast_horizon, title="Title A", sarima_pred=title_a_sarima_pred
)
# Store results if successful
if par_mae_title_a is not None:
    results['Book'].append("Title A")
    results['Model'].append("Parallel Hybrid")
    results['MAE'].append(par_mae_title_a)
    results['MAPE'].append(par_smape_title_a)
    results['RMSE'].append(par_rmse_title_a)
    results['Weight'].append(0.5)
    forecast_dfs.append(pd.DataFrame({
        'Date': title_a_test.index,
        'Book': "Title A",
        'Model': "Parallel Hybrid",
        'Forecast': par_pred_title_a,
        'Lower_CI': par_lower_title_a,
        'Upper_CI': par_upper_title_a,
        'Actual': title_a_test_original
    }))

# Parallel Hybrid (optimised weight)
logger.info("\nOptimising Parallel Hybrid Weights for Title A...")
# Optimise weight and generate forecast
par_best_pred_title_a, par_best_lower_title_a, par_best_upper_title_a, par_best_mae_title_a, par_best_smape_title_a, par_best_rmse_title_a, best_weight_title_a = optimize_parallel_weights(
    title_a_sarima_model, lstm_series_model_title_a, title_a_train, title_a_test,
    series_scaler_title_a, lookback=26, forecast_horizon=forecast_horizon, title="Title A", sarima_pred=title_a_sarima_pred
)
# Store results if successful
if par_best_mae_title_a is not None:
    results['Book'].append("Title A")
    results['Model'].append("Parallel Hybrid (Best Weight)")
    results['MAE'].append(par_best_mae_title_a)
    results['MAPE'].append(par_best_smape_title_a)
    results['RMSE'].append(par_best_rmse_title_a)
    results['Weight'].append(best_weight_title_a)
    forecast_dfs.append(pd.DataFrame({
        'Date': title_a_test.index,
        'Book': "Title A",
        'Model': "Parallel Hybrid (Best Weight)",
        'Forecast': par_best_pred_title_a,
        'Lower_CI': par_best_lower_title_a,
        'Upper_CI': par_best_upper_title_a,
        'Actual': title_a_test_original
    }))

# Process Title B
logger.info("\n===== Hybrid Models for Title B =====")
# Fit SARIMA model and get residuals and predictions
title_b_sarima_model, title_b_residuals, title_b_sarima_pred = fit_sarima(title_b_train, "Title B", forecast_horizon)

# Sequential Hybrid: LSTM on SARIMA residuals
if title_b_residuals is not None:
    # Prepare sequences from residuals
    X_residuals_title_b, y_residuals_title_b, residuals_scaler_title_b = prepare_lstm_sequences(title_b_residuals, lookback=26, forecast_horizon=forecast_horizon)
    logger.info("\nSequential Hybrid Model for Title B...")
    # Train LSTM model on residuals
    lstm_residuals_model_title_b = train_lstm(X_residuals_title_b, y_residuals_title_b, "Title B", "residuals")
    # Generate sequential hybrid forecast
    seq_pred_title_b, seq_lower_title_b, seq_upper_title_b, seq_mae_title_b, seq_smape_title_b, seq_rmse_title_b = forecast_sequential_hybrid(
        title_b_sarima_model, lstm_residuals_model_title_b, title_b_residuals, title_b_test,
        residuals_scaler_title_b, lookback=26, forecast_horizon=forecast_horizon, title="Title B"
    )
    # Store results if successful
    if seq_mae_title_b is not None:
        results['Book'].append("Title B")
        results['Model'].append("Sequential Hybrid")
        results['MAE'].append(seq_mae_title_b)
        results['MAPE'].append(seq_smape_title_b)
        results['RMSE'].append(seq_rmse_title_b)
        results['Weight'].append(None)
        forecast_dfs.append(pd.DataFrame({
            'Date': title_b_test.index,
            'Book': "Title B",
            'Model': "Sequential Hybrid",
            'Forecast': seq_pred_title_b,
            'Lower_CI': seq_lower_title_b,
            'Upper_CI': seq_upper_title_b,
            'Actual': title_b_test_original
        }))
else:
    # Fallback to mean forecast if residuals are unavailable
    seq_pred_title_b, seq_lower_title_b, seq_upper_title_b, seq_mae_title_b, seq_smape_title_b, seq_rmse_title_b = forecast_fallback(
        title_b_train, title_b_test, "Title B", forecast_horizon
    )
    if seq_mae_title_b is not None:
        results['Book'].append("Title B")
        results['Model'].append("Sequential Hybrid")
        results['MAE'].append(seq_mae_title_b)
        results['MAPE'].append(seq_smape_title_b)
        results['RMSE'].append(seq_rmse_title_b)
        results['Weight'].append(None)
        forecast_dfs.append(pd.DataFrame({
            'Date': title_b_test.index,
            'Book': "Title B",
            'Model': "Sequential Hybrid",
            'Forecast': seq_pred_title_b,
            'Lower_CI': seq_lower_title_b,
            'Upper_CI': seq_upper_title_b,
            'Actual': title_b_test_original
        }))

# Parallel Hybrid (fixed weight = 0.5)
X_series_title_b, y_series_title_b, series_scaler_title_b = prepare_lstm_sequences(title_b_train, lookback=26, forecast_horizon=forecast_horizon)
logger.info("\nParallel Hybrid Model (w=0.5) for Title B...")
# Train LSTM model on raw series
lstm_series_model_title_b = train_lstm(X_series_title_b, y_series_title_b, "Title B", "series")
# Generate parallel hybrid forecast with fixed weight
par_pred_title_b, par_lower_title_b, par_upper_title_b, par_mae_title_b, par_smape_title_b, par_rmse_title_b = forecast_parallel_hybrid(
    title_b_sarima_model, lstm_series_model_title_b, title_b_train, title_b_test,
    series_scaler_title_b, weight=0.5, lookback=26, forecast_horizon=forecast_horizon, title="Title B", sarima_pred=title_b_sarima_pred
)

# Store results if successful
if par_mae_title_b is not None:
    results['Book'].append("Title B")
    results['Model'].append("Parallel Hybrid")
    results['MAE'].append(par_mae_title_b)
    results['MAPE'].append(par_smape_title_b)
    results['RMSE'].append(par_rmse_title_b)
    results['Weight'].append(0.5)
    forecast_dfs.append(pd.DataFrame({
        'Date': title_b_test.index,
        'Book': "Title B",
        'Model': "Parallel Hybrid",
        'Forecast': par_pred_title_b,
        'Lower_CI': par_lower_title_b,
        'Upper_CI': par_upper_title_b,
        'Actual': title_b_test_original
    }))

# Parallel Hybrid (optimized weight)
logger.info("\nOptimising Parallel Hybrid Weights for Title B...")
# Optimise weight and generate forecast
par_best_pred_title_b, par_best_lower_title_b, par_best_upper_title_b, par_best_mae_title_b, par_best_smape_title_b, par_best_rmse_title_b, best_weight_title_b = optimize_parallel_weights(
    title_b_sarima_model, lstm_series_model_title_b, title_b_train, title_b_test,
    series_scaler_title_b, lookback=26, forecast_horizon=forecast_horizon, title="Title B", sarima_pred=title_b_sarima_pred
)
# Store results if successful
if par_best_mae_title_b is not None:
    results['Book'].append("Title B")
    results['Model'].append("Parallel Hybrid (Best Weight)")
    results['MAE'].append(par_best_mae_title_b)
    results['MAPE'].append(par_best_smape_title_b)
    results['RMSE'].append(par_best_rmse_title_b)
    results['Weight'].append(best_weight_title_b)
    forecast_dfs.append(pd.DataFrame({
        'Date': title_b_test.index,
        'Book': "Title B",
        'Model': "Parallel Hybrid (Best Weight)",
        'Forecast': par_best_pred_title_b,
        'Lower_CI': par_best_lower_title_b,
        'Upper_CI': par_best_upper_title_b,
        'Actual': title_b_test_original
    }))

# Create results DataFrame
results_df = pd.DataFrame(results)
# Round metrics to 2 decimal places
results_df['MAE'] = results_df['MAE'].round(2)
results_df['MAPE'] = results_df['MAPE'].round(2)
results_df['RMSE'] = results_df['RMSE'].round(2)
# Handle None weights by rounding to 1 decimal place where applicable
results_df['Weight'] = results_df['Weight'].apply(lambda x: round(x, 1) if x is not None else None)

# Display results table
logger.info("\n=== Hybrid Model Results Table ===")
print(tabulate(results_df, headers='keys', tablefmt='pretty', showindex=False))

# Combine forecast DataFrames
forecasts_combined = pd.concat(forecast_dfs, ignore_index=False) if forecast_dfs else pd.DataFrame()
if not forecasts_combined.empty:
    forecasts_combined = forecasts_combined.sort_values(['Date', 'Book', 'Model'])
    logger.info("\n=== Forecast Sample Table ===")
    print(tabulate(forecasts_combined.head(10), headers='keys', tablefmt='pretty'))

# Export to Excel
output_path = './hybrid_results.xlsx'
alternative_path = './hybrid_results.xlsx'
try:
    # Log DataFrames for debugging
    logger.info(f"Results DataFrame: \n{results_df}")
    logger.info(f"Forecasts Combined: \n{forecasts_combined.head() if not forecasts_combined.empty else 'Empty'}")
    # Export metrics and forecasts to Excel
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        results_df.to_excel(writer, sheet_name='Metrics', index=False)
        forecasts_combined.to_excel(writer, sheet_name='Forecasts', index=False) if not forecasts_combined.empty else pd.DataFrame().to_excel(writer, sheet_name='Forecasts', index=False)
    logger.info(f"Results exported to {output_path}")
except Exception as e:
    logger.error(f"Error exporting to Excel with openpyxl: {e}")
    try:
        # Try alternative engine (xlsxwriter)
        with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
            results_df.to_excel(writer, sheet_name='Metrics', index=False)
            forecasts_combined.to_excel(writer, sheet_name='Forecasts', index=False) if not forecasts_combined.empty else pd.DataFrame().to_excel(writer, sheet_name='Forecasts', index=False)
        logger.info(f"Exported to {output_path} using xlsxwriter")
    except Exception as e2:
        logger.error(f"Export failed with xlsxwriter: {e2}")
        try:
            # Fallback to alternative path with openpyxl
            with pd.ExcelWriter(alternative_path, engine='openpyxl') as writer:
                results_df.to_excel(writer, sheet_name='Metrics', index=False)
            logger.info(f"Fallback: Exported to {alternative_path}")
        except Exception as e3:
            logger.error(f"Fallback export failed: {e3}")

# Compare with previous models
try:
    # Load metrics from classical and ML/DL models
    classical_metrics = pd.read_excel('./classical_techniques_results.xlsx', sheet_name='Metrics')
    ml_dl_metrics = pd.read_excel('./ml_dl_results.xlsx', sheet_name='Metrics')
    hybrid_metrics = results_df

    # Display metrics tables
    logger.info("\n=== Comparison with Previous Models ===")
    print("Classical Techniques Metrics:")
    print(tabulate(classical_metrics, headers='keys', tablefmt='pretty', showindex=False))
    print("\nMachine Learning and Deep Learning Metrics:")
    print(tabulate(ml_dl_metrics, headers='keys', tablefmt='pretty', showindex=False))
    print("\nHybrid Model Metrics:")
    print(tabulate(hybrid_metrics, headers='keys', tablefmt='pretty', showindex=False))
    classical_metrics_standardized = classical_metrics[['Book', 'Decomposition_Model', 'classical_MAE']].rename(
        columns={'Decomposition_Model': 'Model', 'classical_MAE': 'MAE'}
    )
    # StandardiSe classical metrics for comparison
    classical_metrics_standardized['Model'] = classical_metrics_standardized['Model'].apply(lambda x: f"Classical ({x})")

    # Combine metrics from all models
    combined_metrics = pd.concat([
        classical_metrics_standardized[['Book', 'Model', 'MAE']],
        ml_dl_metrics[['Book', 'Model', 'MAE']],
        hybrid_metrics[['Book', 'Model', 'MAE']]
    ])

    # Create a pivot table comparing MAE across models
    pivot_table = combined_metrics.pivot_table(
        values='MAE',
        index='Book',
        columns='Model',
        aggfunc='mean'
    ).round(2)
    print("\nPivoted Metrics Comparison (MAE):")
    print(tabulate(pivot_table, headers='keys', tablefmt='pretty'))
except FileNotFoundError as e:
    logger.error(f"Error loading previous metrics: {e}")
except Exception as e:
    logger.error(f"Error in comparison: {e}")

# This code processes sequential and parallel hybrid SARIMA-LSTM models, optimises weights,
# compiles and exports results, and compares with classical and ML/DL models.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tabulate import tabulate
import logging

# Ensure logging is set up (matches Section 5)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Function to calculate MAPE with handling for zero/near-zero actual values
def calculate_mape(actual, forecast, title, epsilon=1e-10):
    """
    Calculate MAPE (%) for given actual and forecast values.
    Parameters:
        actual (pd.Series): Actual test values.
        forecast (pd.Series): Forecasted values.
        title (str): Book title for logging.
        epsilon (float): Small value to avoid division by zero.
    Returns:
        float: MAPE (%) value, or None if calculation fails.
    """
    try:
        actual = actual.reindex(forecast.index).fillna(actual.mean() or epsilon)
        forecast = forecast.fillna(forecast.mean() or epsilon)
        if actual.isna().any() or forecast.isna().any():
            logger.error(f"NaN values detected in actual or forecast for {title}")
            return None
        if np.any(np.isinf(actual)) or np.any(np.isinf(forecast)):
            logger.error(f"Infinite values detected in actual or forecast for {title}")
            return None
        abs_error = np.abs(actual - forecast)
        denominator = np.abs(actual) + epsilon
        percentage_errors = abs_error / denominator
        mape = np.mean(percentage_errors) * 100
        if np.isnan(mape) or np.isinf(mape):
            logger.error(f"Invalid MAPE calculated for {title}: {mape}")
            return None
        logger.info(f"MAPE for {title}: {mape:.2f}%")
        return mape
    except Exception as e:
        logger.error(f"Error calculating MAPE for {title}: {e}")
        return None

# Function to print comparison table of key metrics
def print_metrics_comparison_table(results_df):
    """
    Print a table comparing key metrics (MAE, MAPE, RMSE, Weight) for all models in Section 5.
    Parameters:
        results_df (pd.DataFrame): DataFrame containing model metrics.
    """
    try:
        required_columns = ['Book', 'Model', 'MAE', 'MAPE', 'RMSE', 'Weight']
        if results_df is None or results_df.empty or not all(col in results_df.columns for col in required_columns):
            logger.error("Results DataFrame is empty or missing required columns.")
            print("Error: No results available to display.")
            return
        display_df = results_df.copy()
        display_df['MAE'] = display_df['MAE'].round(2).astype(str)
        display_df['MAPE'] = display_df['MAPE'].apply(lambda x: f"{x:.2f}%" if pd.notnull(x) else '-')
        display_df['RMSE'] = display_df['RMSE'].round(2).astype(str)
        display_df['Weight'] = display_df['Weight'].apply(lambda x: f"{x:.1f}" if pd.notnull(x) else '-')
        display_df = display_df.sort_values(['Book', 'Model'])
        display_df = display_df[['Book', 'Model', 'MAE', 'MAPE', 'RMSE', 'Weight']]
        logger.info("\n=== Comparison Table of Key Metrics for Section 5 Models ===")
        print("\nComparison Table of Key Metrics for Section 5 Models:")
        print(tabulate(display_df, headers=['Book', 'Model', 'MAE', 'MAPE (%)', 'RMSE', 'Weight'], tablefmt='pretty', showindex=False))
    except Exception as e:
        logger.error(f"Error printing comparison table: {e}")
        print(f"Error: Failed to print comparison table - {e}")

# Calculate ARIMA MAPE and print comparison table
def compute_arima_mape_and_compare():
    """
    Calculate ARIMA MAPE, update results, and print comparison table.
    Loads existing results from Excel if results dictionary is missing.
    """
    global results, results_df
    try:
        # Try to load existing results from Excel if results dictionary is missing
        if 'results' not in globals():
            logger.warning("Results dictionary not found. Attempting to load from hybrid_results.xlsx.")
            try:
                results_df = pd.read_excel('./hybrid_results.xlsx', sheet_name='Metrics')
                results = results_df.to_dict('list')
                logger.info("Loaded existing results from hybrid_results.xlsx.")
            except FileNotFoundError:
                logger.warning("hybrid_results.xlsx not found. Initializing new dictionary.")
                results = {'Book': [], 'Model': [], 'MAE': [], 'MAPE': [], 'RMSE': [], 'Weight': []}
        elif results_df is not None and not results_df.empty:
            results = results_df.to_dict('list')  # Use existing results_df if available

        # Check for required variables
        required_vars = [
            ('title_a_sarima_pred', 'title_a_test_original', "Title A"),
            ('title_b_sarima_pred', 'title_b_test_original', "Title B")
        ]
        for pred_var, actual_var, title in required_vars:
            if pred_var not in globals() or actual_var not in globals():
                logger.warning(f"Variable {pred_var} or {actual_var} not defined for {title}. Skipping MAPE calculation.")
                continue
            pred = globals()[pred_var]
            actual = globals()[actual_var]
            if pred is not None and not pred.isna().all():
                mape = calculate_mape(actual, pred, title)
                if mape is not None:
                    results['Book'].append(title)
                    results['Model'].append("ARIMA")
                    mae = mean_absolute_error(actual, pred)
                    rmse = np.sqrt(mean_squared_error(actual, pred))
                    results['MAE'].append(mae)
                    results['MAPE'].append(mape)
                    results['RMSE'].append(rmse)
                    results['Weight'].append(None)
                    print(f"\narima_MAPE for {title}: {mape:.2f}%")
                    print(f"arima_MAE for {title}: {mae:.2f}")
                    print(f"arima_RMSE for {title}: {rmse:.2f}")
            else:
                logger.warning(f"No valid SARIMA predictions for {title}. Skipping MAPE calculation.")

        # Update results DataFrame
        results_df = pd.DataFrame(results)
        if not results_df.empty:
            results_df['MAE'] = results_df['MAE'].round(2)
            results_df['MAPE'] = results_df['MAPE'].round(2)
            results_df['RMSE'] = results_df['RMSE'].round(2)
            results_df['Weight'] = results_df['Weight'].apply(lambda x: round(x, 1) if x is not None else None)

            # Display updated results table
            logger.info("\n=== Updated Results Table with ARIMA MAPE ===")
            print(tabulate(results_df, headers='keys', tablefmt='pretty', showindex=False))

            # Export updated results to Excel
            output_path = './hybrid_results.xlsx'
            alternative_path = './hybrid_results.xlsx'
            try:
                with pd.ExcelWriter(output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                    results_df.to_excel(writer, sheet_name='Metrics', index=False)
                logger.info(f"Updated results exported to {output_path}")
            except Exception as e:
                logger.error(f"Error exporting updated results: {e}")
                try:
                    with pd.ExcelWriter(alternative_path, engine='openpyxl') as writer:
                        results_df.to_excel(writer, sheet_name='Metrics', index=False)
                    logger.info(f"Fallback: Exported to {alternative_path}")
                except Exception as e2:
                    logger.error(f"Fallback export failed: {e2}")

        # Print comparison table
        print_metrics_comparison_table(results_df)

    except Exception as e:
        logger.error(f"Error in compute_arima_mape_and_compare: {e}")
        print(f"Error: Failed to compute ARIMA MAPE or print comparison table - {e}")

# Run the combined function
if __name__ == "__main__":
    compute_arima_mape_and_compare()

## **Section 6 - Monthly prediction**

In [ ]:
# Install required packages (minimized to avoid conflicts)
!pip install pmdarima==2.0.4 xgboost==2.0.3 tabulate==0.9.0 openpyxl==3.1.5 scikit-learn==1.3.2 -q

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
from tabulate import tabulate
pass  # Colab Drive import removed - place data files in ./data
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Mount cloud storage
try:
    pd.read_csv('./two_books_analysis.csv', nrows=1)
except:
    pass  # drive.mount(...) removed for local execution

# Load data
data_path = './two_books_analysis.csv'
try:
    data = pd.read_csv(data_path)
    data['Interval'] = pd.to_datetime(data['Interval'])
    data.set_index('Interval', inplace=True)
except FileNotFoundError:
    print("Error: Data file not found.")
    raise
except Exception as e:
    print(f"Error loading data: {e}")
    raise

In [ ]:
# Split data by book (case-insensitive)
title_a_data = data[data['Title'].str.lower() == 'title_a, the']['Volume']
title_b_data = data[data['Title'].str.lower() == 'very hungry title_b, the']['Volume']

In [ ]:
# Validate data
if title_a_data.empty or title_b_data.empty:
    raise ValueError("Missing data for one or both books.")

In [ ]:
# **Criteria 34: Aggregate to Monthly Data**

# Preprocess data
def preprocess_series(series, apply_log=False):
    # Aggregate to weekly if daily
    time_diffs = series.index[1:] - series.index[:-1]
    if len(time_diffs) > 0 and time_diffs[0] == pd.Timedelta(days=1):
        print("Data appears to be daily. Aggregating to weekly first.")
        series = series.resample('W').sum()
    # Impute zeros
    series = series.replace(0, np.nan).interpolate(method='time')
    series = series.fillna(method='bfill').fillna(method='ffill')
    # Aggregate to monthly
    series = series.resample('M').sum()
    if apply_log:
        if (series <= 0).any():
            print("Warning: Series contains non-positive values; adjusting for log transform.")
            series = series.clip(lower=1e-10)
        series = np.log1p(series)
    return series

title_a_series = preprocess_series(title_a_data, apply_log=False)
title_b_series = preprocess_series(title_b_data, apply_log=True)

In [ ]:
# Filter post-2012-01-01
title_a_series = title_a_series[title_a_series.index >= '2012-01-01']
title_b_series = title_b_series[title_b_series.index >= '2012-01-01']

In [ ]:
# Define forecast horizon
forecast_horizon_months = 8
min_train_size = 50

**remove the 2 following cells later**

In [ ]:
import pandas as pd
import numpy as np

# Modifying this function again for this section
def ensure_test_size(data, test_size, freq='W-SUN', date_column='Date', value_column='Volume'):
    """
    Ensure the test set has the specified size, padding with zeros if needed.

    Parameters:
    - data: DataFrame or Series with time series data
    - test_size: Desired number of periods (weeks or months)
    - freq: Frequency ('W-SUN' for weekly, 'M' for monthly)
    - date_column: Name of the date column
    - value_column: Name of the value column (e.g., 'Volume')

    Returns:
    - Series with exactly test_size rows and a DatetimeIndex
    """
    print(f"=== ensure_test_size called with test_size={test_size}, freq={freq}, date_column={date_column}, value_column={value_column} ===")
    print(f"Input data type: {type(data)}, shape: {data.shape}")
    print(f"Input data index type: {type(data.index)}")

    # Convert to DataFrame if Series
    if isinstance(data, pd.Series):
        print("Converting Series to DataFrame...")
        data = data.to_frame(name=value_column)
        data[date_column] = data.index
        data.reset_index(drop=True, inplace=True)

    print(f"Data head after conversion:\n{data.head()}")

    # Ensure date_column exists
    if date_column not in data.columns:
        raise ValueError(f"Date column '{date_column}' not found in data. Available columns: {data.columns}")

    # Ensure date_column is datetime
    print(f"Converting '{date_column}' to datetime...")
    data[date_column] = pd.to_datetime(data[date_column], errors='coerce')
    if data[date_column].isna().all():
        raise ValueError(f"All dates in '{date_column}' are invalid (NaT). Check input data.")

    # Get last valid date from the index if data is a Series
    if isinstance(data, pd.Series):
        last_date = data.index.max()  # Change here to access index
    else:
        last_date = data[date_column].max()

    if pd.isna(last_date):
        raise ValueError(f"Last date in '{date_column}' is NaT. Check input data.")

    print(f"Last date in data: {last_date}")

    # Create test set dates (start one week after last_date)
    print("Generating test dates...")
    # test_dates = pd.date_range(start=last_date + pd.Timedelta(days=7), periods=test_size, freq=freq, name=date_column)
    test_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=test_size, freq=freq, name=date_column) # Changed to handle monthly frequency correctly

    # Debug: Print the first and last test dates
    print(f"Generated test dates: {test_dates[0]} to {test_dates[-1]}")
    print(f"Generated test dates length: {len(test_dates)}")

    # Create new test data with zeros
    print("Creating test data with zeros...")
    test_data = pd.DataFrame({
        date_column: test_dates,
        value_column: 0,  # Future data, set to 0
        'ISBN': data['ISBN'].iloc[0] if 'ISBN' in data.columns else None,
        'Title': data['Title'].iloc[0] if 'Title' in data.columns else None
    })

    # Convert to Series with DatetimeIndex
    print("Converting test data to Series...")
    result = test_data.set_index(date_column)[value_column]

    # Validate the result
    if len(result) != test_size:
        raise ValueError(f"Test set length ({len(result)}) does not match test_size ({test_size})")
    if result.index.min() != last_date + pd.DateOffset(months=1) if freq == 'M' else last_date + pd.Timedelta(days=7): # Changed to handle monthly frequency correctly
        raise ValueError(f"Test set start date ({result.index.min()}) does not match expected ({last_date + pd.DateOffset(months=1) if freq == 'M' else last_date + pd.Timedelta(days=7)})")  # Changed to handle monthly frequency correctly
    if result.index.max() != test_dates[-1]:
        raise ValueError(f"Test set end date ({result.index.max()}) does not match expected ({test_dates[-1]})")

    print(f"Output test set shape: {result.shape}, index type: {type(result.index)}")
    print(f"Test set date range: {result.index.min()} to {result.index.max()}")
    print("=== ensure_test_size completed ===")

    return result

In [ ]:
# Split data
title_a_test_monthly = ensure_test_size(title_a_series, test_size=forecast_horizon_months, freq='M')
title_a_train_monthly = title_a_series.iloc[:-forecast_horizon_months]
title_b_test_monthly = ensure_test_size(title_b_series, test_size=forecast_horizon_months, freq='M')
title_b_train_monthly = title_b_series.iloc[:-forecast_horizon_months]

In [ ]:
# Store original test series
title_a_test_original = title_a_data.resample('M').sum()[title_a_data.resample('M').sum().index >= '2012-01-01'].iloc[-forecast_horizon_months:]
title_b_test_original = title_b_data.resample('M').sum()[title_b_data.resample('M').sum().index >= '2012-01-01'].iloc[-forecast_horizon_months:]

In [ ]:
# Validate data
if len(title_a_train_monthly) < min_train_size:
    raise ValueError(f"Title_A monthly training data too short: {len(title_a_train_monthly)} months, need {min_train_size}")
if len(title_b_train_monthly) < min_train_size:
    raise ValueError(f"Title_B monthly training data too short: {len(title_b_train_monthly)} months, need {min_train_size}")
if len(title_a_test_monthly) != forecast_horizon_months or len(title_b_test_monthly) != forecast_horizon_months:
    raise ValueError(f"Test data size mismatch: Expected {forecast_horizon_months}, got {len(title_a_test_monthly)} for Title_A, {len(title_b_test_monthly)} for Title_B")

In [ ]:
# Print data ranges
print("Title_A Monthly Train Range:", title_a_train_monthly.index.min(), "to", title_a_train_monthly.index.max(), "size:", len(title_a_train_monthly))
print("Title_A Monthly Test Range:", title_a_test_monthly.index.min(), "to", title_a_test_monthly.index.max(), "size:", len(title_a_test_monthly))
print("Title_B Monthly Train Range:", title_b_train_monthly.index.min(), "to", title_b_train_monthly.index.max(), "size:", len(title_b_train_monthly))
print("Title_B Monthly Test Range:", title_b_test_monthly.index.min(), "to", title_b_test_monthly.index.max(), "size:", len(title_b_test_monthly))

In [ ]:
# **Criteria 35: XGBoost Model for Monthly Data**

# Prepare XGBoost features
def prepare_xgboost_features_monthly(series, lags=[1, 2, 3, 12], windows=[3, 6, 12]):
    df = pd.DataFrame(series, columns=['Volume'])
    for lag in lags:
        df[f'lag_{lag}'] = df['Volume'].shift(lag)
    for window in windows:
        df[f'roll_mean_{window}'] = df['Volume'].rolling(window=window).mean()
        df[f'roll_std_{window}'] = df['Volume'].rolling(window=window).std()
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['is_december'] = (df.index.month == 12).astype(int)
    df = df.dropna()
    if df.isna().any().any() or np.isinf(df).any().any():
        raise ValueError("Features contain NaN or Inf after preparation")
    return df

In [ ]:
# Prepare monthly data
title_a_xgb_data_monthly = prepare_xgboost_features_monthly(title_a_series)
title_b_xgb_data_monthly = prepare_xgboost_features_monthly(title_b_series)

In [ ]:
# Split XGBoost data
title_a_xgb_train_monthly = title_a_xgb_data_monthly.iloc[:-forecast_horizon_months]
title_a_xgb_test_monthly = title_a_xgb_data_monthly.iloc[-forecast_horizon_months:]
title_b_xgb_train_monthly = title_b_xgb_data_monthly.iloc[:-forecast_horizon_months]
title_b_xgb_test_monthly = title_b_xgb_data_monthly.iloc[-forecast_horizon_months:]

In [ ]:
# Define XGBoost pipeline
xgb_pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('xgb', XGBRegressor(objective='reg:squarederror', random_state=42))
])

In [ ]:
# Cross-validation and grid search
tscv = TimeSeriesSplit(n_splits=3, test_size=2)
param_grid = {
    'xgb__n_estimators': [100, 200],
    'xgb__max_depth': [3, 5],
    'xgb__learning_rate': [0.01, 0.1],
    'xgb__subsample': [0.8, 1.0],
    'xgb__colsample_bytree': [0.8, 1.0]
}

In [ ]:
# Run XGBoost grid search
def run_xgboost_grid_search_monthly(X_train, y_train, book_title):
    try:
        grid_search = GridSearchCV(
            xgb_pipeline,
            param_grid,
            cv=tscv,
            scoring='neg_mean_absolute_error',
            n_jobs=-1,
            verbose=1
        )
        grid_search.fit(X_train.drop(columns=['Volume']), y_train)
        print(f"\nBest Parameters for {book_title} (Monthly): {grid_search.best_params_}")
        print(f"Best CV MAE: {-grid_search.best_score_:.2f}")
        return grid_search.best_estimator_
    except Exception as e:
        print(f"Error in XGBoost grid search for {book_title}: {e}")
        return None

In [ ]:
# Train quantile models
def train_quantile_models(X_train, y_train, max_depth=3):
    try:
        lower_model = GradientBoostingRegressor(loss='quantile', alpha=0.025, n_estimators=200, max_depth=max_depth, learning_rate=0.1, random_state=42)
        lower_model.fit(X_train.drop(columns=['Volume']), y_train)
        upper_model = GradientBoostingRegressor(loss='quantile', alpha=0.975, n_estimators=200, max_depth=max_depth, learning_rate=0.1, random_state=42)
        upper_model.fit(X_train.drop(columns=['Volume']), y_train)
        return lower_model, upper_model
    except Exception as e:
        print(f"Error training quantile models: {e}")
        return None, None

In [ ]:
# Rolling forecast with XGBoost
# Disclosure: The code below was created with AI's assistance
def forecast_xgboost_monthly(model, train_data, test_data, title, lower_model=None, upper_model=None, apply_log=False, lags=[1, 2, 3, 12], windows=[3, 6, 12]):
    try:
        if model is None:
            raise ValueError("Invalid XGBoost model")
        current_data = train_data.copy()
        predictions = []
        lower_cis = []
        upper_cis = []

        for step in range(forecast_horizon_months):
            X_pred = current_data.drop(columns=['Volume']).iloc[-1:]
            y_pred = model.predict(X_pred)[0]
            predictions.append(y_pred)
            if lower_model and upper_model:
                lower_ci = lower_model.predict(X_pred)[0]
                upper_ci = upper_model.predict(X_pred)[0]
                lower_cis.append(lower_ci)
                upper_cis.append(upper_ci)
            else:
                lower_cis.append(np.nan)
                upper_cis.append(np.nan)

            next_date = test_data.index[step]
            new_row = pd.DataFrame({
                'Volume': [y_pred],
                'month': [next_date.month],
                'year': [next_date.year],
                'is_december': [1 if next_date.month == 12 else 0]
            }, index=[next_date])
            for lag in lags:
                new_row[f'lag_{lag}'] = current_data['Volume'].shift(lag).iloc[-1] if lag <= len(current_data) else np.nan
            for window in windows:
                new_row[f'roll_mean_{window}'] = current_data['Volume'].iloc[-window:].mean() if len(current_data) >= window else np.nan
                new_row[f'roll_std_{window}'] = current_data['Volume'].iloc[-window:].std() if len(current_data) >= window else np.nan
            current_data = pd.concat([current_data, new_row])
            current_data = current_data.fillna(current_data.mean())

        y_pred = np.array(predictions)
        lower_ci = np.array(lower_cis)
        upper_ci = np.array(upper_cis)

        actual = title_b_test_original if apply_log else title_a_test_original
        if apply_log:
            y_pred = np.expm1(y_pred)
            lower_ci = np.expm1(lower_ci)
            upper_ci = np.expm1(upper_ci)

        mae = mean_absolute_error(actual, y_pred)
        smape = np.mean(2 * np.abs(actual - y_pred) / (np.abs(actual) + np.abs(y_pred) + 1e-10)) * 100
        rmse = np.sqrt(mean_squared_error(actual, y_pred))
        mape = mean_absolute_percentage_error(actual, y_pred) * 100

        plt.figure(figsize=(12, 6))
        plt.plot(train_data.index[-24:], train_data['Volume'][-24:], label='Training Data (Last 24 Months)')
        plt.plot(test_data.index, actual, label='Actual Test Data', color='green')
        plt.plot(test_data.index, y_pred, label='Monthly XGBoost Forecast', color='red')
        plt.fill_between(test_data.index, lower_ci, upper_ci, color='red', alpha=0.2, label='95% Confidence Interval')
        plt.title(f'{title} Monthly XGBoost Forecast ({forecast_horizon_months} Months)')
        plt.xlabel('Date')
        plt.ylabel('Sales Volume')
        plt.legend()
        plt.show()

        print(f"\nmonthly_xgb_MAE for {title}: {mae:.2f}")
        print(f"monthly_xgb_sMAPE for {title}: {smape:.2f}%")
        print(f"monthly_xgb_RMSE for {title}: {rmse:.2f}")
        print(f"monthly_xgb_MAPE for {title}: {mape:.2f}%")
        coverage = np.mean((actual >= lower_ci) & (actual <= upper_ci)) * 100
        print(f"CI Coverage for {title}: {coverage:.2f}%")

        return y_pred, lower_ci, upper_ci, mae, smape, rmse, mape
    except Exception as e:
        print(f"Error in XGBoost forecast for {title}: {e}")
        return None, None, None, None, None, None, None

In [ ]:
# Train and forecast with XGBoost
title_a_xgb_model_monthly = run_xgboost_grid_search_monthly(
    title_a_xgb_train_monthly, title_a_xgb_train_monthly['Volume'], "Title A"
)
title_a_lower_model, title_a_upper_model = train_quantile_models(title_a_xgb_train_monthly, title_a_xgb_train_monthly['Volume'], max_depth=3)
title_a_xgb_pred_monthly, title_a_xgb_lower, title_a_xgb_upper, title_a_xgb_mae_monthly, title_a_xgb_smape_monthly, title_a_xgb_rmse_monthly, title_a_xgb_mape_monthly = forecast_xgboost_monthly(
    title_a_xgb_model_monthly, title_a_xgb_train_monthly, title_a_xgb_test_monthly, "Title A", title_a_lower_model, title_a_upper_model, apply_log=False
)

title_b_xgb_model_monthly = run_xgboost_grid_search_monthly(
    title_b_xgb_train_monthly, title_b_xgb_train_monthly['Volume'], "Title B"
)
title_b_lower_model, title_b_upper_model = train_quantile_models(title_b_xgb_train_monthly, title_b_xgb_train_monthly['Volume'], max_depth=3)
title_b_xgb_pred_monthly, title_b_xgb_lower, title_b_xgb_upper, title_b_xgb_mae_monthly, title_b_xgb_smape_monthly, title_b_xgb_rmse_monthly, title_b_xgb_mape_monthly = forecast_xgboost_monthly(
    title_b_xgb_model_monthly, title_b_xgb_train_monthly, title_b_xgb_test_monthly, "Title B", title_b_lower_model, title_b_upper_model, apply_log=True
)

In [ ]:
# **Criteria 36: SARIMA Model for Monthly Data**

# Fit SARIMA with exogenous variables
def fit_sarima_monthly(series, exog, title):
    print(f"\nFitting SARIMA for {title} (Monthly)...")
    try:
        model = auto_arima(
            series,
            exogenous=exog,
            seasonal=True,
            m=12,
            start_p=0, max_p=2,
            start_q=0, max_q=2,
            d=None, max_d=1,
            start_P=0, max_P=1,
            start_Q=0, max_Q=1,
            D=None, max_D=1,
            stepwise=True,
            suppress_warnings=True,
            trace=True
        )
        return model
    except Exception as e:
        print(f"Error fitting SARIMA for {title}: {e}")
        return None

In [ ]:
# Prepare exogenous variables
def prepare_exog(series):
    df = pd.DataFrame(index=series.index)
    df['is_december'] = (df.index.month == 12).astype(int)  # Only December indicator
    return df

In [ ]:
# Forecast with SARIMA
# Disclosure: the code below was created with AI's assistance
def forecast_sarima_monthly(model, train_series, test_series, train_exog, test_exog, title, apply_log=False):
    try:
        if model is None:
            raise ValueError("Invalid SARIMA model")
        pred, conf_int = model.predict(n_periods=len(test_series), exogenous=test_exog, return_conf_int=True)
        pred_series = pd.Series(pred, index=test_series.index)
        conf_int_df = pd.DataFrame(conf_int, index=test_series.index, columns=['Lower', 'Upper'])

        actual = title_b_test_original if apply_log else title_a_test_original
        if apply_log:
            pred_series = np.expm1(pred_series)
            conf_int_df['Lower'] = np.expm1(conf_int_df['Lower'])
            conf_int_df['Upper'] = np.expm1(conf_int_df['Upper'])

        mae = mean_absolute_error(actual, pred_series)
        smape = np.mean(2 * np.abs(actual - pred_series) / (np.abs(actual) + np.abs(pred_series) + 1e-10)) * 100
        rmse = np.sqrt(mean_squared_error(actual, pred_series))
        mape = mean_absolute_percentage_error(actual, pred_series) * 100

        # Check for None values in metrics and replace with a placeholder
        if mae is None: mae = np.nan
        if smape is None: smape = np.nan
        if rmse is None: rmse = np.nan  # Replace None with np.nan for RMSE
        if mape is None: mape = np.nan

        # Create plot
        plt.figure(figsize=(12, 6))
        plt.plot(train_series.index[-24:], train_series[-24:], label='Training Data (Last 24 Months)')
        plt.plot(test_series.index, actual, label='Actual Test Data', color='green')
        plt.plot(test_series.index, pred_series, label='Monthly SARIMA Forecast', color='red')
        plt.fill_between(test_series.index, conf_int_df['Lower'], conf_int_df['Upper'], color='red', alpha=0.2, label='95% Confidence Interval')
        plt.title(f'{title} Monthly SARIMA Forecast ({forecast_horizon_months} Months)')
        plt.xlabel('Date')
        plt.ylabel('Sales Volume')
        plt.legend()
        plt.show()

        print(f"\nmonthly_sarima_MAE for {title}: {mae:.2f}")
        print(f"monthly_sarima_sMAPE for {title}: {smape:.2f}%")
        print(f"monthly_sarima_RMSE for {title}: {rmse:.2f}")
        print(f"monthly_sarima_MAPE for {title}: {mape:.2f}%")
        coverage = np.mean((actual >= conf_int_df['Lower']) & (actual <= conf_int_df['Upper'])) * 100
        print(f"CI Coverage for {title}: {coverage:.2f}%")

        return pred_series, conf_int_df['Lower'], conf_int_df['Upper'], mae, smape, rmse, mape
    except Exception as e:
        print(f"Error in SARIMA forecast for {title}: {e}")
        # Return np.nan for all metrics in case of an error
        return None, None, None, np.nan, np.nan, np.nan, np.nan

In [ ]:
# Fit and forecast with SARIMA
title_a_exog = prepare_exog(title_a_series)
title_a_exog_train = title_a_exog.iloc[:-forecast_horizon_months]
title_a_exog_test = title_a_exog.iloc[-forecast_horizon_months:]
title_a_sarima_model_monthly = fit_sarima_monthly(title_a_train_monthly, title_a_exog_train, "Title A")
title_a_sarima_pred_monthly, title_a_sarima_lower, title_a_sarima_upper, title_a_sarima_mae_monthly, title_a_sarima_smape_monthly, title_a_sarima_rmse_monthly, title_a_sarima_mape_monthly = forecast_sarima_monthly(
    title_a_sarima_model_monthly, title_a_train_monthly, title_a_test_monthly, title_a_exog_train, title_a_exog_test, "Title A", apply_log=False
)

title_b_exog = prepare_exog(title_b_series)
title_b_exog_train = title_b_exog.iloc[:-forecast_horizon_months]
title_b_exog_test = title_b_exog.iloc[-forecast_horizon_months:]
title_b_sarima_model_monthly = fit_sarima_monthly(title_b_train_monthly, title_b_exog_train, "Title B")
title_b_sarima_pred_monthly, title_b_sarima_lower, title_b_sarima_upper, title_b_sarima_mae_monthly, title_b_sarima_smape_monthly, title_b_sarima_rmse_monthly, title_b_sarima_mape_monthly = forecast_sarima_monthly(
    title_b_sarima_model_monthly, title_b_train_monthly, title_b_test_monthly, title_b_exog_train, title_b_exog_test, "Title B", apply_log=True
)

In [ ]:
# **Criteria 37: Compare Weekly vs. Monthly Predictions**

# Load weekly metrics
weekly_metrics_files = [
    './classical_techniques_results.xlsx',
    './ml_dl_results.xlsx',
    './hybrid_results.xlsx'
]
weekly_metrics = {}
for file in weekly_metrics_files:
    try:
        metrics_df = pd.read_excel(file, sheet_name='Metrics')
        for _, row in metrics_df.iterrows():
            model = row.get('Model', 'SARIMA')
            book = row['Book']
            mae = row.get('MAE', float('inf'))
            smape = row.get('sMAPE', float('inf'))
            rmse = row.get('RMSE', float('inf'))
            weekly_metrics[f"{book} - {model}"] = {'MAE': mae, 'sMAPE': smape, 'RMSE': rmse}
    except Exception as e:
        print(f"Error loading {file}: {e}")

In [ ]:
# Compare predictions
def compare_predictions(book_title, monthly_xgb_metrics, monthly_sarima_metrics, weekly_metrics):
    print(f"\n=== Comparison for {book_title} ===")
    print(f"Monthly XGBoost MAE: {monthly_xgb_metrics['MAE']:.2f}, sMAPE: {monthly_xgb_metrics['sMAPE']:.2f}%, RMSE: {monthly_xgb_metrics['RMSE']:.2f}")
    print(f"Monthly SARIMA MAE: {monthly_sarima_metrics['MAE']:.2f}, sMAPE: {monthly_sarima_metrics['sMAPE']:.2f}%, RMSE: {monthly_sarima_metrics['RMSE']:.2f}")
    weekly_data = {}
    for model, metrics in weekly_metrics.items():
        if book_title in model:
            weekly_data[model] = metrics
            print(f"Weekly {model} MAE: {metrics['MAE']:.2f}, sMAPE: {metrics['sMAPE']:.2f}%, RMSE: {metrics['RMSE']:.2f}")
    comment = (
        "Title A" in book_title and
        "Title A shows stable sales with annual peaks (e.g., December). Monthly predictions smooth these spikes, leading to higher errors, but are better for long-term trends (e.g., reprint planning). Weekly models capture short-term fluctuations (e.g., media-driven sales), suiting stock control." or
        "Title B has high volatility. Monthly predictions reduce noise but miss sharp spikes, better for budgeting. Weekly models excel at capturing sudden demand shifts, critical for inventory management."
    )
    print(f"Comment: {comment}")

In [ ]:
# Compare predictions
title_a_monthly_metrics = {
    'XGBoost': {'MAE': title_a_xgb_mae_monthly, 'sMAPE': title_a_xgb_smape_monthly, 'RMSE': title_a_xgb_rmse_monthly},
    'SARIMA': {'MAE': title_a_sarima_mae_monthly, 'sMAPE': title_a_sarima_smape_monthly, 'RMSE': title_a_sarima_rmse_monthly}
}
title_b_monthly_metrics = {
    'XGBoost': {'MAE': title_b_xgb_mae_monthly, 'sMAPE': title_b_xgb_smape_monthly, 'RMSE': title_b_xgb_rmse_monthly},
    'SARIMA': {'MAE': title_b_sarima_mae_monthly, 'sMAPE': title_b_sarima_smape_monthly, 'RMSE': title_b_sarima_rmse_monthly}
}
compare_predictions("Title A", title_a_monthly_metrics['XGBoost'], title_a_monthly_metrics['SARIMA'], weekly_metrics)
compare_predictions("Title B", title_b_monthly_metrics['XGBoost'], title_b_monthly_metrics['SARIMA'], weekly_metrics)

In [ ]:
# Collect results
results = {
    'Book': [],
    'Model': [],
    'MAE': [],
    'sMAPE': [],
    'RMSE': [],
    'MAPE': []
}
forecast_dfs = []

In [ ]:
# Store XGBoost results
if title_a_xgb_pred_monthly is not None and title_b_xgb_pred_monthly is not None:
    results['Book'].extend(["Title A", "Title B"])
    results['Model'].extend(["Monthly XGBoost", "Monthly XGBoost"])
    results['MAE'].extend([title_a_xgb_mae_monthly, title_b_xgb_mae_monthly])
    results['sMAPE'].extend([title_a_xgb_smape_monthly, title_b_xgb_smape_monthly])
    results['RMSE'].extend([title_a_xgb_rmse_monthly, title_b_xgb_rmse_monthly])
    results['MAPE'].extend([title_a_xgb_mape_monthly, title_b_xgb_mape_monthly])

    title_a_xgb_forecast_df = pd.DataFrame({
        'Date': title_a_test_monthly.index,
        'Book': "Title A",
        'Model': "Monthly XGBoost",
        'Forecast': title_a_xgb_pred_monthly,
        'Lower_CI': title_a_xgb_lower,
        'Upper_CI': title_a_xgb_upper,
        'Actual': title_a_test_original
    })
    title_b_xgb_forecast_df = pd.DataFrame({
        'Date': title_b_test_monthly.index,
        'Book': "Title B",
        'Model': "Monthly XGBoost",
        'Forecast': title_b_xgb_pred_monthly,
        'Lower_CI': title_b_xgb_lower,
        'Upper_CI': title_b_xgb_upper,
        'Actual': title_b_test_original
    })
else:
    title_a_xgb_forecast_df = pd.DataFrame()
    title_b_xgb_forecast_df = pd.DataFrame()

In [ ]:
# Store SARIMA results
if title_a_sarima_pred_monthly is not None and title_b_sarima_pred_monthly is not None:
    results['Book'].extend(["Title A", "Title B"])
    results['Model'].extend(["Monthly SARIMA", "Monthly SARIMA"])
    results['MAE'].extend([title_a_sarima_mae_monthly, title_b_sarima_mae_monthly])
    results['sMAPE'].extend([title_a_sarima_smape_monthly, title_b_sarima_smape_monthly])
    results['RMSE'].extend([title_a_sarima_rmse_monthly, title_b_sarima_rmse_monthly])
    results['MAPE'].extend([title_a_sarima_mape_monthly, title_b_sarima_mape_monthly])

    title_a_sarima_forecast_df = pd.DataFrame({
        'Date': title_a_test_monthly.index,
        'Book': "Title A",
        'Model': "Monthly SARIMA",
        'Forecast': title_a_sarima_pred_monthly,
        'Lower_CI': title_a_sarima_lower,
        'Upper_CI': title_a_sarima_upper,
        'Actual': title_a_test_original
    })
    title_b_sarima_forecast_df = pd.DataFrame({
        'Date': title_b_test_monthly.index,
        'Book': "Title B",
        'Model': "Monthly SARIMA",
        'Forecast': title_b_sarima_pred_monthly,
        'Lower_CI': title_b_sarima_lower,
        'Upper_CI': title_b_sarima_upper,
        'Actual': title_b_test_original
    })
else:
    title_a_sarima_forecast_df = pd.DataFrame()
    title_b_sarima_forecast_df = pd.DataFrame()

forecast_dfs.extend([
    title_a_xgb_forecast_df,
    title_b_xgb_forecast_df,
    title_a_sarima_forecast_df,
    title_b_sarima_forecast_df
])

In [ ]:
# Filter out empty DataFrames
forecast_dfs = [df for df in forecast_dfs if not df.empty]

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(results)
results_df['MAE'] = results_df['MAE'].round(2)
results_df['sMAPE'] = results_df['sMAPE'].round(2)
results_df['RMSE'] = results_df['RMSE'].round(2)
results_df['MAPE'] = results_df['MAPE'].round(2)

In [ ]:
# Display results table
print("\n=== Monthly Prediction Results Table ===")
print(tabulate(results_df, headers='keys', tablefmt='pretty', showindex=False))

In [ ]:
# Combine forecast DataFrames
if forecast_dfs:
    forecasts_combined = pd.concat(forecast_dfs, ignore_index=False)
    forecasts_combined = forecasts_combined.sort_values(['Date', 'Book', 'Model'])
    print("\n=== Forecast Sample Table ===")
    print(tabulate(forecasts_combined.head(10), headers='keys', tablefmt='pretty'))
else:
    print("\nWarning: No forecast DataFrames available.")

In [ ]:
# Export to Excel
output_path = './monthly_results.xlsx'
try:
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        results_df.to_excel(writer, sheet_name='Metrics', index=False)
        if forecast_dfs:
            forecasts_combined.to_excel(writer, sheet_name='Forecasts', index=False)
    print(f"\nResults exported to {output_path}")
except Exception as e:
    print(f"Error exporting to Excel: {e}")

## **Code Summary**

In [ ]:
# Track code variables
%whos

**Code Summary**

## **Final Comparison**

## **Flow Chart of the Process**

In [ ]:
# Install graphviz if not already installed
!pip install graphviz

from graphviz import Digraph

# Create a directed graph
dot = Digraph(comment='Book Sales Forecasting Process')

# Define nodes
# Data Processing
dot.node('A', 'Data Collection')
dot.node('B', 'Data Cleaning & Preparation')
dot.node('C', 'Time Series Preprocessing')
dot.node('D', 'Data Splitting')
dot.node('E', 'Training Set')
dot.node('F', 'Test Set')

# Classical Modelling
dot.node('G', 'Classical Statistical Modelling')
dot.node('G1', 'SARIMA/SARIMAX Models')
dot.node('G2', 'Auto ARIMA')
dot.node('G3', 'ETS Models')

# Deep Learning
dot.node('H', 'Deep Learning Implementation')
dot.node('H1', 'Data Scaling')
dot.node('H2', 'Sequence Generation')
dot.node('H3', 'LSTM Network for Direct Series')
dot.node('H5', 'XGBoost Model')

# Hybrid Approaches
dot.node('I', 'Hybrid Forecasting')
dot.node('I1', 'Parallel Hybrid')
dot.node('I2', 'Sequential Hybrid')
dot.node('H4', 'LSTM Network for Residuals')
dot.node('J', 'Weight Optimisation')

# Validation & Tuning
dot.node('K', 'Cross-Validation')
dot.node('L', 'Hyperparameter Tuning')
dot.node('L1', 'RandomSearch')
dot.node('L2', 'Hyperparameter Optimisation')
dot.node('M', 'Final Model Selection')

# Forecasting & Evaluation
dot.node('N', 'Forecast Generation')
dot.node('O', 'Forecast Evaluation')
dot.node('O1', 'MAE Calculation')
dot.node('O2', 'RMSE Calculation')
dot.node('O3', 'SMAPE Calculation\n(Hybrid, Monthly)')
dot.node('O4', 'Confidence Intervals')

# Reporting
dot.node('P', 'Visualisation & Reporting')
dot.node('P1', 'Performance Comparison')
dot.node('P2', 'Excel Export')
dot.node('P3', 'Visualisation with Matplotlib/Seaborn')

# Define edges
# Main flow
dot.edge('A', 'B')
dot.edge('B', 'C')
dot.edge('C', 'D')
dot.edge('D', 'E')
dot.edge('D', 'F')

# Classical models
dot.edge('E', 'G')
dot.edge('G', 'G1')
dot.edge('G', 'G2')
dot.edge('G', 'G3')

# Deep learning
dot.edge('E', 'H')
dot.edge('H', 'H1')
dot.edge('H1', 'H2')
dot.edge('H2', 'H3')
dot.edge('H2', 'H5')  # Edge to XGBoost
dot.edge('H5', 'I')   # XGBoost to Hybrid Forecasting

# Hybrid approaches
dot.edge('G1', 'I')
dot.edge('G2', 'I')
dot.edge('H3', 'I')
dot.edge('I', 'I1')
dot.edge('I', 'I2')
dot.edge('I2', 'H4')
dot.edge('I1', 'J')
dot.edge('I2', 'J')

# Optimisation and validation
dot.edge('J', 'K')
dot.edge('K', 'L')
dot.edge('L', 'L1')
dot.edge('L', 'L2')
dot.edge('L1', 'M')
dot.edge('L2', 'M')

# Forecasting and evaluation
dot.edge('M', 'N')
dot.edge('F', 'N')
dot.edge('N', 'O')
dot.edge('O', 'O1')
dot.edge('O', 'O2')
dot.edge('O', 'O3')
dot.edge('O', 'O4')

# Reporting
dot.edge('O1', 'P')
dot.edge('O2', 'P')
dot.edge('O3', 'P')
dot.edge('O4', 'P')
dot.edge('P', 'P1')
dot.edge('P', 'P2')
dot.edge('P', 'P3')

# Render the graph
dot.render('book_sales_forecasting_process', format='png', cleanup=True)
dot.render('book_sales_forecasting_process', format='svg', cleanup=True)

# Display the graph
from IPython.display import Image, SVG
Image('book_sales_forecasting_process.png')

## **End of Code**